In [ ]:
# -*- coding: utf-8 -*-
"""
PINN (TF v1 + DeepXDE) — Coupled transport with electrokinetics & surface/precip equilibrium
Units: time=day, length=m, potential=V; concentrations: mol/m^3 (water); bulk stores: mol/m^3 (bulk)

- Fast chemistry -> algebraic equilibrium + invariant transport (Ψ_Pb = θ c_Pb + SOPb + Pp)
- Acid/base network transports a = cH - cOH and enforces cH*cOH = Kw algebraically
- Stable numerics: safe θ(ψ), stable quadratic (q-trick), clipping, finite guards, grad clipping
"""

import tensorflow as tf
import deepxde as dde
import numpy as np
import random
import os

# ===== Global time scaling (PDEs are in day units) =====
SEC_PER_DAY = 86400.0  # s/day

# ===== TF v1 setup =====
tf.compat.v1.disable_eager_execution()
tf.compat.v1.reset_default_graph()
tf.compat.v1.set_random_seed(0)
random.seed(0)
np.random.seed(0)

# ---------------------------
# 0) CONFIG (day-based transport; m, V)
# ---------------------------
CFG = {
    "GLOBAL": {
        "CONSTANTS": {"F": 96485.3329, "R": 8.314462618, "T": 298.15},
        # Kw @25℃, here in (mol/m^3)^2; pH=7 => cH≈1e-4 mol/m^3 (water)
        "CHEM": {"Kw": 1.0e-8},
        "EO": {
            "zeta_mV": -30.0,                     # mV; clay effective value, pH/ionic-strength sensitive
            "eps_w": 80.0 * 8.8541878128e-12,     # F/m
            "mu_water_Pa_s": 1.0e-3,              # Pa*s
        },
        "POROUS": {"eo_tau_model": "bruggeman", "diff_model": "bruggeman", "eo_power": 2.0},
    },
    "DOMAIN": {"xmin": 0.0, "xmax": 0.2, "tmin": 0.0, "tmax": 5.0},
    "SCENARIO": {"cathode_drainage": "fixed_zero"},
    "NUMERICS": {
        "DATASET": {"n_res": 12000, "n_ic": 1200, "n_left": 1200, "n_right": 1200, "pred_n": 121},
        "NET": {"hidden_layers": 5, "width": 20, "act_scale": 10.0, "trainable_layer_gain": True},
        "TRAIN": {
            "water_iters": 1200,
            "elec_A_iters": 500,
            "hydro_elec_outer_iters": 1,
            "hydro_elec_water_iters": 1200,
            "hydro_elec_elec_iters": 500,
            "hplus_iters": 5600,
            "pb_inv_iters": 2600,
            "hplus_refine_iters": 0,
            "pb_refine_iters": 0,
            "batch_size": 512,
            "hplus_batch_size": 1536,
            "hplus_use_lbfgs": False,
            "adam_lr": 1e-3,
            "hplus_adam_lr": 2e-4,
            "hplus_grad_clip": 1.0,
            "pb_adam_lr": 8e-4,
            "pb_grad_clip": 5.0,
            "lbfgs": {"maxiter": 5000, "maxfun": 5000, "maxcor": 50, "maxls": 50,
                      "ftol": np.finfo(float).eps, "gtol": np.finfo(float).eps},
            "freeze_k_phi": 1,
            "freeze_k_c": 1,
        },
        # Amini, Haghighat & Juanes (2022), Algorithm 1 adapted to the
        # acid/base-Pb blocks. Water/electric also get a small sequential
        # closure so electroosmosis and Archie conductivity share theta/phi.
        "SEQUENTIAL": {
            "outer_max_iters": 0,
            "outer_min_iters": 1,
            "acid_iters": 1800,
            "pb_iters": 2600,
            "parameter_rel_tol": 5.0e-2,
            "pH_rms_tol": 5.0e-2,
            "pH_max_tol": 2.5e-1,
            "psi_rel_tol": 1.0e-2,
            "fixed_rel_tol": 1.0e-2,
            "diagnostic_nt": 81,
            "diagnostic_nx": 121,
        },
        "FULL_COUPLED": {
            "enabled": True,
            "iters": 1600,
            "batch_size": 1024,
            "adam_lr": 2.0e-4,
            "acid_loss_weight": 1.0,
            "pb_loss_weight": 1.0,
            "print_every": 100,
        },
    },
    "FIELDS": {
        # === Water (Richards, day units) ===
        "WATER": {
            "PARAM": {
                "SOIL": {"n": 1.843, "alpha": 0.02343789, "Ks": 4.32e-5, "theta_r": 0.068, "theta_s": 0.50, "Ss_theta": 1.0e-3, "K_sat_head_slope": 0.0},  # clay/kaolinite WRC; Ks=5e-10 m/s = 4.32e-5 m/day
                "head_transition_tau": 0.02,
                "bandai_beta_m": 0.01,
                "ic_theta_weight": 500.0,
                "boundary_theta_weight": 0.0,
                "boundary_head_weight": 80.0,
                "mass_integral_weight": 500.0,
                "mass_integral_nx": 81,
                "mass_integral_nt": 41,
                "include_eo": True,
                # For theta050/saturated runs: keep theta fixed at theta_s=0.50 and skip WaterNet training.
                "skip_water_training_at_saturation": True,
                "saturation_psi_tol": 1e-12,
            },
            "INITIAL": {"psi_ic": 0.0},  # initial pressure head, m
            "BOUNDARY": {
                "left": {
                    "face": "LEFT",
                    "type": "dirichlet",
                    "psi": 0.0,
                },
                "right": {
                    "face": "RIGHT",
                    "type": "dirichlet",
                    "psi": 0.0,
                },
            },
        },
        # === Electric (constant sigma Laplace) ===
        "ELECTRIC": {
            "PARAM": {
                "sigma_model": "archie_rhoades",
                "sigma_s": 1.8e-2,         # S/m surface conductivity
                "sigma_b": 2.5e-1,         # S/m pore-water/bulk-brine conductivity
                "rhoades_a": 1.3,
                "rhoades_b": -0.2,
                "sigma_sat": 7.425e-2,     # sigma_s + sigma_b*theta_s*(a*theta_s+b)
                "sigma_const": 7.425e-2,   # fallback / legacy name
            },
            "BOUNDARY": {"electrodes": {"anode_face": "LEFT", "cathode_face": "RIGHT", "phi_anode": 5.0, "phi_cathode": 0.0}},
        },
        # === H+ (NP; Faradaic fluxes; NO kinetic water source) ===
        "HPLUS": {
            "PARAM": {
                "DL": 0.0,                             # Kim2005: no mechanical/hydrodynamic dispersion term
                "Dw": 9.312e-9 * 86400.0,              # m^2/day (m^2/s * s/day)
                "z": +1.0, "Dw_OH": 5.273e-9 * 86400.0, "z_OH": -1.0,
                "use_softplus": True, "include_eo": True,
                "pH_min": 0.0, "pH_max": 14.0,
                "pH_obs_weight": 0.0,
                "a_ic_gate_tau": 0.01,
                "bc_tmin_eps": 0.0,
                "faraday_ramp_tau": 0.01,
                "pH_init_left": 1.45,
                "pH_init_right": 12.85,
                "pH_init_raw_scale": 0.25,
                "a_raw_scale": 20.0,
                "pb_source_stop_gradient": True,
                "H_retardation": 20.0,             # soil acid buffering; was 4.6
                "aqueous_tortuosity_model": "millington_quirk_modified",  # D_i* = Dw_i * theta^beta / theta_s^2
                "mq_theta_power": 4.736965594166206,  # beta chosen so D*_sat/Dw = 0.15 at theta_s=0.50
                "tortuosity_factor": 0.4,
                "faraday_bc_mode": "species_nonadv",
                "faraday_flux_weight": 250.0,
                "coion_flux_weight": 300.0,
                "coion_flux_weight_min": 300.0,
                "pH_range_weight": 0.0,
                "pH_range_ramped": True,
                "pH_left_max_phys": 4.0,
                "pH_right_min_phys": 10.0,
                "residual_weight": 1.0,
                "faraday_saturation_power": 0.0,
                # Residual-based adaptive sampling / adaptive loss / time-slab continuation for stiff pH fronts.
                # These switches only affect AcidBaseNet training; water/electric/Pb code is unchanged.
                "adaptive_residual_sampling": False,
                # True RAR/RBAS: most collocation points are selected from a candidate pool
                # according to the current PDE residual magnitude; a small uniform background
                # is kept to avoid losing global coverage.
                "residual_adaptive_frac": 0.70,    # high-|residual| collocation fraction
                "residual_uniform_frac": 0.30,     # uniform background fraction
                "residual_cache_refresh": 50,      # refresh high-residual cache every N Adam steps
                "residual_cache_candidates": 8192, # candidate pool size for residual ranking
                "residual_cache_keep": 4096,       # number of top residual candidates retained
                "residual_score_power": 1.0,       # score = |residual|**power

                "adaptive_loss_weights": False,
                "adaptive_loss_update_every": 100,
                "adaptive_loss_alpha": 0.35,
                "adaptive_loss_min_scale": 0.20,
                "adaptive_loss_max_scale": 5.00,

                "time_slab_training": False,
                "time_slabs": [0.10, 0.25, 0.50, 1.0, 2.0, 5.0, 10.0],
                "time_slab_mode": "expanding",    # train on [tmin, slab_end], not disjoint intervals
                "training_schedule": None
            },
            "INITIAL": {"pH_ic": 7.0, "c_ic": None},   # if None, computed from pH
            "BOUNDARY": {
                # Convention: positive flux = inject (into domain), negative = extract (out of domain)
                "left_BC":  "dirichlet",
                "right_BC": "dirichlet",
                "left_flux_sign":  "inject",   # inject / extract
                "right_flux_sign": "extract",  # inject / extract
                # Voltage-controlled Faradaic source: I = sigma_eff(Se) * |dphi/dx|.
                "I_app_Aperm2": None,
                "faraday_current_mode": "archie_voltage",
                "current_efficiency": 0.19,
                "I_APP_MODE": {"type": "fixed_current", "tH": 0.19},
                # Dirichlet reservoir-pH boundary, Kim-style and easier than species flux BC.
                "dirichlet_mode": "reservoir_pH",
                "reservoir_depth_m": 0.20,      # V_reservoir / electrode_area; 2 L / 100 cm^2
                "reservoir_pH_ramp_tau": 0.05,  # day
                "pH_loss_scale": 10.0,
                "pH_dirichlet_weight": 30.0,
                "reservoir_clip_to_config": True,
                "pH_left":  0.0, "pH_right": 14.0,
            },
        },
        # === Pb invariant transport (Ψ_Pb) ===
        "PB": {
            "PARAM": {
                "DL": 0.0,
                "psi_raw_scale": 20.0,
                "dae_res_norm": 10.0,
                "dae_res_weight": 5.0,
                "dae_alg_weight": 5.0,
                "constitutive_weight": 20.0,
                "constitutive_norm": 1.0,
                "left_constitutive_weight": 1000.0,
                "right_constitutive_weight": 300.0,
                "mass_balance_weight": 0.0,      # local dM/dt check; integral loss carries conservation
                "mass_balance_nx": 31,
                "mass_balance_batch_size": 128,
                "mass_integral_weight": 200.0,
                "mass_integral_nx": 21,
                "mass_integral_nt": 21,
                "solver": "pinn_component_equilibrium",
                "fv_nx": 161,
                "fv_nt": 1201,
                "Dw": 9.25e-10 * 86400.0,   # m^2/day
                "z": +2.0, "use_softplus": True, "include_eo": True,
                "acid_release_pH": 5.0,
                "acid_release_width": 0.35,
            },
            "INITIAL": {"c_ic": 0.0},       # initial dissolved c_Pb (water) for building Ψ_ic
            # Pb has no Faradaic source: anode is strict zero total flux; cathode is natural outflow.
            "BOUNDARY": {"left": {"type": "flux", "J": 0.0}, "right": {"type": "open_outflow"}},
            "CHEM": {
                # Equilibrium constants built from kinetic ratios; no seconds in residuals
                "k_pr_f": 1.0e5,    # m^3/(mol*s)   SOH + H+ -> SOH2+
                "k_pr_b": 2.5e3,    # 1/s
                "k_dpr_f": 1.0e-5,  # 1/s          SOH -> SO- + H+
                "k_dpr_b": 4.8e-2,  # m^3/(mol*s)  SO- + H+ -> SOH
                "k_ad_f": 1.0e5,    # m^3/(mol*s)  SOH + Pb2+ -> SOPb+ + H+
                "k_ad_b": 7.1e5,    # m^3/(mol*s)  reverse
                "m": 2.0,                           # free-Pb2+ + 2OH- saturation exponent
                "Ksp_PbOH2": 1.43e-11,             # (mol/m^3)^3; effective Pb hydroxide/oxyhydroxide Ksp
                "HYDROLYSIS": {
                    # Pb2+ + jOH- <-> Pb(OH)j^(2-j); log beta values in mol/L units.
                    # Converted internally to mol/m^3 units by subtracting 3*j from log10(beta).
                    # Representative IUPAC/PHREEQC-style mononuclear hydrolysis set; sensitivity required.
                    "include": False,
                    "species": ["PbOH+", "Pb(OH)2(aq)", "Pb(OH)3-", "Pb(OH)4--"],
                    "log_beta_OH_molL": [6.54, 11.06, 13.97, 15.20],
                    "include_polynuclear": False
                },
                "PRECIP": {
                    "mode": "solubility_product_complementarity",
                    "smooth_min_eps": 1.0e-10
                },
            },
        },
        # === Initial surface/solid stores in bulk basis (mol/m^3 bulk) ===
        "SURF_SOLID": {"INITIAL": {"SOH0": 0.0, "SOPb0": 5.0, "SOH2_0": 0.0, "SOm0": 40.0, "Pp0": 0.0}},
    },
}

def _env_float(name, default=None):
    raw = os.environ.get(name)
    if raw is None or str(raw).strip() == "":
        return default
    return float(raw)


def _psi_from_theta0(theta0, soil):
    theta0 = float(theta0)
    theta_r = float(soil["theta_r"])
    theta_s = float(soil["theta_s"])
    if not (theta_r < theta0 <= theta_s):
        raise ValueError(f"PINN_THETA0={theta0:g} must be in (theta_r={theta_r:g}, theta_s={theta_s:g}]")
    Se = (theta0 - theta_r) / max(theta_s - theta_r, 1e-12)
    Se = float(np.clip(Se, 1e-8, 1.0))
    if Se >= 1.0 - 1e-12:
        return 0.0
    n = float(soil["n"])
    m = 1.0 - 1.0 / n
    alpha = float(soil["alpha"])
    return -((Se ** (-1.0 / m) - 1.0) ** (1.0 / n)) / alpha


def apply_water_initial_case(cfg):
    water = cfg["FIELDS"]["WATER"]
    soil = water["PARAM"]["SOIL"]
    theta0_env = _env_float("PINN_THETA0")
    psi_env = _env_float("PINN_PSI_IC")
    if theta0_env is None and psi_env is None:
        return cfg
    psi_ic = float(psi_env) if psi_env is not None else _psi_from_theta0(theta0_env, soil)
    water["INITIAL"]["psi_ic"] = psi_ic
    water_case = os.environ.get("PINN_WATER_CASE")
    if not water_case:
        water_case = f"theta{int(round(float(theta0_env) * 100)):03d}" if theta0_env is not None else "psi_ic"
    cfg.setdefault("SCENARIO", {})["water_case"] = water_case
    cfg["SCENARIO"]["theta0"] = theta0_env
    cfg["SCENARIO"]["psi_ic_m"] = psi_ic
    if theta0_env is not None:
        theta_s = float(soil["theta_s"])
        ab_bc = os.environ.get("PINN_UNSAT_AB_BC", "flux").strip().lower()
        if float(theta0_env) < theta_s - 1e-12 and ab_bc in ("flux", "archie_flux", "faraday_flux"):
            Hbd = cfg["FIELDS"]["HPLUS"]["BOUNDARY"]
            Hbd["left_BC"] = "flux"
            Hbd["right_BC"] = "flux"
            cfg["SCENARIO"]["acidbase_boundary_case"] = "archie_flux"
            print("[A/B_BC] unsaturated: H/OH use Archie-voltage flux boundary")
        elif ab_bc in ("dirichlet", "reservoir", "reservoir_ph"):
            cfg["SCENARIO"]["acidbase_boundary_case"] = "reservoir_pH"
    if theta0_env is None:
        print(f"[WATER_CASE] {water_case} | psi_ic={psi_ic:g} m")
    else:
        print(f"[WATER_CASE] {water_case} | theta0={float(theta0_env):g} -> psi_ic={psi_ic:g} m")
    return cfg

def apply_cathode_drainage_case(cfg):
    case = os.environ.get("OH_DRAINAGE_CASE", cfg.get("SCENARIO", {}).get("cathode_drainage", "head_open")).strip().lower()
    psi_ic = float(cfg["FIELDS"]["WATER"]["INITIAL"].get("psi_ic", 0.0))
    water_left = cfg["FIELDS"]["WATER"]["BOUNDARY"]["left"]
    water_right = cfg["FIELDS"]["WATER"]["BOUNDARY"]["right"]
    pb_left = cfg["FIELDS"]["PB"]["BOUNDARY"]["left"]
    pb_right = cfg["FIELDS"]["PB"]["BOUNDARY"]["right"]
    if case == "head_equal":
        cathode_head = psi_ic
        scenario = "head_equal"
    elif case == "head_open":
        cathode_head = 0.0
        scenario = "head_open"
    elif case == "fixed_zero":
        anode_head = 0.0
        cathode_head = 0.0
        scenario = "fixed_zero"
    else:
        raise ValueError(f"Unknown OH_DRAINAGE_CASE={case!r}; use head_open, head_equal, or fixed_zero")
    anode_head = locals().get("anode_head", psi_ic)
    water_left.clear()
    water_left.update({"type": "dirichlet", "face": "LEFT", "psi": anode_head})
    water_right.clear()
    water_right.update({"type": "dirichlet", "face": "RIGHT", "psi": cathode_head})
    pb_left.clear()
    pb_left.update({"type": "flux", "J": 0.0})
    pb_right.clear()
    pb_right.update({"type": "open_outflow"})
    cfg["SCENARIO"]["cathode_drainage"] = scenario
    cfg["SCENARIO"]["anode_head_m"] = anode_head
    cfg["SCENARIO"]["cathode_head_m"] = cathode_head
    print(f"[SCENARIO] {scenario} | psi_ic={psi_ic:g} m | psi_left={anode_head:g} m | psi_right={cathode_head:g} m | Pb left=flux0 right=open_outflow")
    return cfg

CFG = apply_water_initial_case(CFG)
CFG = apply_cathode_drainage_case(CFG)

# -------- helpers & constants --------
def pH_to_c_m3(pH):  return 10.0 ** (-float(pH)) * 1000.0
if CFG["FIELDS"]["HPLUS"]["INITIAL"]["c_ic"] is None:
    CFG["FIELDS"]["HPLUS"]["INITIAL"]["c_ic"] = pH_to_c_m3(CFG["FIELDS"]["HPLUS"]["INITIAL"]["pH_ic"])

DOMAIN = CFG["DOMAIN"]
def x_of_face(face_name):
    if face_name == "LEFT":  return float(DOMAIN["xmin"])
    if face_name == "RIGHT": return float(DOMAIN["xmax"])
    raise ValueError("face must be LEFT/RIGHT")

def signed_flux_along_pos_x(mag, face, direction):
    # positive x is to the right.
    if direction not in ("into_domain","out_of_domain"):
        raise ValueError("direction must be 'into_domain'/'out_of_domain'")
    if face == "LEFT":  return +mag if direction=="into_domain" else -mag
    return -mag if direction=="into_domain" else +mag

# ---- dataset grids ----
DATASET = CFG["NUMERICS"]["DATASET"]
pred_n = DATASET["pred_n"]
x_lin = np.linspace(DOMAIN["xmin"], DOMAIN["xmax"], pred_n).astype(np.float32)
t_lin = np.linspace(DOMAIN["tmin"], DOMAIN["tmax"], pred_n).astype(np.float32)
x_pred, t_pred = np.meshgrid(x_lin, t_lin)
t_star = t_pred.flatten().reshape(-1,1).astype(np.float32)
x_star = x_pred.flatten().reshape(-1,1).astype(np.float32)

def get_collocations(box, n):
    x = np.random.uniform(box["xmin"], box["xmax"], n).reshape(-1, 1)
    t = np.random.uniform(box["tmin"], box["tmax"], n).reshape(-1, 1)
    return t.astype(np.float32), x.astype(np.float32)

t_res, x_res = get_collocations(DOMAIN, DATASET["n_res"])
t_ic,  x_ic  = get_collocations({"xmin":DOMAIN["xmin"],"xmax":DOMAIN["xmax"],
                                 "tmin":DOMAIN["tmin"],"tmax":DOMAIN["tmin"]}, DATASET["n_ic"])

def face_samples(face, n, tmin_override=None):
    x0 = x_of_face(face)
    t0 = DOMAIN["tmin"] if tmin_override is None else max(float(tmin_override), DOMAIN["tmin"])
    t = np.random.uniform(t0, DOMAIN["tmax"], n).reshape(-1,1).astype(np.float32)
    x = np.full_like(t, x0, dtype=np.float32)
    return t, x

TF_CONFIG = tf.compat.v1.ConfigProto(allow_soft_placement=True, log_device_placement=False)

# ---- unpack ----
SOIL   = CFG["FIELDS"]["WATER"]["PARAM"]["SOIL"]
EO_PAR = CFG["GLOBAL"]["EO"]
ELEC_PAR = CFG["FIELDS"]["ELECTRIC"]["PARAM"]
ELEC_BD = CFG["FIELDS"]["ELECTRIC"]["BOUNDARY"]["electrodes"]
H_PAR, H_INIT, H_BC = CFG["FIELDS"]["HPLUS"]["PARAM"], CFG["FIELDS"]["HPLUS"]["INITIAL"], CFG["FIELDS"]["HPLUS"]["BOUNDARY"]
PB_PAR, PB_INIT, PB_BC = CFG["FIELDS"]["PB"]["PARAM"], CFG["FIELDS"]["PB"]["INITIAL"], CFG["FIELDS"]["PB"]["BOUNDARY"]
SURF_INIT = CFG["FIELDS"]["SURF_SOLID"]["INITIAL"]
W_INIT, W_BC = CFG["FIELDS"]["WATER"]["INITIAL"], CFG["FIELDS"]["WATER"]["BOUNDARY"]

ANODE_FACE   = ELEC_BD["anode_face"]; CATHODE_FACE = ELEC_BD["cathode_face"]
phi_anode, phi_cathode = ELEC_BD["phi_anode"], ELEC_BD["phi_cathode"]

# Fixed-current auto-fill is intentionally disabled here.
# Faradaic H/OH flux is computed later from prescribed voltage and Archie-type sigma_eff(Se).

# ---- constants ----
F_c = tf.constant(CFG["GLOBAL"]["CONSTANTS"]["F"], tf.float32)
R_c = tf.constant(CFG["GLOBAL"]["CONSTANTS"]["R"], tf.float32)
T_c = tf.constant(CFG["GLOBAL"]["CONSTANTS"]["T"], tf.float32)
Kw_const = tf.constant(CFG["GLOBAL"]["CHEM"]["Kw"], tf.float32)
sigma_sat_const = tf.constant(float(ELEC_PAR.get("sigma_sat", ELEC_PAR.get("sigma_const", 7.425e-2))), tf.float32)
sigma_const = tf.constant(float(ELEC_PAR.get("sigma_const", ELEC_PAR.get("sigma_sat", 7.425e-2))), tf.float32)
sigma_s_const = tf.constant(float(ELEC_PAR.get("sigma_s", 1.8e-2)), tf.float32)
sigma_b_const = tf.constant(float(ELEC_PAR.get("sigma_b", 2.5e-1)), tf.float32)
rhoades_a_const = tf.constant(float(ELEC_PAR.get("rhoades_a", 1.3)), tf.float32)
rhoades_b_const = tf.constant(float(ELEC_PAR.get("rhoades_b", -0.2)), tf.float32)

DL_H = tf.constant(H_PAR["DL"], tf.float32); Dw_H = tf.constant(H_PAR["Dw"], tf.float32); z_H  = tf.constant(H_PAR["z"], tf.float32)
DL_OH = tf.constant(H_PAR.get("DL", H_PAR["DL"]), tf.float32)
Dw_OH = tf.constant(H_PAR.get("Dw_OH", 5.273e-9 * 86400.0), tf.float32)
z_OH  = tf.constant(float(H_PAR.get("z_OH", -1.0)), tf.float32)
A_CLIP = tf.constant(float(H_PAR.get("a_clip", 1.0e3)), tf.float32)
DL_Pb= tf.constant(PB_PAR["DL"], tf.float32); Dw_Pb= tf.constant(PB_PAR["Dw"], tf.float32); z_Pb= tf.constant(PB_PAR["z"], tf.float32)

# ---------------------------
# 1) Unsaturated functions (day units) — robust variants
# ---------------------------
nvg = tf.constant([SOIL["n"]], tf.float32); mvg = 1. - 1./nvg
ksvg = tf.constant([SOIL["Ks"]], tf.float32)  # m/day
alphavg = tf.constant([SOIL["alpha"]], tf.float32)
thetaRvg = tf.constant([SOIL["theta_r"]], tf.float32)
thetaSvg = tf.constant([SOIL["theta_s"]], tf.float32)

zeta_mV_const = EO_PAR["zeta_mV"]; eps_w = EO_PAR["eps_w"]; mu_water_Pa_s = EO_PAR["mu_water_Pa_s"]

def grad0(y, x):
    g = tf.gradients(y, x, unconnected_gradients='zero')[0]
    if g is None:
        return tf.zeros_like(x)
    return g

# PDF-style nondimensional coordinates for all neural nets: t_bar in [0,1], x_bar in [0,1].
X_MIN_TF = tf.constant(float(DOMAIN["xmin"]), tf.float32)
T_MIN_TF = tf.constant(float(DOMAIN["tmin"]), tf.float32)
X_SCALE_TF = tf.constant(max(float(DOMAIN["xmax"] - DOMAIN["xmin"]), 1e-12), tf.float32)
T_SCALE_TF = tf.constant(max(float(DOMAIN["tmax"] - DOMAIN["tmin"]), 1e-12), tf.float32)

def nondim_tx(t, x):
    return tf.concat([(tf.cast(t, tf.float32) - T_MIN_TF) / T_SCALE_TF,
                      (tf.cast(x, tf.float32) - X_MIN_TF) / X_SCALE_TF], 1)

def nondim_X(X):
    X = tf.cast(X, tf.float32)
    return tf.concat([(X[:, 0:1] - T_MIN_TF) / T_SCALE_TF,
                      (X[:, 1:2] - X_MIN_TF) / X_SCALE_TF], 1)

def theta_function(h):
    # Unsaturated VG below water table; small saturated storage above it.
    s = tf.maximum(-alphavg * h, 0.0)                 # base >= 0
    term3 = tf.pow(1.0 + tf.pow(s, nvg), -mvg)
    theta_unsat = thetaRvg + (thetaSvg - thetaRvg) * term3
    Ss_theta = tf.constant(float(SOIL.get("Ss_theta", 0.0)), tf.float32)
    theta_sat = thetaSvg + Ss_theta * tf.maximum(h, 0.0)
    return tf.where(h >= 0.0, theta_sat, theta_unsat)

def K_function(h):
    theta_h = theta_function(h)
    Se = (theta_h - thetaRvg)/(thetaSvg-thetaRvg+1e-12)
    # Clay n is close to 1, so the Mualem derivative is singular at exact saturation.
    Se_clip = tf.clip_by_value(Se, 1e-8, 1.0 - 1e-6)
    inner = tf.clip_by_value(1.0 - tf.pow(Se_clip, 1.0/mvg), 1e-12, 1.0)
    term2 = 1.0 - tf.pow(inner, mvg)
    K_unsat = ksvg * tf.pow(Se_clip, 0.5) * tf.pow(term2, 2.0)
    sat_slope = tf.constant(float(SOIL.get("K_sat_head_slope", 0.0)), tf.float32)
    K_sat = ksvg * tf.exp(sat_slope * tf.maximum(h, 0.0))
    return tf.where(h >= 0.0, K_sat, K_unsat)  # m/day

def sat_vars(theta):
    theta_s = thetaSvg; theta_r = thetaRvg
    Se = (theta - theta_r) / (theta_s - theta_r + 1e-12)
    Se = tf.clip_by_value(Se, 1e-8, 1.0)
    return theta_s, theta_r, Se

def tau_from_Se(Se, model):
    if model == "bruggeman":
        return tf.pow(Se + 1e-30, -0.5)
    return tf.ones_like(Se)

# k_eo(θ) : m^2/(V·day)
def keo_of_theta(theta):
    _, _, Se = sat_vars(theta)
    tau_eo = tau_from_Se(Se, CFG["GLOBAL"]["POROUS"]["eo_tau_model"])
    chi_eo = tf.pow(Se, CFG["GLOBAL"]["POROUS"]["eo_power"])
    zeta = tf.constant(zeta_mV_const*1e-3, tf.float32)
    return (tf.constant(eps_w, tf.float32) * zeta * chi_eo * tf.constant(SEC_PER_DAY, tf.float32)) / (tf.constant(mu_water_Pa_s, tf.float32) * tau_eo)

def sigma_eff_of_theta(theta):
    """Rhoades Archie-type conductivity: sigma_s + sigma_b*theta*(a*theta+b)."""
    model = str(ELEC_PAR.get("sigma_model", "const")).lower()
    if model.startswith("archie") or model.startswith("rhoades"):
        theta_c = tf.maximum(theta, 0.0)
        sigma = sigma_s_const + sigma_b_const * theta_c * (rhoades_a_const * theta_c + rhoades_b_const)
        return tf.maximum(sigma, 1e-12)
    return tf.ones_like(theta) * sigma_const

def Dstar_of_theta(theta, Dw):
    # Modified Millington-Quirk power law: D*_sat/Dw = theta_s^(beta-2).
    theta_c = tf.clip_by_value(theta, 0.0, thetaSvg)
    beta = tf.constant(float(H_PAR.get("mq_theta_power", 10.0/3.0)), tf.float32)
    return Dw * tf.pow(theta_c, beta) / (tf.pow(thetaSvg, 2.0) + 1e-30)  # m^2/day

def diffusion_pieces(theta, q_adv, DL, Dw, z_val):
    Dstar = Dstar_of_theta(theta, Dw)
    Deff  = Dstar + DL * tf.abs(q_adv)            # m^2/day
    ustar = (z_val*F_c/(R_c*T_c)) * Dstar         # m^2/(V·day)
    return Deff, ustar

# ---------------------------
# 2) Generic losses  (with per-term weights)
# ---------------------------
_EPS = 1e-12
def _loss_from_type(loss_type, err, lhs=None, rhs=None, delta=1.0):
    lt=loss_type.lower()
    if lt=="mse": return tf.reduce_mean(tf.square(err))
    if lt=="mae": return tf.reduce_mean(tf.abs(err))
    if lt=="huber":
        d=tf.constant(float(delta),tf.float32); ae=tf.abs(err)
        return tf.reduce_mean(tf.where(ae<=d,0.5*tf.square(err), d*(ae-0.5*d)))
    if lt=="relative_mse":
        denom = tf.reduce_mean(tf.square(rhs)) if rhs is not None else tf.reduce_mean(tf.square(lhs)) if lhs is not None else tf.constant(1.0,tf.float32)
        return tf.reduce_mean(tf.square(err))/(denom+_EPS)
    return tf.reduce_mean(tf.square(err))

def build_loss(terms_cfg, tensors):
    parts = {}; total = 0.0
    for tcfg in terms_cfg:
        name  = tcfg.get("name","term")
        ltype = tcfg.get("type","mse")
        delta = tcfg.get("delta",1.0)
        w_val = float(tcfg.get("w", 1.0))
        if "target" in tcfg:
            x = tensors[tcfg["target"]]
            li = _loss_from_type(ltype, x, delta=delta)
        else:
            lhs = tensors[tcfg["lhs"]]; rhs_key=tcfg.get("rhs","zero")
            rhs = tf.zeros_like(lhs) if rhs_key=="zero" else tensors[rhs_key]
            err = lhs - rhs
            li  = _loss_from_type(ltype, err, lhs=lhs, rhs=rhs, delta=delta)
        li_w = tf.constant(w_val, tf.float32) * li
        parts[name] = li_w
        total = total + parts[name]
    return total, parts

LOS = {
    # 给电场也配上权重（边界更硬）
    "elec": {
        "terms": [
            {"name":"res","type":"mse","target":"residual_res", "w":1.0},
            {"name":"bc_left","type":"huber","lhs":"phi_left_pred","rhs":"phi_left_true","delta":0.01, "w":100.0},
            {"name":"bc_right","type":"huber","lhs":"phi_right_pred","rhs":"phi_right_true","delta":0.01, "w":100.0}
        ]
    },
    "hplus_flux": [
        {"name":"res","type":"relative_mse","target":"residual_res", "w":1.0},
        {"name":"left_flux","type":"mse","lhs":"J_left_pred","rhs":"J_left_true", "w":80.0},
        {"name":"right_flux","type":"mse","lhs":"J_right_pred","rhs":"J_right_true", "w":80.0},
        {"name":"ic","type":"mse","lhs":"c_ic_pred","rhs":"c_ic_true", "w":25.0}
    ],
}

# ---------------------------
# 3) Dense MLP
# ---------------------------
NETC = CFG["NUMERICS"]["NET"]; TRNC = CFG["NUMERICS"]["TRAIN"]; SEQC = CFG["NUMERICS"]["SEQUENTIAL"]
LOSS_HISTORY_BY_STAGE = []

def record_loss_history(stage, model, step_interval=100):
    hist = [float(v) for v in np.asarray(getattr(model, "loss_hist", []), dtype=float).reshape(-1) if np.isfinite(v)]
    record_loss_values(stage, hist, step_interval)

def record_loss_values(stage, hist, step_interval=100):
    hist = [float(v) for v in np.asarray(hist, dtype=float).reshape(-1) if np.isfinite(v)]
    LOSS_HISTORY_BY_STAGE.append({
        "stage": stage,
        "step_interval": int(step_interval),
        "loss": hist,
    })
    if hist:
        print(f"[LossHistory] {stage}: points={len(hist)}, first={hist[0]:.3e}, last={hist[-1]:.3e}")
    else:
        print(f"[LossHistory] {stage}: no recorded Adam loss points")
class DenseMLP:
    def __init__(self, layers, act_scale=10.0, trainable_gain=True):
        self.layers=layers; self.act_scale=act_scale; self.trainable_gain=trainable_gain
        self.weights=[]; self.biases=[]; self.A=[]
        for l in range(len(layers)-1):
            in_dim,out_dim=layers[l],layers[l+1]
            std=np.sqrt(2.0/(in_dim+out_dim))
            W=tf.Variable(tf.random.truncated_normal([in_dim,out_dim], stddev=std), dtype=tf.float32, trainable=True)
            b=tf.Variable(np.zeros([1,out_dim]), dtype=tf.float32, trainable=True)
            a=tf.Variable(0.05, dtype=tf.float32, trainable=trainable_gain)
            self.weights.append(W); self.biases.append(b); self.A.append(a)
    def forward(self, X):
        H=nondim_X(X)
        for l in range(len(self.weights)):
            H=tf.add(tf.matmul(H,self.weights[l]), self.biases[l])
            if l < len(self.weights)-1:
                H=tf.tanh(self.act_scale*self.A[l]*H)
        return H

def dense_eval_from_weights(weights, biases, gains, t, x):
    H = nondim_tx(t, x)
    for l in range(len(weights)):
        H = tf.add(tf.matmul(H, weights[l]), biases[l])
        if l < len(weights) - 1:
            H = tf.tanh(NETC["act_scale"] * gains[l] * H)
    return H

def water_head_from_raw(raw, t, x):
    # Bandai-Ghezzehei style pressure-head transform: h = beta - exp(N).
    # It keeps the water network in the unsaturated/saturated-pressure range
    # without hard-wiring the whole interior to a boundary value.
    beta = tf.constant(float(CFG["FIELDS"]["WATER"]["PARAM"].get("bandai_beta_m", 0.01)), tf.float32)
    psi = beta - tf.exp(tf.clip_by_value(raw, -30.0, 30.0))
    return tf.where(tf.math.is_finite(psi), psi, tf.zeros_like(psi))

def water_head_from_weights(weights, biases, gains, t, x):
    return water_head_from_raw(dense_eval_from_weights(weights, biases, gains, t, x), t, x)


def make_constant_water_weights(layers, psi_value=0.0):
    """Return non-trainable-looking DenseMLP weights that give a constant water head.

    The water network transform is psi = beta - exp(raw).  Setting all weights to
    zero and the final bias to log(beta - psi_value) makes
        water_head_from_weights(...) == psi_value
    everywhere.  For the saturated case we use psi_value=0, so theta=theta_s.
    """
    beta = float(CFG["FIELDS"]["WATER"]["PARAM"].get("bandai_beta_m", 0.01))
    raw_const = np.log(max(beta - float(psi_value), 1e-12))
    W = []
    B = []
    A = []
    for i in range(len(layers) - 1):
        W.append(np.zeros((int(layers[i]), int(layers[i + 1])), dtype=np.float32))
        b = np.zeros((1, int(layers[i + 1])), dtype=np.float32)
        if i == len(layers) - 2:
            b[:] = np.float32(raw_const)
        B.append(b)
        A.append(np.array(0.05, dtype=np.float32))
    return W, B, A


def use_fixed_saturated_water():
    """Auto-detect the saturated water-content case and skip WaterNet training.

    The run scripts set theta050 through psi_ic=0.  With the VG function used here,
    any non-negative pressure head gives the saturated branch; psi=0 gives exactly
    theta_s (0.50, unless changed in SOIL).  In this case training WaterNet is
    unnecessary and can introduce instability, so we freeze water at psi=0.
    """
    wp = CFG["FIELDS"]["WATER"].get("PARAM", {})
    if not bool(wp.get("skip_water_training_at_saturation", True)):
        return False
    psi_ic = float(CFG["FIELDS"]["WATER"].get("INITIAL", {}).get("psi_ic", 0.0))
    left_psi = float(CFG["FIELDS"]["WATER"].get("BOUNDARY", {}).get("left", {}).get("psi", psi_ic))
    right_psi = float(CFG["FIELDS"]["WATER"].get("BOUNDARY", {}).get("right", {}).get("psi", psi_ic))
    tol = float(wp.get("saturation_psi_tol", 1e-12))
    return (psi_ic >= -tol) and (left_psi >= -tol) and (right_psi >= -tol)

# ---------------------------
# 4) Water / Electric / H+ 
# ---------------------------
class WaterNet:
    def __init__(self, layers, include_eo_in_water=False, elec_weights=None, wbc_cfg=None, init_from=None, freeze_k=0):
        self.mlp=DenseMLP(layers, NETC["act_scale"], NETC["trainable_layer_gain"])
        if init_from is not None:
            w0,b0,a0=init_from
            for i in range(len(self.mlp.weights)):
                self.mlp.weights[i]=tf.Variable(np.array(w0[i]), dtype=tf.float32, trainable=(i>=freeze_k))
                self.mlp.biases[i] =tf.Variable(np.array(b0[i]), dtype=tf.float32, trainable=(i>=freeze_k))
                self.mlp.A[i]      =tf.Variable(a0[i], dtype=tf.float32, trainable=(i>=freeze_k))
        (self.t_res,self.x_res,self.t_ic,self.x_ic,self.t_left,self.x_left,self.t_right,self.x_right)= \
            [tf.compat.v1.placeholder(tf.float32,[None,1]) for _ in range(8)]
        self.sess=tf.compat.v1.Session(config=TF_CONFIG)
        self.include_eo=include_eo_in_water; self.elec_weights=elec_weights
        self.wbc = wbc_cfg if wbc_cfg is not None else {"left": {"type":"dirichlet", "psi":W_INIT["psi_ic"]}, "right":{"type":"dirichlet", "psi":0.0}}

        # Core residuals
        self.psi_res,self.residual_res=self.net_res(self.t_res,self.x_res)
        Ldom = max(float(DOMAIN["xmax"] - DOMAIN["xmin"]), 1e-12)
        Tspan = max(float(DOMAIN["tmax"] - DOMAIN["tmin"]), 1e-12)
        theta_scale_val = max(abs(float(SOIL["theta_s"]) - float(SOIL["theta_r"])), 1e-6)
        q_scale_val = max(abs(float(SOIL.get("Ks", 0.0))), 1e-12)
        water_res_norm = tf.constant(float(CFG["FIELDS"]["WATER"]["PARAM"].get("res_norm", max(theta_scale_val / Tspan, q_scale_val / Ldom, 1e-8))), tf.float32)
        self.residual_res_nd = self.residual_res / water_res_norm
        self.psi_ic_pred=self.net_psi(tf.concat([self.t_ic,self.x_ic],1))
        self.psi_ic_true=tf.fill(tf.shape(self.psi_ic_pred), tf.constant(W_INIT["psi_ic"],tf.float32))
        self.theta_ic_pred=theta_function(self.psi_ic_pred)
        self.theta_ic_true=theta_function(self.psi_ic_true)

        # ---- Build boundary conditions dynamically from config ----
        terms = [
            {"name":"res","type":"mse","target":"residual_res_nd", "w":1.0},
            {"name":"ic_theta", "type":"mse","lhs":"theta_ic_pred","rhs":"theta_ic_true", "w":float(CFG["FIELDS"]["WATER"]["PARAM"].get("ic_theta_weight", 500.0))},
        ]
        tensors={"residual_res":self.residual_res, "residual_res_nd":self.residual_res_nd, "psi_ic_pred":self.psi_ic_pred, "psi_ic_true":self.psi_ic_true,
                 "theta_ic_pred":self.theta_ic_pred, "theta_ic_true":self.theta_ic_true}

        lcfg = self.wbc.get("left", {"type":"dirichlet", "psi":W_INIT["psi_ic"]})
        rcfg = self.wbc.get("right", {"type":"dirichlet", "psi":0.0})
        head_scale_val = max(abs(float(W_INIT["psi_ic"])), abs(float(lcfg.get("psi", W_INIT["psi_ic"]))), abs(float(rcfg.get("psi", 0.0))), 1.0)
        head_scale = tf.constant(head_scale_val, tf.float32)
        if str(lcfg.get("type", "dirichlet")).lower() != "dirichlet" or str(rcfg.get("type", "dirichlet")).lower() != "dirichlet":
            raise ValueError("Water boundary only supports dirichlet head in this model copy.")
        self.psi_left_pred = self.net_psi(tf.concat([self.t_left,self.x_left],1))
        _t0_left = tf.constant(float(CFG["DOMAIN"]["tmin"]), tf.float32)
        _tau_left = tf.constant(float(CFG["FIELDS"]["WATER"]["PARAM"].get("head_transition_tau", 0.02)), tf.float32)
        _gate_left = tf.where(self.t_left <= _t0_left, tf.zeros_like(self.t_left), 1.0 - tf.exp(-(self.t_left - _t0_left) / tf.maximum(_tau_left, 1e-6)))
        self.psi_left_true = tf.constant(float(W_INIT["psi_ic"]), tf.float32) + _gate_left * (tf.constant(float(lcfg.get("psi", W_INIT["psi_ic"])), tf.float32) - tf.constant(float(W_INIT["psi_ic"]), tf.float32))
        self.theta_left_pred = theta_function(self.psi_left_pred)
        self.theta_left_true = theta_function(self.psi_left_true)
        self.psi_left_err_scaled = (self.psi_left_pred - self.psi_left_true) / head_scale
        terms.append({"name":"left_theta","type":"mse","lhs":"theta_left_pred","rhs":"theta_left_true","w":float(CFG["FIELDS"]["WATER"]["PARAM"].get("boundary_theta_weight", 0.0))})
        terms.append({"name":"left_head_scaled","type":"mse","target":"psi_left_err_scaled","w":float(CFG["FIELDS"]["WATER"]["PARAM"].get("boundary_head_weight", 80.0))})
        tensors.update({"psi_left_pred": self.psi_left_pred, "psi_left_true": self.psi_left_true, "psi_left_err_scaled": self.psi_left_err_scaled,
                        "theta_left_pred": self.theta_left_pred, "theta_left_true": self.theta_left_true})
        self.psi_right_pred = self.net_psi(tf.concat([self.t_right,self.x_right],1))
        _t0 = tf.constant(float(CFG["DOMAIN"]["tmin"]), tf.float32)
        _tau = tf.constant(float(CFG["FIELDS"]["WATER"]["PARAM"].get("head_transition_tau", 0.02)), tf.float32)
        _gate = tf.where(self.t_right <= _t0, tf.zeros_like(self.t_right), 1.0 - tf.exp(-(self.t_right - _t0) / tf.maximum(_tau, 1e-6)))
        self.psi_right_true = tf.constant(float(W_INIT["psi_ic"]), tf.float32) + _gate * (tf.constant(float(rcfg.get("psi", W_INIT["psi_ic"])), tf.float32) - tf.constant(float(W_INIT["psi_ic"]), tf.float32))
        self.theta_right_pred = theta_function(self.psi_right_pred)
        self.theta_right_true = theta_function(self.psi_right_true)
        self.psi_right_err_scaled = (self.psi_right_pred - self.psi_right_true) / head_scale
        terms.append({"name":"right_theta","type":"mse","lhs":"theta_right_pred","rhs":"theta_right_true","w":float(CFG["FIELDS"]["WATER"]["PARAM"].get("boundary_theta_weight", 0.0))})
        terms.append({"name":"right_head_scaled","type":"mse","target":"psi_right_err_scaled","w":float(CFG["FIELDS"]["WATER"]["PARAM"].get("boundary_head_weight", 80.0))})
        tensors.update({"psi_right_pred": self.psi_right_pred, "psi_right_true": self.psi_right_true, "psi_right_err_scaled": self.psi_right_err_scaled,
                        "theta_right_pred": self.theta_right_pred, "theta_right_true": self.theta_right_true})

        self.water_mass_integral_res = self.net_mass_integral_res()
        terms.append({"name":"water_mass_integral","type":"mse","target":"water_mass_integral_res",
                      "w":float(CFG["FIELDS"]["WATER"]["PARAM"].get("mass_integral_weight", 0.0))})
        tensors["water_mass_integral_res"] = self.water_mass_integral_res

        # Build loss (with weights)
        self.loss,self.loss_parts=build_loss(terms,tensors)

        # Optimizer with global grad clipping
        self.global_step=tf.Variable(0,trainable=False)
        lr=tf.compat.v1.train.exponential_decay(TRNC["adam_lr"], self.global_step, 1000,0.9)
        opt = tf.compat.v1.train.AdamOptimizer(lr)
        vars_all = [v for v in (self.mlp.weights + self.mlp.biases + self.mlp.A) if getattr(v, "trainable", True)]
        grads_vars = [(g, v) for g, v in opt.compute_gradients(self.loss, var_list=vars_all) if g is not None]
        grads, vars_ = zip(*grads_vars)
        grads, _ = tf.clip_by_global_norm(grads, 5.0)
        self.train_op = opt.apply_gradients(list(zip(grads, vars_)), global_step=self.global_step)

        self.lbfgs=dde.optimizers.tensorflow_compat_v1.scipy_optimizer.ScipyOptimizerInterface(
            self.loss,method='L-BFGS-B',options=TRNC["lbfgs"])
        self.sess.run(tf.compat.v1.global_variables_initializer()); self.loss_hist=[]
        self._best_loss=np.inf; self._best_weights=None; self._best_it=None

    def net_psi(self, X): 
        return water_head_from_raw(self.mlp.forward(X), X[:, 0:1], X[:, 1:2])

    def _phi_x_if_needed(self,t,x):
        if (not self.include_eo) or (self.elec_weights is None):
            return None
        w_e,b_e,a_e=self.elec_weights
        H=nondim_tx(t,x)
        for l in range(len(w_e)):
            H=tf.add(tf.matmul(H,w_e[l]), b_e[l])
            if l < len(w_e)-1: H=tf.tanh(NETC["act_scale"]*a_e[l]*H)
        phi=H; return tf.gradients(phi,x)[0]

    def net_q_parts(self,t,x):
        X=tf.concat([t,x],1); psi=self.net_psi(X); K=K_function(psi); theta=theta_function(psi)
        psi_x=tf.gradients(psi,x)[0]; q_hyd=-K*(psi_x)
        q_eo=tf.zeros_like(q_hyd)
        if self.include_eo:
            phi_x=self._phi_x_if_needed(t,x)
            if phi_x is not None: q_eo=keo_of_theta(theta)*phi_x
        return q_hyd, q_eo, q_hyd + q_eo

    def net_q_total(self,t,x):
        # total water flux (hydraulic + electroosmotic if enabled)
        return self.net_q_parts(t,x)[2]

    def net_psix(self,t,x):
        X=tf.concat([t,x],1); psi=self.net_psi(X); return tf.gradients(psi,x)[0]

    def net_res(self,t,x):
        X=tf.concat([t,x],1); psi=self.net_psi(X); theta=theta_function(psi)
        q_tot=self.net_q_total(t,x)
        res=tf.gradients(q_tot,x)[0] + tf.gradients(theta,t)[0]
        return psi,res

    def net_mass_integral_res(self):
        wp = CFG["FIELDS"]["WATER"]["PARAM"]
        nx = max(int(wp.get("mass_integral_nx", 81)), 3)
        nt = max(int(wp.get("mass_integral_nt", 41)), 3)
        xmin = float(CFG["DOMAIN"]["xmin"]); xmax = float(CFG["DOMAIN"]["xmax"])
        tmin = float(CFG["DOMAIN"]["tmin"]); tmax = float(CFG["DOMAIN"]["tmax"])
        early = np.array([0.0, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.35], dtype=np.float32)
        uniform = np.linspace(tmin, tmax, nt, dtype=np.float32)
        t_vals = np.unique(np.clip(np.concatenate([uniform, early]), tmin, tmax)).astype(np.float32)
        t_vals.sort()
        nt_eff = int(t_vals.size)
        t_col = tf.constant(t_vals.reshape(-1, 1), tf.float32)
        x_row = tf.constant(np.linspace(xmin, xmax, nx, dtype=np.float32).reshape(1, -1), tf.float32)
        tt = tf.reshape(tf.tile(t_col, [1, nx]), [-1, 1])
        xx = tf.reshape(tf.tile(x_row, [nt_eff, 1]), [-1, 1])
        theta = tf.reshape(theta_function(self.net_psi(tf.concat([tt, xx], 1))), [nt_eff, nx])
        dx = tf.constant((xmax - xmin) / float(nx - 1), tf.float32)
        xw = tf.concat([tf.ones([1], tf.float32) * 0.5, tf.ones([nx - 2], tf.float32), tf.ones([1], tf.float32) * 0.5], 0)[None, :]
        S = dx * tf.reduce_sum(theta * xw, axis=1, keepdims=True)
        x_left = tf.ones_like(t_col) * tf.constant(xmin, tf.float32)
        x_right = tf.ones_like(t_col) * tf.constant(xmax, tf.float32)
        flux_out = self.net_q_total(t_col, x_right) - self.net_q_total(t_col, x_left)
        dt = t_col[1:] - t_col[:-1]
        step = 0.5 * (flux_out[1:] + flux_out[:-1]) * dt
        cum_flux = tf.concat([tf.zeros([1, 1], tf.float32), tf.cumsum(step, axis=0)], axis=0)
        L = tf.constant(max(xmax - xmin, 1e-6), tf.float32)
        return (S - S[0:1] + cum_flux) / L

    def _remember_best(self, loss, label):
        if np.isfinite(loss) and loss < self._best_loss:
            self._best_loss=float(loss); self._best_it=label; self._best_weights=self.export_weights()
            return True
        return False

    def _restore_best_checkpoint(self):
        if self._best_weights is None:
            return False
        w0,b0,a0=self._best_weights
        assigns=[]
        for var,val in zip(self.mlp.weights,w0): assigns.append(tf.compat.v1.assign(var,np.asarray(val,dtype=np.float32)))
        for var,val in zip(self.mlp.biases,b0): assigns.append(tf.compat.v1.assign(var,np.asarray(val,dtype=np.float32)))
        for var,val in zip(self.mlp.A,a0): assigns.append(tf.compat.v1.assign(var,np.asarray(val,dtype=np.float32)))
        self.sess.run(assigns)
        print(f"[Water] restored best checkpoint: {self._best_it}, L={self._best_loss:.3e}")
        return True

    def train(self,N_iter,batch=True,batch_size=512, face_feed=None):
        t_left,x_left = face_feed["left"]; t_right,x_right = face_feed["right"]
        names=sorted(self.loss_parts.keys())
        feed=None
        for it in range(N_iter):
            idx=np.random.choice(t_res.shape[0],batch_size,replace=False); tr,xr=t_res[idx,:],x_res[idx,:]
            feed={self.t_res:tr,self.x_res:xr,self.t_ic:t_ic,self.x_ic:x_ic,
                  self.t_left:t_left,self.x_left:x_left,self.t_right:t_right,self.x_right:x_right}
            self.sess.run(self.train_op,feed)
            if it%100==0:
                vals=self.sess.run([self.loss]+[self.loss_parts[n] for n in names],feed)
                loss_now=float(vals[0])
                best_mark=" *best" if self._remember_best(loss_now, f"adam:{it}") else ""
                print(f"[Water] It {it:5d} L={loss_now:.3e}{best_mark} | "+", ".join([f"{n}={v:.2e}" for n,v in zip(names,vals[1:])]))
                self.loss_hist.append(loss_now)
        if feed is not None:
            self.lbfgs.minimize(self.sess, feed_dict=feed, fetches=[self.loss])
            loss_after=float(self.sess.run(self.loss, feed))
            self._remember_best(loss_after, "lbfgs")
            print("[Water] LBFGS done.")
            self._restore_best_checkpoint()

    def export_weights(self):
        return self.sess.run(self.mlp.weights), self.sess.run(self.mlp.biases), self.sess.run(self.mlp.A)

class ElectricNet:
    def __init__(self,layers,init_from=None,freeze_k=0, phi_left=10.0, phi_right=0.0, water_weights=None):
        self.mlp=DenseMLP(layers, NETC["act_scale"], NETC["trainable_layer_gain"])
        if init_from is not None:
            w0,b0,a0=init_from
            for i in range(len(self.mlp.weights)):
                self.mlp.weights[i]=tf.Variable(np.array(w0[i]), dtype=tf.float32, trainable=(i>=freeze_k))
                self.mlp.biases[i] =tf.Variable(np.array(b0[i]), dtype=tf.float32, trainable=(i>=freeze_k))
                self.mlp.A[i]      =tf.Variable(a0[i], dtype=tf.float32, trainable=(i>=freeze_k))
        (self.t_res,self.x_res,self.t_left,self.x_left,self.t_right,self.x_right)= \
            [tf.compat.v1.placeholder(tf.float32,[None,1]) for _ in range(6)]
        self.water_weights = water_weights
        if water_weights is not None:
            ww, wb, wa = water_weights
            self.water_w = [tf.constant(np.array(W), tf.float32) for W in ww]
            self.water_b = [tf.constant(np.array(B), tf.float32) for B in wb]
            self.water_a = [tf.constant(np.array(A), tf.float32) for A in wa]
        self.sess=tf.compat.v1.Session(config=TF_CONFIG)
        self.phi_res,self.residual_res=self.net_res(self.t_res,self.x_res)
        Ldom = max(float(DOMAIN["xmax"] - DOMAIN["xmin"]), 1e-12)
        dphi = max(abs(float(phi_left) - float(phi_right)), 1.0)
        sigma_ref = max(abs(float(ELEC_PAR.get("sigma_sat", ELEC_PAR.get("sigma_const", 7.425e-2)))), 1e-12)
        elec_res_norm = tf.constant(float(ELEC_PAR.get("res_norm", max(sigma_ref * dphi / (Ldom * Ldom), 1e-8))), tf.float32)
        self.residual_res_nd = self.residual_res / elec_res_norm
        self.phi_left_pred=self.net_phi(tf.concat([self.t_left,self.x_left],1))
        self.phi_right_pred=self.net_phi(tf.concat([self.t_right,self.x_right],1))
        self.phi_left_true=tf.fill(tf.shape(self.phi_left_pred), tf.constant(phi_left,tf.float32))
        self.phi_right_true=tf.fill(tf.shape(self.phi_right_pred), tf.constant(phi_right,tf.float32))
        phi_scale = tf.constant(dphi, tf.float32)
        self.phi_left_pred_nd = self.phi_left_pred / phi_scale
        self.phi_left_true_nd = self.phi_left_true / phi_scale
        self.phi_right_pred_nd = self.phi_right_pred / phi_scale
        self.phi_right_true_nd = self.phi_right_true / phi_scale
        tensors={"residual_res":self.residual_res_nd,
                 "phi_left_pred":self.phi_left_pred_nd,"phi_left_true":self.phi_left_true_nd,
                 "phi_right_pred":self.phi_right_pred_nd,"phi_right_true":self.phi_right_true_nd}
        self.loss,self.loss_parts=build_loss(LOS["elec"]["terms"],tensors)

        # Optimizer with global grad clipping
        self.global_step=tf.Variable(0,trainable=False)
        lr=tf.compat.v1.train.exponential_decay(TRNC["adam_lr"], self.global_step, 1000,0.9)
        opt = tf.compat.v1.train.AdamOptimizer(lr)
        grads_vars = opt.compute_gradients(self.loss)
        grads, vars_ = zip(*grads_vars)
        grads, _ = tf.clip_by_global_norm(grads, 5.0)
        self.train_op = opt.apply_gradients(list(zip(grads, vars_)), global_step=self.global_step)

        self.lbfgs=dde.optimizers.tensorflow_compat_v1.scipy_optimizer.ScipyOptimizerInterface(
            self.loss,method='L-BFGS-B',options=TRNC["lbfgs"])
        self.sess.run(tf.compat.v1.global_variables_initializer()); self.loss_hist=[]
    def net_phi(self,X): return self.mlp.forward(X)
    def net_res(self,t,x):
        X=tf.concat([t,x],1); phi=self.net_phi(X); phi_x=tf.gradients(phi,x)[0]
        if self.water_weights is None:
            sigma=tf.fill(tf.shape(t), sigma_sat_const)
        else:
            psi = water_head_from_weights(self.water_w, self.water_b, self.water_a, t, x)
            sigma = sigma_eff_of_theta(theta_function(psi))
        return phi, tf.gradients(sigma*phi_x,x)[0]
    def train(self,N_iter,batch=True,batch_size=512, face_feed=None):
        t_left,x_left = face_feed["left"]; t_right,x_right = face_feed["right"]
        names=sorted(self.loss_parts.keys())
        for it in range(N_iter):
            idx=np.random.choice(t_res.shape[0],batch_size,replace=False); tr,xr=t_res[idx,:],x_res[idx,:]
            feed={self.t_res:tr,self.x_res:xr,self.t_left:t_left,self.x_left:x_left,self.t_right:t_right,self.x_right:x_right}
            self.sess.run(self.train_op,feed)
            if it%100==0:
                vals=self.sess.run([self.loss]+[self.loss_parts[n] for n in names],feed)
                print(f"[Elec ] It {it:5d} L={vals[0]:.3e} | "+", ".join([f"{n}={v:.2e}" for n,v in zip(names,vals[1:])]))
                self.loss_hist.append(vals[0])
        self.lbfgs.minimize(self.sess, feed_dict=feed, fetches=[self.loss]); print("[Elec ] LBFGS done.")
    def export_weights(self): return self.sess.run(self.mlp.weights), self.sess.run(self.mlp.biases), self.sess.run(self.mlp.A)

class AcidBaseNet:
    """
    Transported state: a = c_H - c_OH  [mol/m^3_water]
    Closure:
        c_H  = (a + sqrt(a^2 + 4Kw))/2
        c_OH = (-a + sqrt(a^2 + 4Kw))/2

    Boundary strategy:
    - anode (left): prescribe species-resolved non-advective Faradaic flux
        JH_F = +JF, JOH_F = 0
    - cathode (right): prescribe species-resolved non-advective Faradaic flux
        JH_F = 0, JOH_F = -JF
      This removes the acid-equivalent wrong-branch solution in which OH flux
      is used at the anode, or H flux at the cathode, to satisfy Ja = JH - JOH.
    """
    def __init__(self, layers, water_weights, elec_weights, init_from=None, freeze_k=0,
                 include_eo=True, leftBC="flux", rightBC="flux",
                 JH_left_true_val=0.0, JOH_left_true_val=0.0,
                 JH_right_true_val=0.0, JOH_right_true_val=0.0,
                 a_ic_val=0.0,
                 cH_left_dir=None, cH_right_dir=None,
                 chem_kernel=None, pb_weights=None, include_pb_source=False):
        self.mlp=DenseMLP(layers, NETC["act_scale"], NETC["trainable_layer_gain"])
        if init_from is not None:
            w0,b0,a0=init_from
            for i in range(len(self.mlp.weights)):
                self.mlp.weights[i]=tf.Variable(np.array(w0[i]), dtype=tf.float32, trainable=(i>=freeze_k))
                self.mlp.biases[i] =tf.Variable(np.array(b0[i]), dtype=tf.float32, trainable=(i>=freeze_k))
                self.mlp.A[i]      =tf.Variable(a0[i], dtype=tf.float32, trainable=(i>=freeze_k))

        self.water_w,self.water_b,self.water_a=water_weights
        self.elec_w,self.elec_b,self.elec_a=elec_weights
        self.water_w = [tf.constant(np.array(W), tf.float32) for W in self.water_w]
        self.water_b = [tf.constant(np.array(B), tf.float32) for B in self.water_b]
        self.water_a = [tf.constant(np.array(A), tf.float32) for A in self.water_a]
        self.elec_w  = [tf.constant(np.array(W), tf.float32) for W in self.elec_w]
        self.elec_b  = [tf.constant(np.array(B), tf.float32) for B in self.elec_b]
        self.elec_a  = [tf.constant(np.array(A), tf.float32) for A in self.elec_a]

        self.chem = chem_kernel
        self.include_pb_source = bool(include_pb_source)
        self.pb_w = self.pb_b = self.pb_a = None
        if pb_weights is not None:
            w_pb, b_pb, a_pb = pb_weights
            self.pb_w = [tf.constant(np.array(W), tf.float32) for W in w_pb]
            self.pb_b = [tf.constant(np.array(B), tf.float32) for B in b_pb]
            self.pb_a = [tf.constant(np.array(A), tf.float32) for A in a_pb]

        self.include_eo=bool(include_eo)
        self.leftBC=str(leftBC).lower()
        self.rightBC=str(rightBC).lower()

        (self.t_res,self.x_res,self.t_ic,self.x_ic,self.t_left,self.x_left,self.t_right,self.x_right)= \
            [tf.compat.v1.placeholder(tf.float32,[None,1]) for _ in range(8)]
        self.sess=tf.compat.v1.Session(config=TF_CONFIG)

        self.a_res, self.cH_res, self.cOH_res, self.residual_a_res = self.net_res(self.t_res,self.x_res)
        # Hard-coded neutral initial condition via the net_a ansatz; kept only for debugging.
        self.a_ic_pred = self.net_a(tf.concat([self.t_ic,self.x_ic],1))
        self.a_ic_true = tf.fill(tf.shape(self.a_ic_pred), tf.constant(a_ic_val, tf.float32))

        self.JH_left_pred  = self.net_JH(self.t_left,self.x_left)
        self.JOH_left_pred = self.net_JOH(self.t_left,self.x_left)
        self.JH_right_pred = self.net_JH(self.t_right,self.x_right)
        self.JOH_right_pred= self.net_JOH(self.t_right,self.x_right)
        self.Ja_left_pred  = self.JH_left_pred - self.JOH_left_pred
        self.Ja_right_pred = self.JH_right_pred - self.JOH_right_pred
        self.JaF_left_pred  = self.net_Ja_nonadv(self.t_left,self.x_left)
        self.JaF_right_pred = self.net_Ja_nonadv(self.t_right,self.x_right)
        self.JH_F_left_pred, self.JOH_F_left_pred = self.net_species_nonadv(self.t_left,self.x_left)
        self.JH_F_right_pred, self.JOH_F_right_pred = self.net_species_nonadv(self.t_right,self.x_right)

        self.c_left_pred  = self.net_c(tf.concat([self.t_left,self.x_left],1))
        self.c_right_pred = self.net_c(tf.concat([self.t_right,self.x_right],1))
        self.cOH_left_pred  = self.net_cOH(tf.concat([self.t_left,self.x_left],1))
        self.cOH_right_pred = self.net_cOH(tf.concat([self.t_right,self.x_right],1))
        self.pH_left_pred  = self.net_pH_unclipped(tf.concat([self.t_left,self.x_left],1))
        self.pH_right_pred = self.net_pH_unclipped(tf.concat([self.t_right,self.x_right],1))

        ramp_left = self._faraday_ramp(self.t_left)
        ramp_right = self._faraday_ramp(self.t_right)
        dynamic_faraday = str(H_BC.get("faraday_current_mode", "fixed_current")).lower() == "archie_voltage"
        if dynamic_faraday:
            Jmag_left = self._faraday_flux_from_archie_current(self.t_left, self.x_left)
            Jmag_right = self._faraday_flux_from_archie_current(self.t_right, self.x_right)
            self.JH_left_true = Jmag_left * ramp_left
            self.JOH_left_true = tf.zeros_like(self.JH_left_true)
            self.JH_right_true = tf.zeros_like(Jmag_right)
            self.JOH_right_true = -Jmag_right * ramp_right
        else:
            sat_left = self._faraday_saturation_scale(self.t_left, self.x_left)
            sat_right = self._faraday_saturation_scale(self.t_right, self.x_right)
            self.JH_left_true   = tf.fill(tf.shape(self.JH_left_pred), tf.constant(JH_left_true_val, tf.float32)) * ramp_left * sat_left
            self.JOH_left_true  = tf.fill(tf.shape(self.JOH_left_pred), tf.constant(JOH_left_true_val, tf.float32)) * ramp_left * sat_left
            self.JH_right_true  = tf.fill(tf.shape(self.JH_right_pred), tf.constant(JH_right_true_val, tf.float32)) * ramp_right * sat_right
            self.JOH_right_true = tf.fill(tf.shape(self.JOH_right_pred), tf.constant(JOH_right_true_val, tf.float32)) * ramp_right * sat_right
        self.Ja_left_true   = self.JH_left_true - self.JOH_left_true
        self.Ja_right_true  = self.JH_right_true - self.JOH_right_true
        Ldom = max(float(DOMAIN["xmax"] - DOMAIN["xmin"]), 1e-12)
        eta_i = float(H_BC.get("current_efficiency", 1.0))
        dphi = abs(float(ELEC_BD["phi_anode"]) - float(ELEC_BD["phi_cathode"]))
        Ja_ref_val = max(eta_i * float(ELEC_PAR.get("sigma_sat", ELEC_PAR.get("sigma_const", 7.425e-2))) * dphi / Ldom / float(CFG["GLOBAL"]["CONSTANTS"]["F"]) * SEC_PER_DAY, 1e-8)
        self.res_a_ref = tf.fill(tf.shape(self.residual_a_res), tf.constant(Ja_ref_val / Ldom, tf.float32))

        if cH_left_dir is None:
            cH_left_dir = pH_to_c_m3(H_BC.get("pH_left", H_INIT["pH_ic"]))
        if cH_right_dir is None:
            cH_right_dir = pH_to_c_m3(H_BC.get("pH_right", H_INIT["pH_ic"]))
        self.pH_left_true = self._boundary_pH_target(self.t_left, "left")
        self.pH_right_true = self._boundary_pH_target(self.t_right, "right")
        self.c_left_true = self._pH_to_cH_tensor(self.pH_left_true)
        self.c_right_true = self._pH_to_cH_tensor(self.pH_right_true)
        self.a_left_pred = self.net_a(tf.concat([self.t_left, self.x_left], 1))
        self.a_right_pred = self.net_a(tf.concat([self.t_right, self.x_right], 1))
        self.a_left_true = self._a_from_pH_tensor(self.pH_left_true)
        self.a_right_true = self._a_from_pH_tensor(self.pH_right_true)
        self.a_bc_ref = tf.constant(float(H_PAR.get("a_dirichlet_loss_scale", 10.0)), tf.float32)

        def _finite(z):
            return tf.where(tf.math.is_finite(z), z, tf.zeros_like(z))
        def _mse(z):
            return tf.reduce_mean(tf.square(z))
        def _rel_mse(err, ref):
            return tf.reduce_mean(tf.square(err)) / (tf.reduce_mean(tf.square(ref)) + 1e-12)

        self.residual_a_nd = _finite(self.residual_a_res / tf.maximum(self.res_a_ref, 1e-12))
        self.residual_weight_ph = tf.compat.v1.placeholder_with_default(
            tf.constant(float(H_PAR.get("residual_weight", 1.0)), tf.float32), shape=())
        self.faraday_flux_weight_ph = tf.compat.v1.placeholder_with_default(
            tf.constant(float(H_PAR.get("faraday_flux_weight", 140.0)), tf.float32), shape=())
        self.coion_flux_weight_ph = tf.compat.v1.placeholder_with_default(
            tf.constant(float(H_PAR.get("coion_flux_weight", 0.0)), tf.float32), shape=())
        loss_parts = {
            "res_a": self.residual_weight_ph * _mse(self.residual_a_nd),
        }

        # Species-resolved electrode boundary conditions.
        # The transported state is still a = cH - cOH, but the Faradaic boundary
        # is no longer imposed only through Ja_F = JH_F - JOH_F.  We constrain
        # the two non-advective species fluxes separately so the network cannot
        # use the wrong co-ion branch to satisfy the acid-equivalent flux:
        #   anode:   JH_F = +JF,  JOH_F = 0
        #   cathode: JH_F = 0,    JOH_F = -JF
        # Advective transport is supplied by the separately trained water field.
        faraday_mode = str(H_PAR.get("faraday_bc_mode", "acid_equivalent")).lower()
        pH_loss_scale = tf.constant(float(H_BC.get("pH_loss_scale", 10.0)), tf.float32)
        pH_dirichlet_weight = float(H_BC.get("pH_dirichlet_weight", 30.0))
        if self.leftBC == "dirichlet":
            loss_parts["left_dirichlet_a"] = pH_dirichlet_weight * _mse(_finite((self.a_left_pred - self.a_left_true) / self.a_bc_ref))
        elif faraday_mode in ("acid_equivalent", "species", "species_flux", "species_nonadv"):
            loss_parts["left_HF_flux"] = self.faraday_flux_weight_ph * _rel_mse(
                _finite(self.JH_F_left_pred - self.JH_left_true), self.JH_left_true
            )
            if float(H_PAR.get("coion_flux_weight", 0.0)) > 0.0:
                loss_parts["left_OHF_zero"] = self.coion_flux_weight_ph * _rel_mse(
                    _finite(self.JOH_F_left_pred - self.JOH_left_true), self.JH_left_true
                )
        else:
            loss_parts["left_H_flux"]  = self.faraday_flux_weight_ph * _rel_mse(_finite(self.JH_left_pred  - self.JH_left_true),  self.JH_left_true)
            loss_parts["left_OH_flux"] = 40.0  * _mse(_finite(self.JOH_left_pred - self.JOH_left_true))

        if self.rightBC == "dirichlet":
            loss_parts["right_dirichlet_a"] = pH_dirichlet_weight * _mse(_finite((self.a_right_pred - self.a_right_true) / self.a_bc_ref))
        elif faraday_mode in ("acid_equivalent", "species", "species_flux", "species_nonadv"):
            if float(H_PAR.get("coion_flux_weight", 0.0)) > 0.0:
                loss_parts["right_HF_zero"] = self.coion_flux_weight_ph * _rel_mse(
                    _finite(self.JH_F_right_pred - self.JH_right_true), self.JOH_right_true
                )
            loss_parts["right_OHF_flux"] = self.faraday_flux_weight_ph * _rel_mse(
                _finite(self.JOH_F_right_pred - self.JOH_right_true), self.JOH_right_true
            )
        else:
            loss_parts["right_H_flux"]  = 40.0  * _mse(_finite(self.JH_right_pred  - self.JH_right_true))
            loss_parts["right_OH_flux"] = self.faraday_flux_weight_ph * _rel_mse(_finite(self.JOH_right_pred - self.JOH_right_true), self.JOH_right_true)

        pH_w = float(H_PAR.get("pH_obs_weight", 0.0))
        if pH_w > 0.0:
            loss_parts["weak_pH_left"]  = pH_w * _mse(_finite(self.pH_left_pred  - self.pH_left_true))
            loss_parts["weak_pH_right"] = pH_w * _mse(_finite(self.pH_right_pred - self.pH_right_true))

        # Physics-only admissibility hinge: electrolysis should make the anode acidic
        # and cathode alkaline. Ramp the hinge with the Faradaic ramp so it does not
        # conflict with the neutral initial condition at t=0.
        pH_range_w = float(H_PAR.get("pH_range_weight", 0.0))
        if pH_range_w > 0.0:
            pH_left_max = tf.constant(float(H_PAR.get("pH_left_max_phys", 4.0)), tf.float32)
            pH_right_min = tf.constant(float(H_PAR.get("pH_right_min_phys", 10.0)), tf.float32)
            if bool(H_PAR.get("pH_range_ramped", True)):
                pH0 = tf.constant(float(H_INIT.get("pH_ic", 7.0)), tf.float32)
                left_limit = pH0 - (pH0 - pH_left_max) * ramp_left
                right_limit = pH0 + (pH_right_min - pH0) * ramp_right
            else:
                left_limit = pH_left_max
                right_limit = pH_right_min
            loss_parts["phys_pH_left_acid"] = pH_range_w * _mse(tf.nn.relu(_finite(self.pH_left_pred) - left_limit))
            loss_parts["phys_pH_right_alk"] = pH_range_w * _mse(tf.nn.relu(right_limit - _finite(self.pH_right_pred)))

        self.loss_parts = loss_parts
        self.loss = tf.add_n(list(self.loss_parts.values()))

        # backward-compatible aliases for older cells
        self.c_res = self.cH_res
        self.residual_res = self.residual_a_res
        self.J_left_pred = self.Ja_left_pred
        self.J_right_pred = self.Ja_right_pred
        self.c_ic_pred = self.net_c(tf.concat([self.t_ic,self.x_ic],1))
        self.c_ic_true = tf.fill(tf.shape(self.c_ic_pred), tf.constant(float(H_INIT["c_ic"]), tf.float32))

        self.global_step=tf.Variable(0,trainable=False)
        base_lr = float(TRNC.get("hplus_adam_lr", TRNC["adam_lr"]))
        lr=tf.compat.v1.train.exponential_decay(base_lr, self.global_step, 1000,0.9)
        opt = tf.compat.v1.train.AdamOptimizer(lr)
        vars_all = [v for v in (self.mlp.weights + self.mlp.biases + self.mlp.A) if getattr(v, "trainable", True)]
        grads_vars = [(g, v) for g, v in opt.compute_gradients(self.loss, var_list=vars_all) if g is not None]
        if not grads_vars:
            raise ValueError("AcidBaseNet has no trainable gradients; check freeze_k_c and net_a.")
        grads, vars_ = zip(*grads_vars)
        safe_grads, finite_counts, total_counts = [], [], []
        for g in grads:
            g = tf.convert_to_tensor(g)
            finite_mask = tf.math.is_finite(g)
            finite_counts.append(tf.reduce_sum(tf.cast(finite_mask, tf.float32)))
            total_counts.append(tf.cast(tf.size(g), tf.float32))
            safe_grads.append(tf.where(finite_mask, g, tf.zeros_like(g)))
        self.grad_finite_frac = tf.add_n(finite_counts) / tf.maximum(tf.add_n(total_counts), 1.0)
        self.grad_norm = tf.linalg.global_norm(safe_grads)
        safe_grads, _ = tf.clip_by_global_norm(safe_grads, float(TRNC.get("hplus_grad_clip", 5.0)))
        self.train_op = opt.apply_gradients(list(zip(safe_grads, vars_)), global_step=self.global_step)
        self.lbfgs=dde.optimizers.tensorflow_compat_v1.scipy_optimizer.ScipyOptimizerInterface(
            self.loss,method='L-BFGS-B',options=TRNC["lbfgs"])
        self.sess.run(tf.compat.v1.global_variables_initializer())
        self.loss_hist=[]
        # Python-side adaptive loss scales used only as feed_dict multipliers.
        # They do not change the graph structure and are reset for every AcidBaseNet instance.
        self._adapt_scale_res = 1.0
        self._adapt_scale_flux = 1.0
        self._adapt_scale_coion = 1.0
        self._res_cache = None
        self._res_cache_meta = None
        self._best_loss = np.inf
        self._best_weights = None
        self._best_it = None
        self._zero_grad_warned = False

    def _faraday_ramp(self, t):
        tau = tf.constant(float(H_PAR.get("faraday_ramp_tau", 0.0)), tf.float32)
        t0 = tf.constant(float(CFG["DOMAIN"]["tmin"]), tf.float32)
        if float(H_PAR.get("faraday_ramp_tau", 0.0)) <= 0.0:
            return tf.ones_like(t)
        return 1.0 - tf.exp(-tf.maximum(t - t0, 0.0) / tf.maximum(tau, 1e-12))

    def _faraday_saturation_scale(self, t, x):
        return tf.ones_like(t)

    def _faraday_flux_from_archie_current(self, t, x):
        psi = water_head_from_weights(self.water_w, self.water_b, self.water_a, t, x)
        theta = theta_function(psi)
        _, phi_x = self._phi_grad(t, x)
        sigma_eff = sigma_eff_of_theta(theta)
        eta_i = tf.constant(float(H_BC.get("current_efficiency", 1.0)), tf.float32)
        i_mag = tf.abs(-sigma_eff * phi_x)  # A/m^2 under prescribed voltage
        J_day = eta_i * i_mag / F_c * tf.constant(SEC_PER_DAY, tf.float32)
        return tf.where(tf.math.is_finite(J_day), J_day, tf.zeros_like(J_day))

    def _faraday_flux_from_initial_theta_current(self, t):
        Ldom = max(float(DOMAIN["xmax"] - DOMAIN["xmin"]), 1e-12)
        dphi = abs(float(ELEC_BD["phi_anode"]) - float(ELEC_BD["phi_cathode"]))
        if "_initial_sigma_eff_num" in globals():
            sigma = float(_initial_sigma_eff_num())
        else:
            sigma = float(ELEC_PAR.get("sigma_sat", ELEC_PAR.get("sigma_const", 7.425e-2)))
        eta_i = float(H_BC.get("current_efficiency", 1.0))
        J_const = eta_i * sigma * dphi / Ldom / float(CFG["GLOBAL"]["CONSTANTS"]["F"]) * SEC_PER_DAY
        return tf.fill(tf.shape(t), tf.constant(J_const, tf.float32))

    def _a_from_pH_tensor(self, pH):
        ln10 = tf.constant(np.log(10.0), tf.float32)
        cH = 1000.0 * tf.exp(-ln10 * pH)
        cOH = Kw_const / tf.maximum(cH, 1e-30)
        return cH - cOH

    def _pH_to_cH_tensor(self, pH):
        ln10 = tf.constant(np.log(10.0), tf.float32)
        return 1000.0 * tf.exp(-ln10 * pH)

    def _boundary_pH_target(self, t, side):
        mode = str(H_BC.get("dirichlet_mode", "static")).lower()
        key = "pH_left" if side == "left" else "pH_right"
        pH0 = tf.constant(float(H_INIT.get("pH_ic", 7.0)), tf.float32)
        if mode not in ("reservoir", "reservoir_ph"):
            return tf.fill(tf.shape(t), tf.constant(float(H_BC.get(key, H_INIT.get("pH_ic", 7.0))), tf.float32))

        Ldom = max(float(DOMAIN["xmax"] - DOMAIN["xmin"]), 1e-12)
        if str(H_BC.get("faraday_current_mode", "fixed_current")).lower() == "archie_voltage" and H_BC.get("I_app_Aperm2") is None:
            J_day = self._faraday_flux_from_initial_theta_current(t)
        else:
            if H_BC.get("I_app_Aperm2") is None:
                dphi = abs(float(ELEC_BD["phi_anode"]) - float(ELEC_BD["phi_cathode"]))
                Iapp = float(ELEC_PAR.get("sigma_sat", ELEC_PAR.get("sigma_const", 7.425e-2))) * dphi / Ldom
            else:
                Iapp = float(H_BC["I_app_Aperm2"])
            J_const = float(H_BC.get("current_efficiency", 1.0)) * Iapp / float(CFG["GLOBAL"]["CONSTANTS"]["F"]) * SEC_PER_DAY
            J_day = tf.fill(tf.shape(t), tf.constant(J_const, tf.float32))
        depth = max(float(H_BC.get("reservoir_depth_m", Ldom)), 1e-12)
        tday = tf.maximum(t - tf.constant(float(DOMAIN["tmin"]), tf.float32), 0.0)
        c0_H = float(H_INIT.get("c_ic", pH_to_c_m3(H_INIT.get("pH_ic", 7.0))))
        c0_OH = float(CFG["GLOBAL"]["CHEM"]["Kw"]) / max(c0_H, 1e-30)
        c = tf.constant(c0_H if side == "left" else c0_OH, tf.float32) + (J_day / tf.constant(depth, tf.float32)) * tday
        ln10 = tf.constant(np.log(10.0), tf.float32)
        if side == "left":
            raw = -tf.math.log(tf.maximum(c / 1000.0, 1e-14)) / ln10
        else:
            raw = 14.0 + tf.math.log(tf.maximum(c / 1000.0, 1e-14)) / ln10
        tau = tf.constant(float(H_BC.get("reservoir_pH_ramp_tau", H_PAR.get("faraday_ramp_tau", 0.01))), tf.float32)
        ramp = 1.0 - tf.exp(-tday / tf.maximum(tau, 1e-12))
        target = pH0 + ramp * (raw - pH0)
        if bool(H_BC.get("reservoir_clip_to_config", True)):
            limit = tf.constant(float(H_BC.get(key, H_INIT.get("pH_ic", 7.0))), tf.float32)
            target = tf.maximum(target, limit) if side == "left" else tf.minimum(target, limit)
        return tf.clip_by_value(target, float(H_PAR.get("pH_min", 0.0)), float(H_PAR.get("pH_max", 14.5)))

    def net_a(self, X):
        # Do not impose a left-acid/right-alkaline profile in the interior.
        # The acid/base front should be learned from PDE + Faraday boundary fluxes.
        raw0 = self.mlp.forward(X)
        raw = 8.0 * tf.tanh(raw0 / 8.0)
        a_scale = tf.constant(float(H_PAR.get("a_raw_scale", 20.0)), tf.float32)

        t = X[:, 0:1]
        t0 = tf.constant(float(CFG["DOMAIN"]["tmin"]), tf.float32)
        tau_gate = tf.constant(float(H_PAR.get("a_ic_gate_tau", 0.02)), tf.float32)
        gate = tf.where(
            t <= t0,
            tf.zeros_like(t),
            1.0 - tf.exp(-(t - t0) / tf.maximum(tau_gate, 1e-12)),
        )
        a_ic = tf.constant(float(H_INIT.get("a_ic", 0.0)), tf.float32)
        a_free = a_ic + a_scale * raw
        a = (1.0 - gate) * a_ic + gate * a_free
        return tf.where(tf.math.is_finite(a), a, tf.zeros_like(a))

    def acid_base_from_a(self, a):
        # Stable algebraic Kw closure for a = cH - cOH and cH*cOH = Kw.
        s = tf.sqrt(tf.square(a) + 4.0 * Kw_const)
        cH_pos = 0.5 * (a + s)
        cH_neg = (2.0 * Kw_const) / tf.maximum(s - a, 1e-30)
        cOH_neg = 0.5 * (-a + s)
        cOH_pos = (2.0 * Kw_const) / tf.maximum(s + a, 1e-30)
        cH = tf.where(a >= 0.0, cH_pos, cH_neg)
        cOH = tf.where(a <= 0.0, cOH_neg, cOH_pos)
        cH = tf.where(tf.math.is_finite(cH), cH, tf.zeros_like(cH))
        cOH = tf.where(tf.math.is_finite(cOH), cOH, tf.zeros_like(cOH))
        return cH, cOH

    def net_c(self, X):
        cH, _ = self.acid_base_from_a(self.net_a(X))
        return cH

    def net_cOH(self, X):
        _, cOH = self.acid_base_from_a(self.net_a(X))
        return cOH

    def net_pH_unclipped(self, X):
        cH = tf.maximum(self.net_c(X), 1e-30)
        ln10 = tf.constant(np.log(10.0), tf.float32)
        pH = -tf.math.log(cH / 1000.0) / ln10
        return tf.where(tf.math.is_finite(pH), pH, tf.zeros_like(pH))

    def net_pH(self, X):
        pH = self.net_pH_unclipped(X)
        pH_min = tf.constant(float(H_PAR.get("pH_min", 0.0)), tf.float32)
        pH_max = tf.constant(float(H_PAR.get("pH_max", 14.5)), tf.float32)
        return tf.clip_by_value(pH, pH_min, pH_max)

    def _water_fields(self,t,x):
        psi=water_head_from_weights(self.water_w, self.water_b, self.water_a, t, x)
        theta=theta_function(psi)
        K=K_function(psi)
        psi_x=grad0(psi,x)
        q_hyd=-K*psi_x
        return theta,q_hyd

    def _phi_grad(self,t,x):
        H=nondim_tx(t,x)
        for l in range(len(self.elec_w)):
            H=tf.add(tf.matmul(H,self.elec_w[l]), self.elec_b[l])
            if l<len(self.elec_w)-1:
                H=tf.tanh(NETC["act_scale"]*self.elec_a[l]*H)
        phi=H
        return phi, grad0(phi,x)

    def _species_fields(self,t,x):
        X=tf.concat([t,x],1)
        a = self.net_a(X)
        cH, cOH = self.acid_base_from_a(a)
        cHx = grad0(cH, x)
        cOHx = grad0(cOH, x)
        theta,q_hyd=self._water_fields(t,x)
        _,phi_x=self._phi_grad(t,x)
        q_adv=q_hyd + (keo_of_theta(theta)*phi_x if self.include_eo else 0.0)
        return a,cH,cOH,cHx,cOHx,theta,q_adv,phi_x

    def net_JH(self,t,x):
        a,cH,cOH,cHx,cOHx,theta,q_adv,phi_x = self._species_fields(t,x)
        Deff_H, u_H = diffusion_pieces(theta, q_adv, DL_H, Dw_H, z_H)
        JH = q_adv*cH - Deff_H*cHx - u_H*cH*phi_x
        return tf.where(tf.math.is_finite(JH), JH, tf.zeros_like(JH))

    def net_JOH(self,t,x):
        a,cH,cOH,cHx,cOHx,theta,q_adv,phi_x = self._species_fields(t,x)
        Deff_OH, u_OH = diffusion_pieces(theta, q_adv, DL_OH, Dw_OH, z_OH)
        JOH = q_adv*cOH - Deff_OH*cOHx - u_OH*cOH*phi_x
        return tf.where(tf.math.is_finite(JOH), JOH, tf.zeros_like(JOH))

    def net_Ja(self,t,x):
        return self.net_JH(t,x) - self.net_JOH(t,x)

    def net_species_nonadv(self,t,x):
        a,cH,cOH,cHx,cOHx,theta,q_adv,phi_x = self._species_fields(t,x)
        Deff_H, u_H = diffusion_pieces(theta, q_adv, DL_H, Dw_H, z_H)
        Deff_OH, u_OH = diffusion_pieces(theta, q_adv, DL_OH, Dw_OH, z_OH)
        JH_F = -Deff_H*cHx - u_H*cH*phi_x
        JOH_F = -Deff_OH*cOHx - u_OH*cOH*phi_x
        finite = lambda z: tf.where(tf.math.is_finite(z), z, tf.zeros_like(z))
        return finite(JH_F), finite(JOH_F)

    def net_Ja_nonadv(self,t,x):
        # Faraday boundary constrains only non-advective electrochemical flux:
        # Ja_F = Ja - q*a = (JH-JOH) - q*(cH-cOH).
        JH_F, JOH_F = self.net_species_nonadv(t,x)
        JaF = JH_F - JOH_F
        return tf.where(tf.math.is_finite(JaF), JaF, tf.zeros_like(JaF))

    def net_J(self,t,x):
        # compatibility: return H+ flux
        return self.net_JH(t,x)

    def _pb_psi_from_weights(self, t, x):
        if self.pb_w is None:
            return None
        H=nondim_tx(t,x)
        for l in range(len(self.pb_w)):
            H = tf.add(tf.matmul(H, self.pb_w[l]), self.pb_b[l])
            if l < len(self.pb_w) - 1:
                H = tf.tanh(NETC["act_scale"] * self.pb_a[l] * H)

        # Use the same Psi_Pb ansatz as PbInvNet.net_Psi. This makes the
        # acid/base coupling source see the same transported Pb inventory as
        # the Pb transport model, including the hard initial condition.
        raw_scale = tf.constant(float(PB_PAR.get("psi_raw_scale", 20.0)), tf.float32)
        raw = raw_scale * tf.tanh(H[:, 0:1] / tf.maximum(raw_scale, 1e-6))

        Hw = water_head_from_weights(self.water_w, self.water_b, self.water_a, t, x)
        theta0 = theta_function(Hw)
        psi0_store = tf.constant(float(SURF_INIT["SOPb0"] + SURF_INIT["Pp0"]), tf.float32)
        c_ic_water = tf.constant(float(CFG["FIELDS"]["PB"]["INITIAL"]["c_ic"]), tf.float32)
        Psi0 = tf.maximum(theta0 * c_ic_water + psi0_store, 1e-6)
        eta0 = tf.math.log(tf.math.expm1(Psi0) + 1e-12)

        t0 = tf.constant(float(CFG["DOMAIN"]["tmin"]), tf.float32)
        tau = tf.constant(float(PB_PAR.get("psi_ic_gate_tau", 0.02)), tf.float32)
        gate = tf.where(t <= t0, tf.zeros_like(t), 1.0 - tf.exp(-(t - t0) / tf.maximum(tau, 1e-6)))
        Psi = tf.nn.softplus(eta0 + gate * raw)
        if self.chem is not None:
            Psi = tf.clip_by_value(Psi, 0.0, self.chem.S_tot + 1.0e4)
        return tf.where(tf.math.is_finite(Psi), Psi, tf.zeros_like(Psi))

    def _pb_acid_source(self, t, x, theta_loc, cH_w, cOH_w):
        """Frozen previous-iterate Pb chemistry source for one acid/base block."""
        if (not self.include_pb_source) or (self.chem is None) or (self.pb_w is None):
            return tf.zeros_like(t)
        Psi_b = self._pb_psi_from_weights(t, x)
        if Psi_b is None:
            return tf.zeros_like(t)
        # Kim-style coupled source: previous Pb fixes CT and the active Ksp
        # branch; precipitated Pb still responds to the current pH field.
        *_, active_gate = self.chem.reconstruct_pb_equil(t, x, Psi_b, return_gate=True)
        _, _, _, _, _, Pp, _ = self.chem.reconstruct_pb_equil_from_fields(
            theta_loc, cH_w, cOH_w, Psi_b, t,
            precip_gate_override=active_gate > 0.5,
            stop_acid=bool(H_PAR.get("pb_source_stop_gradient", True)),
        )
        S_ab = 2.0 * grad0(Pp, t)
        S_ab = tf.where(tf.math.is_finite(S_ab), S_ab, tf.zeros_like(S_ab))
        if bool(H_PAR.get("pb_source_stop_gradient", True)):
            S_ab = tf.stop_gradient(S_ab)
        return S_ab

    def net_res(self,t,x):
        a,cH,cOH,cHx,cOHx,theta,q_adv,phi_x = self._species_fields(t,x)
        Deff_H,  u_H  = diffusion_pieces(theta, q_adv, DL_H,  Dw_H,  z_H)
        Deff_OH, u_OH = diffusion_pieces(theta, q_adv, DL_OH, Dw_OH, z_OH)
        JH  = q_adv*cH  - Deff_H*cHx   - u_H*cH*phi_x
        JOH = q_adv*cOH - Deff_OH*cOHx - u_OH*cOH*phi_x
        Ja = JH - JOH
        theta_t = grad0(theta, t)
        cH_t = grad0(cH, t)
        cOH_t = grad0(cOH, t)
        S_pb = self._pb_acid_source(t, x, theta, cH, cOH)
        RH = tf.constant(float(H_PAR.get("H_retardation", 1.0)), tf.float32)
        # Kim Eq. (11)/(13), extended to variable water content.
        storage_H = RH * (theta * cH_t + theta_t * cH)
        storage_OH = theta * cOH_t + theta_t * cOH
        storage = storage_H - storage_OH
        res = storage + grad0(Ja,x) - S_pb
        finite = lambda z: tf.where(tf.math.is_finite(z), z, tf.zeros_like(z))
        return finite(a), finite(cH), finite(cOH), finite(res)

    def _weight_feed(self, it, N_iter):
        """Return feed_dict weights. Base weights follow the existing schedule;
        optional adaptive scales equalize weighted loss-group contributions."""
        frac = float(it + 1) / max(float(N_iter), 1.0)
        schedule = H_PAR.get("training_schedule", None)
        res_w = float(H_PAR.get("residual_weight", 1.0))
        flux_w = float(H_PAR.get("faraday_flux_weight", 140.0))
        if schedule:
            for stage in schedule:
                if frac <= float(stage.get("until", 1.0)):
                    res_w = float(stage.get("residual_weight", res_w))
                    flux_w = float(stage.get("faraday_flux_weight", flux_w))
                    break
        coion_w = float(H_PAR.get("coion_flux_weight", 0.0))

        if bool(H_PAR.get("adaptive_loss_weights", True)):
            res_w *= float(self._adapt_scale_res)
            flux_w *= float(self._adapt_scale_flux)
            coion_w *= float(self._adapt_scale_coion)

        # The co-ion zero-flux constraints are branch-selection constraints,
        # not ordinary residual terms.  Do not let adaptive weighting weaken
        # them below the configured floor.
        if coion_w > 0.0:
            coion_floor = float(H_PAR.get("coion_flux_weight_min", H_PAR.get("coion_flux_weight", 0.0)))
            coion_w = max(coion_w, coion_floor)

        self._last_weight_values = (res_w, flux_w, coion_w)
        return {
            self.residual_weight_ph: res_w,
            self.faraday_flux_weight_ph: flux_w,
            self.coion_flux_weight_ph: coion_w,
        }, res_w, flux_w, coion_w

    def _sample_uniform_tx(self, n, t_low=None, t_high=None):
        xmin, xmax = float(DOMAIN["xmin"]), float(DOMAIN["xmax"])
        tmin = float(DOMAIN["tmin"]) if t_low is None else float(t_low)
        tmax = float(DOMAIN["tmax"]) if t_high is None else float(t_high)
        tmax = max(tmax, tmin + 1e-9)
        tr = np.random.uniform(tmin, tmax, (int(n), 1)).astype(np.float32)
        xr = np.random.uniform(xmin, xmax, (int(n), 1)).astype(np.float32)
        return tr, xr

    def _sample_face_window(self, template_t, template_x, t_low=None, t_high=None):
        """Sample boundary points in the current time slab while preserving face x."""
        n = int(template_t.shape[0])
        tmin = float(DOMAIN["tmin"]) if t_low is None else float(t_low)
        tmax = float(DOMAIN["tmax"]) if t_high is None else float(t_high)
        tmax = max(tmax, tmin + 1e-9)
        tr = np.random.uniform(tmin, tmax, (n, 1)).astype(np.float32)
        xval = float(np.asarray(template_x[:1]).reshape(-1)[0])
        xr = np.full((n, 1), xval, dtype=np.float32)
        return tr, xr

    def _refresh_residual_cache(self, t_low=None, t_high=None):
        """Residual-based adaptive refinement (RAR/RBAS).

        A large candidate pool is sampled uniformly inside the active time window.
        The current AcidBase PDE residual is evaluated on the candidates, and the
        highest-|residual| points are cached.  The training batch then draws most
        collocation points from this cache, plus a small uniform background.
        No governing equation, boundary condition, or chemistry expression is changed.
        """
        n_cand = int(H_PAR.get("residual_cache_candidates", 8192))
        n_keep = int(H_PAR.get("residual_cache_keep", max(512, n_cand // 2)))
        n_cand = max(1, n_cand)
        n_keep = max(1, min(n_keep, n_cand))
        tc, xc = self._sample_uniform_tx(n_cand, t_low, t_high)
        try:
            res_val = self.sess.run(
                self.residual_a_res,
                {self.t_res: tc, self.x_res: xc}
            )
            res_abs = np.abs(np.asarray(res_val).reshape(-1))
            res_abs = np.where(np.isfinite(res_abs), res_abs, 0.0)
            power = float(H_PAR.get("residual_score_power", 1.0))
            power = max(power, 1e-6)
            score = np.power(res_abs + 1e-30, power)
            if np.all(score <= 0.0):
                keep = np.random.choice(n_cand, n_keep, replace=False)
            else:
                keep = np.argpartition(score, -n_keep)[-n_keep:]
            self._res_cache = (tc[keep, :].astype(np.float32), xc[keep, :].astype(np.float32))
            self._res_cache_score = score[keep].astype(np.float64)
            self._res_cache_meta = (float(t_low) if t_low is not None else None,
                                    float(t_high) if t_high is not None else None)
        except Exception as exc:
            print(f"[A/B RAR] residual cache refresh skipped: {exc}")
            self._res_cache = None
            self._res_cache_score = None
            self._res_cache_meta = None

    def _residual_batch(self, batch_size, t_low=None, t_high=None, it=0):
        """Build a residual-collocation batch for the active time slab.

        If adaptive sampling is enabled, the batch is:
            residual-adaptive points from the high-|PDE residual| cache
            + uniform background points.
        This is true residual-based adaptive sampling; electrode/early-time
        heuristics are intentionally not used here.
        """
        tmin = float(DOMAIN["tmin"]) if t_low is None else float(t_low)
        tmax = float(DOMAIN["tmax"]) if t_high is None else float(t_high)
        tmax = max(tmax, tmin + 1e-9)
        batch_size = int(batch_size)

        if (not bool(H_PAR.get("adaptive_residual_sampling", True))) or batch_size <= 0:
            return self._sample_uniform_tx(batch_size, tmin, tmax)

        adapt_frac = float(H_PAR.get("residual_adaptive_frac", 0.70))
        uniform_frac = float(H_PAR.get("residual_uniform_frac", max(0.0, 1.0 - adapt_frac)))
        adapt_frac = min(max(adapt_frac, 0.0), 1.0)
        uniform_frac = min(max(uniform_frac, 0.0), 1.0)
        # Normalize if the two fractions were edited inconsistently.
        sfrac = adapt_frac + uniform_frac
        if sfrac <= 0.0:
            return self._sample_uniform_tx(batch_size, tmin, tmax)
        adapt_frac /= sfrac
        n_adapt = int(round(batch_size * adapt_frac))
        n_adapt = max(0, min(batch_size, n_adapt))
        n_uniform = batch_size - n_adapt

        ts, xs = [], []

        # Uniform background coverage.
        if n_uniform > 0:
            tr_u, xr_u = self._sample_uniform_tx(n_uniform, tmin, tmax)
            ts.append(tr_u); xs.append(xr_u)

        # High-residual adaptive points.
        if n_adapt > 0:
            refresh_every = int(H_PAR.get("residual_cache_refresh", 50))
            meta = (float(tmin), float(tmax))
            cache_missing = self._res_cache is None or self._res_cache_meta != meta
            if cache_missing or (refresh_every > 0 and it % refresh_every == 0):
                self._refresh_residual_cache(tmin, tmax)
            if self._res_cache is not None:
                ct, cx = self._res_cache
                idx = np.random.choice(ct.shape[0], n_adapt, replace=(ct.shape[0] < n_adapt))
                ts.append(ct[idx, :]); xs.append(cx[idx, :])
            else:
                tr_a, xr_a = self._sample_uniform_tx(n_adapt, tmin, tmax)
                ts.append(tr_a); xs.append(xr_a)

        tr = np.vstack(ts).astype(np.float32)
        xr = np.vstack(xs).astype(np.float32)
        perm = np.random.permutation(tr.shape[0])
        return tr[perm, :], xr[perm, :]

    def _update_adaptive_loss_scales(self, names, vals):
        """Python-side adaptive weighting based on current weighted loss groups.
        It balances residual / Faradaic-flux / co-ion diagnostic groups without
        changing the governing equations."""
        if not bool(H_PAR.get("adaptive_loss_weights", True)):
            return
        groups = {"res": 0.0, "flux": 0.0, "coion": 0.0}
        for name, val in zip(names, vals):
            v = float(val)
            if not np.isfinite(v):
                continue
            if name == "res_a":
                groups["res"] += max(v, 0.0)
            elif (("JaF_flux" in name) or ("HF_flux" in name) or ("OHF_flux" in name)
                  or ("_H_flux" in name) or ("_OH_flux" in name)):
                groups["flux"] += max(v, 0.0)
            elif ("HF_zero" in name) or ("OHF_zero" in name):
                groups["coion"] += max(v, 0.0)

        active = {k: v for k, v in groups.items() if v > 1e-16}
        if len(active) < 2:
            return
        logs = [np.log(v + 1e-16) for v in active.values()]
        target = float(np.exp(np.mean(logs)))
        alpha = float(H_PAR.get("adaptive_loss_alpha", 0.35))
        min_s = float(H_PAR.get("adaptive_loss_min_scale", 0.2))
        max_s = float(H_PAR.get("adaptive_loss_max_scale", 5.0))

        def _upd(current, value):
            factor = (target / (value + 1e-16)) ** alpha
            return float(np.clip(current * factor, min_s, max_s))

        if "res" in active:
            self._adapt_scale_res = _upd(self._adapt_scale_res, active["res"])
        if "flux" in active:
            self._adapt_scale_flux = _upd(self._adapt_scale_flux, active["flux"])
        if "coion" in active:
            self._adapt_scale_coion = _upd(self._adapt_scale_coion, active["coion"])

    def _remember_best(self, loss_value, it_label):
        loss_value = float(loss_value)
        if np.isfinite(loss_value) and loss_value < self._best_loss:
            self._best_loss = loss_value
            self._best_weights = self.export_weights()
            self._best_it = it_label
            return True
        return False

    def _restore_best_checkpoint(self):
        if self._best_weights is None:
            return False
        w0, b0, a0 = self._best_weights
        assigns = []
        for var, val in zip(self.mlp.weights, w0):
            assigns.append(tf.compat.v1.assign(var, np.asarray(val, dtype=np.float32)))
        for var, val in zip(self.mlp.biases, b0):
            assigns.append(tf.compat.v1.assign(var, np.asarray(val, dtype=np.float32)))
        for var, val in zip(self.mlp.A, a0):
            assigns.append(tf.compat.v1.assign(var, np.asarray(val, dtype=np.float32)))
        self.sess.run(assigns)
        return True

    def _train_window(self, N_iter, batch_size, face_feed, t_low=None, t_high=None, slab_label="full", run_lbfgs=False):
        base_t_left, base_x_left = face_feed["left"]
        base_t_right, base_x_right = face_feed["right"]
        names = sorted(self.loss_parts.keys())
        feed = None
        update_every = int(H_PAR.get("adaptive_loss_update_every", 100))
        t_low_print = float(DOMAIN["tmin"]) if t_low is None else float(t_low)
        t_high_print = float(DOMAIN["tmax"]) if t_high is None else float(t_high)
        print(f"[A/B ] training window {slab_label}: t in [{t_low_print:.4g}, {t_high_print:.4g}] day, iters={N_iter}")

        for it in range(int(N_iter)):
            tr, xr = self._residual_batch(batch_size, t_low, t_high, it)
            if bool(H_PAR.get("time_slab_training", False)):
                tl, xl = self._sample_face_window(base_t_left, base_x_left, t_low, t_high)
                trgt, xrgt = self._sample_face_window(base_t_right, base_x_right, t_low, t_high)
            else:
                tl, xl = base_t_left, base_x_left
                trgt, xrgt = base_t_right, base_x_right

            wfeed, res_w, flux_w, coion_w = self._weight_feed(it, N_iter)
            feed = {
                self.t_res: tr, self.x_res: xr,
                self.t_ic: t_ic, self.x_ic: x_ic,
                self.t_left: tl, self.x_left: xl,
                self.t_right: trgt, self.x_right: xrgt,
            }
            feed.update(wfeed)
            self.sess.run(self.train_op, feed)

            if (update_every > 0) and (it % update_every == 0):
                vals_for_adapt = self.sess.run([self.loss_parts[n] for n in names], feed)
                self._update_adaptive_loss_scales(names, vals_for_adapt)

            if it % 100 == 0:
                vals = self.sess.run([self.loss, self.grad_norm, self.grad_finite_frac] + [self.loss_parts[n] for n in names], feed)
                loss_now = float(vals[0])
                grad_now = float(vals[1])
                finite_now = float(vals[2])
                best_mark = " *best" if self._remember_best(loss_now, f"{slab_label}:{it}") else ""
                bad_grad = finite_now < float(H_PAR.get("min_grad_finite_frac", 0.5))
                if (grad_now <= 1e-12 or bad_grad) and not self._zero_grad_warned:
                    print(f"[A/B warn] near-zero/non-finite gradient at {slab_label} It {it}; finite_grad={finite_now:.2f}.")
                    self._zero_grad_warned = True
                print(
                    f"[A/B ] {slab_label} It {it:5d} L={loss_now:.3e}{best_mark} | "
                    f"grad={grad_now:.2e}, finite_grad={finite_now:.2f} | "
                    f"w_res={res_w:.2g}, w_flux={flux_w:.2g}, w_coion={coion_w:.2g} | "
                    f"scale=({self._adapt_scale_res:.2g},{self._adapt_scale_flux:.2g},{self._adapt_scale_coion:.2g}) | "
                    + ", ".join([f"{n}={v:.2e}" for n, v in zip(names, vals[3:])])
                )
                self.loss_hist.append(loss_now)
                if bad_grad:
                    print(f"[A/B warn] stopping {slab_label} early and restoring best checkpoint because gradients are non-finite.")
                    self._restore_best_checkpoint()
                    break

        if feed is None:
            tr, xr = self._residual_batch(batch_size, t_low, t_high, 0)
            tl, xl = self._sample_face_window(base_t_left, base_x_left, t_low, t_high)
            trgt, xrgt = self._sample_face_window(base_t_right, base_x_right, t_low, t_high)
            wfeed, _, _, _ = self._weight_feed(0, 1)
            feed = {
                self.t_res: tr, self.x_res: xr,
                self.t_ic: t_ic, self.x_ic: x_ic,
                self.t_left: tl, self.x_left: xl,
                self.t_right: trgt, self.x_right: xrgt,
            }
            feed.update(wfeed)

        if run_lbfgs and bool(TRNC.get("hplus_use_lbfgs", True)):
            self.lbfgs.minimize(self.sess, feed_dict=feed, fetches=[self.loss])
            loss_after_lbfgs = float(self.sess.run(self.loss, feed))
            self._remember_best(loss_after_lbfgs, f"{slab_label}:lbfgs")
            print(f"[A/B ] LBFGS done for {slab_label}.")
        return feed

    def train(self, N_iter, batch=True, batch_size=512, face_feed=None):
        if face_feed is None:
            raise ValueError("AcidBaseNet.train requires face_feed with left/right boundary arrays.")

        # Time-slab continuation: train successively on expanding time windows.
        # This keeps early sharp pH transients learned before exposing the network to the full 0--tmax domain.
        if bool(H_PAR.get("time_slab_training", False)):
            tmin = float(DOMAIN["tmin"])
            tmax = float(DOMAIN["tmax"])
            raw_slabs = list(H_PAR.get("time_slabs", [tmax]))
            slabs = sorted({float(s) for s in raw_slabs if float(s) > tmin})
            if len(slabs) == 0 or slabs[-1] < tmax - 1e-12:
                slabs.append(tmax)
            slabs = [min(s, tmax) for s in slabs]
            n_slabs = len(slabs)
            base_iters = int(N_iter) // n_slabs
            rem = int(N_iter) - base_iters * n_slabs
            last_feed = None
            previous_end = tmin
            for k, slab_end in enumerate(slabs):
                slab_iters = base_iters + (1 if k < rem else 0)
                if slab_iters <= 0:
                    continue
                mode = str(H_PAR.get("time_slab_mode", "expanding")).lower()
                if mode == "disjoint":
                    slab_low = previous_end
                else:
                    slab_low = tmin
                slab_label = f"slab{k+1}/{n_slabs}@{slab_end:g}d"
                self._res_cache = None
                self._res_cache_meta = None
                last_feed = self._train_window(
                    slab_iters, batch_size, face_feed,
                    t_low=slab_low, t_high=slab_end,
                    slab_label=slab_label,
                    run_lbfgs=False,
                )
                previous_end = slab_end
            if bool(TRNC.get("hplus_use_lbfgs", True)) and last_feed is not None:
                # One final LBFGS pass on the full sampled domain, not the last random batch.
                base_t_left, base_x_left = face_feed["left"]
                base_t_right, base_x_right = face_feed["right"]
                wfeed, _, _, _ = self._weight_feed(max(int(N_iter) - 1, 0), max(int(N_iter), 1))
                full_feed = {
                    self.t_res: t_res, self.x_res: x_res,
                    self.t_ic: t_ic, self.x_ic: x_ic,
                    self.t_left: base_t_left, self.x_left: base_x_left,
                    self.t_right: base_t_right, self.x_right: base_x_right,
                }
                full_feed.update(wfeed)
                self.lbfgs.minimize(self.sess, feed_dict=full_feed, fetches=[self.loss])
                print("[A/B ] final full-domain LBFGS done.")
            else:
                print("[A/B ] LBFGS skipped.")
            if self._restore_best_checkpoint():
                print(f"[A/B ] restored best checkpoint: {self._best_it}, L={self._best_loss:.3e}")
            return

        # Original single-window training path, with adaptive sampling/weights still available.
        self._train_window(
            N_iter, batch_size, face_feed,
            t_low=float(DOMAIN["tmin"]), t_high=float(DOMAIN["tmax"]),
            slab_label="full",
            run_lbfgs=bool(TRNC.get("hplus_use_lbfgs", True)),
        )
        if self._restore_best_checkpoint():
            print(f"[A/B ] restored best checkpoint: {self._best_it}, L={self._best_loss:.3e}")

    def export_weights(self):
        return self.sess.run(self.mlp.weights), self.sess.run(self.mlp.biases), self.sess.run(self.mlp.A)

# backward-compatible class alias
HPlusNet = AcidBaseNet

# ---------------------------
# 5) Chemistry equilibrium kernel + invariant reconstruction (stable)
# ---------------------------
class ChemEquilKernel:
    """
    Algebraic reconstruction for Pb component chemistry.
    Inputs via nets: theta(t,x), cH_w(t,x), cOH_w(t,x)
    Unknowns (bulk): SOH, SOH2+, SO-, SOPb+, Pp; water species include free Pb2+
    and mononuclear Pb-OH complexes. The transported component is:
        Psi_Pb = theta*C_Pb,aq,total + SOPb + Pp
    where precipitation saturation is controlled by free Pb2+ activity proxy.
    """
    def __init__(self, cfg_pb_chem, water_w, ab_w, surf_init_bulk):
        self.k = cfg_pb_chem
        self.m = tf.constant(float(cfg_pb_chem.get("m", 2.0)), tf.float32)
        self.Ksp = tf.constant(float(cfg_pb_chem.get("Ksp_PbOH2", 1.43e-11)), tf.float32)
        self.water_w = water_w
        self.ab_w = ab_w
        self.K_pr   = tf.constant(self.k["k_pr_f"]/self.k["k_pr_b"], tf.float32)
        self.K_dprp = tf.constant(self.k["k_dpr_f"]/self.k["k_dpr_b"], tf.float32)
        self.K_ad   = tf.constant(self.k["k_ad_f"]/self.k["k_ad_b"], tf.float32)
        self.SOPb0 = tf.constant(float(surf_init_bulk["SOPb0"] + surf_init_bulk.get("Pp0", 0.0)), tf.float32)
        self.S_tot = tf.constant(float(surf_init_bulk["SOH0"] + surf_init_bulk["SOPb0"] +
                                       surf_init_bulk["SOH2_0"] + surf_init_bulk["SOm0"]), tf.float32)
        hyd = cfg_pb_chem.get("HYDROLYSIS", {})
        log_beta_L = list(hyd.get("log_beta_OH_molL", [6.54, 11.06, 13.97, 15.20]))
        while len(log_beta_L) < 4:
            log_beta_L.append(-300.0)
        if not bool(hyd.get("include", True)):
            log_beta_L = [-300.0, -300.0, -300.0, -300.0]
        self.beta_oh_m3 = [10.0 ** (float(log_beta_L[j]) - 3.0 * float(j + 1)) for j in range(4)]
        self.beta_oh = [tf.constant(v, tf.float32) for v in self.beta_oh_m3]

    def _hydrolysis_factors_tf(self, cOH_w):
        cOH = tf.maximum(cOH_w, 1e-30)
        b1 = self.beta_oh[0] * cOH
        b2 = self.beta_oh[1] * tf.square(cOH)
        b3 = self.beta_oh[2] * tf.pow(cOH, 3.0)
        b4 = self.beta_oh[3] * tf.pow(cOH, 4.0)
        alpha = tf.clip_by_value(1.0 + b1 + b2 + b3 + b4, 1.0, 1e16)
        z_num = 2.0 + b1 - b3 - 2.0 * b4
        z_eff = tf.clip_by_value(z_num / tf.maximum(alpha, 1e-30), -2.0, 2.0)
        return alpha, z_eff, (b1, b2, b3, b4)

    def _mlp_eval(self, weights, t, x):
        w,b,a = weights
        H = nondim_tx(t, x)
        for l in range(len(w)):
            H = tf.add(tf.matmul(H, w[l]), b[l])
            if l < len(w)-1:
                H = tf.tanh(NETC["act_scale"] * a[l] * H)
        return H

    def theta(self, t, x):
        w,b,a = self.water_w
        psi = water_head_from_weights(w, b, a, t, x)
        return theta_function(psi)

    def _a_from_pH_tensor(self, pH):
        ln10 = tf.constant(np.log(10.0), tf.float32)
        cH = 1000.0 * tf.exp(-ln10 * pH)
        cOH = Kw_const / tf.maximum(cH, 1e-30)
        return cH - cOH

    def _ab_net_a(self, t, x):
        raw = tf.clip_by_value(self._mlp_eval(self.ab_w, t, x), -8.0, 8.0)
        a_scale = tf.constant(float(H_PAR.get("a_raw_scale", 20.0)), tf.float32)
        t0 = tf.constant(float(CFG["DOMAIN"]["tmin"]), tf.float32)
        tau_gate = tf.constant(float(H_PAR.get("a_ic_gate_tau", 0.02)), tf.float32)
        gate = tf.where(
            t <= t0,
            tf.zeros_like(t),
            1.0 - tf.exp(-(t - t0) / tf.maximum(tau_gate, 1e-12)),
        )
        a_ic = tf.constant(float(H_INIT.get("a_ic", 0.0)), tf.float32)
        a_free = a_ic + a_scale * raw
        a_val = (1.0 - gate) * a_ic + gate * a_free
        return tf.where(tf.math.is_finite(a_val), a_val, tf.zeros_like(a_val))

    def _acid_base_conc(self, t, x):
        a_val = self._ab_net_a(t, x)
        s = tf.sqrt(tf.square(a_val) + 4.0 * Kw_const)
        cH_pos = 0.5 * (a_val + s)
        cH_neg = (2.0 * Kw_const) / tf.maximum(s - a_val, 1e-30)
        cOH_neg = 0.5 * (-a_val + s)
        cOH_pos = (2.0 * Kw_const) / tf.maximum(s + a_val, 1e-30)
        cH = tf.where(a_val >= 0.0, cH_pos, cH_neg)
        cOH = tf.where(a_val <= 0.0, cOH_neg, cOH_pos)
        cH = tf.where(tf.math.is_finite(cH), cH, tf.zeros_like(cH))
        cOH = tf.where(tf.math.is_finite(cOH), cOH, tf.zeros_like(cOH))
        return cH, cOH

    def _acid_base_a(self, t, x):
        return self._ab_net_a(t, x)

    def cH_w(self, t, x):
        cH, _ = self._acid_base_conc(t, x)
        return cH

    def cOH_w(self, t, x):
        _, cOH = self._acid_base_conc(t, x)
        return cOH

    def reconstruct_pb_equil_from_fields(self, theta_loc, cH_w, cOH_w, Psi_b, t,
                                         precip_gate_override=None, stop_acid=True,
                                         return_gate=False):
        def finite(z):
            return tf.where(tf.math.is_finite(z), z, tf.zeros_like(z))

        Psi_b = tf.clip_by_value(finite(Psi_b), 0.0, self.S_tot + 1.0e4)
        theta_loc = tf.stop_gradient(tf.clip_by_value(finite(theta_loc), 1e-6, 1.0))
        cH_w = tf.clip_by_value(finite(cH_w), 1e-12, 1e3)
        cOH_w = tf.clip_by_value(finite(cOH_w), 1e-10, 1e4)
        if stop_acid:
            cH_w = tf.stop_gradient(cH_w)
            cOH_w = tf.stop_gradient(cOH_w)
        cH_b = tf.maximum(theta_loc * cH_w, 1e-12)

        # Kim local partition: native pool + incoming/reactive pool.
        # cPb_aq_total remains water-basis; theta*cPb_aq_total is the bulk Paq.
        aH = self.K_pr * cH_b
        bH = self.K_dprp / cH_b
        B = tf.clip_by_value(1.0 + aH + bH, 1.0, 1e8)

        ln10 = tf.constant(np.log(10.0), tf.float32)
        pH_loc = -tf.math.log(tf.maximum(cH_w, 1e-30) / 1000.0) / ln10
        alpha_native = tf.clip_by_value(0.27 * pH_loc - 0.23, 0.0, 1.0)
        alpha_native = tf.clip_by_value(finite(alpha_native), 0.0, 1.0)

        c0 = self.SOPb0
        ads_capacity = self.S_tot
        native_pool = tf.minimum(Psi_b, c0)
        native_target = alpha_native * c0
        locked_sopb = tf.minimum(native_pool, native_target)
        released_native = tf.maximum(native_pool - locked_sopb, 0.0)

        incoming_pool = tf.maximum(Psi_b - native_pool, 0.0)
        vacant_sites = tf.maximum(ads_capacity - locked_sopb, 0.0)
        eps_alpha = tf.constant(1.0e-6, tf.float32)
        alpha_reactive = tf.clip_by_value(alpha_native, 0.0, 1.0 - eps_alpha)
        adsorption_odds = alpha_reactive / tf.maximum(1.0 - alpha_reactive, eps_alpha)

        ads_incoming_noP = tf.minimum(vacant_sites, alpha_reactive * incoming_pool)
        aq_noP_bulk = tf.maximum(released_native + incoming_pool - ads_incoming_noP, 0.0)

        cPb_sat_water = tf.clip_by_value(self.Ksp / tf.maximum(tf.pow(cOH_w, self.m), 1e-30), 0.0, 1e9)
        cPb_sat_bulk = tf.clip_by_value(theta_loc * cPb_sat_water, 0.0, 1e9)
        natural_precip_gate = aq_noP_bulk > (1.0 + 1.0e-6) * tf.maximum(cPb_sat_bulk, 1e-30)
        if precip_gate_override is None:
            precip_gate = natural_precip_gate
        else:
            precip_gate = tf.cast(precip_gate_override, tf.bool)

        aq_sat_bulk = tf.minimum(cPb_sat_bulk, released_native + incoming_pool)
        ads_incoming_sat = tf.minimum(vacant_sites, tf.minimum(incoming_pool, adsorption_odds * aq_sat_bulk))

        c_aq_bulk = tf.where(precip_gate, aq_sat_bulk, aq_noP_bulk)
        SOPb_reactive = tf.where(precip_gate, ads_incoming_sat, ads_incoming_noP)
        SOPb = locked_sopb + SOPb_reactive

        Pp = tf.nn.relu(Psi_b - SOPb - c_aq_bulk)
        c_aq_bulk = Psi_b - SOPb - Pp
        cPb_aq_total = tf.clip_by_value(c_aq_bulk / tf.maximum(theta_loc, 1e-12), 0.0, 1e9)

        S_free = tf.maximum(ads_capacity - SOPb, 0.0)
        SOH = S_free / tf.maximum(B, 1e-12)
        SOH2 = aH * SOH
        SOm = bH * SOH

        out = (finite(cPb_aq_total), finite(SOPb), finite(SOH),
               finite(SOH2), finite(SOm), finite(Pp), finite(cOH_w))
        if return_gate:
            return out + (tf.cast(natural_precip_gate, tf.float32),)
        return out

    def reconstruct_pb_equil(self, t, x, Psi_b, precip_gate_override=None,
                             stop_acid=True, return_gate=False):
        theta_loc = self.theta(t, x)
        cH_w = self.cH_w(t, x)
        cOH_w = self.cOH_w(t, x)
        return self.reconstruct_pb_equil_from_fields(
            theta_loc, cH_w, cOH_w, Psi_b, t,
            precip_gate_override=precip_gate_override,
            stop_acid=stop_acid,
            return_gate=return_gate,
        )

# ---------------------------
# 6) Pb invariant NP (day units) with equilibrium reconstruction
# ---------------------------

class PbInvNet:
    def __init__(self, layers, water_weights, elec_weights, chem_kernel,
                 leftBC="flux", rightBC="open_outflow",
                 c_ic_water=0.0,
                 J_left_true_val=0.0,
                 dir_left_c=None, dir_right_c=None,
                 init_from=None, freeze_k=0, active_gate_weights=None, stop_acid_in_pb=True):
        """
        Component-PINN for Pb transport with local-equilibrium reconstruction.

        Literature-backed structure:
        - the neural network outputs Psi_Pb and an auxiliary conservative flux;
          the flux is hard-constrained to J(0,t)=0;
        - dissolved Pb, sorbed Pb, and precipitated Pb are reconstructed from
          mass-action surface complexation, site conservation, and the
          solubility-product complementarity condition.

        This removes the previous unsupported DAE shortcut in which a neural
        network also predicted free Pb2+ and any leftover mass was forced into
        Pp by relu(Psi - aqueous - adsorbed).
        """
        pb_layers = list(layers)
        pb_layers[-1] = 2
        self.mlp = DenseMLP(pb_layers, NETC["act_scale"], NETC["trainable_layer_gain"])
        if init_from is not None:
            w0, b0, a0 = init_from
            for i in range(len(self.mlp.weights)):
                self.mlp.weights[i] = tf.Variable(np.array(w0[i]), dtype=tf.float32, trainable=(i >= freeze_k))
                self.mlp.biases[i] = tf.Variable(np.array(b0[i]), dtype=tf.float32, trainable=(i >= freeze_k))
                self.mlp.A[i] = tf.Variable(a0[i], dtype=tf.float32, trainable=(i >= freeze_k))
        self.water_w, self.water_b, self.water_a = water_weights
        self.elec_w,  self.elec_b,  self.elec_a  = elec_weights
        self.chem = chem_kernel
        self.active_gate_weights = active_gate_weights
        self.stop_acid_in_pb = bool(stop_acid_in_pb)
        self.c_ic_water = float(c_ic_water)
        self.Psi_ic_store = float(SURF_INIT["SOPb0"] + SURF_INIT["Pp0"])
        self.leftBC, self.rightBC = leftBC, rightBC

        (self.t_res, self.x_res, self.t_ic, self.x_ic,
         self.t_left, self.x_left, self.t_right, self.x_right,
         self.t_mass) = [tf.compat.v1.placeholder(tf.float32, [None, 1]) for _ in range(9)]
        self.sess = tf.compat.v1.Session(config=TF_CONFIG)

        self.Psi_res, self.residual_res, self.constitutive_res = self.net_res(self.t_res, self.x_res)
        self.J_left_aux_pred  = self.net_J(self.t_left,  self.x_left)
        self.J_right_aux_pred = self.net_J(self.t_right, self.x_right)
        self.J_left_pred  = self.net_J_phys(self.t_left,  self.x_left)
        self.J_right_pred = self.net_J_phys(self.t_right, self.x_right)
        Jl_aux, Jl_adv, Jl_diff, Jl_em, *_ = self.net_J_components(self.t_left, self.x_left)
        self.left_constitutive_res = self._finite(Jl_aux - (Jl_adv + Jl_diff + Jl_em))
        Jr_aux, Jr_adv, Jr_diff, Jr_em, *_ = self.net_J_components(self.t_right, self.x_right)
        self.right_constitutive_res = self._finite(Jr_aux - (Jr_adv + Jr_diff + Jr_em))
        self.Psi_left_pred  = self.net_Psi(tf.concat([self.t_left,  self.x_left],  1))
        self.Psi_right_pred = self.net_Psi(tf.concat([self.t_right, self.x_right], 1))
        self.Psi_ic_pred    = self.net_Psi(tf.concat([self.t_ic,    self.x_ic],    1))
        self.c_left_pred  = self.dissolved_c(self.t_left, self.x_left)
        self.c_right_pred = self.dissolved_c(self.t_right, self.x_right)

        theta_ic = self.chem.theta(self.t_ic, self.x_ic)
        self.Psi_ic_true = theta_ic * tf.fill(tf.shape(self.t_ic), tf.constant(c_ic_water, tf.float32)) \
                           + tf.fill(tf.shape(self.t_ic), tf.constant(SURF_INIT["SOPb0"] + SURF_INIT["Pp0"], tf.float32))
        self.c_left_true = tf.fill(tf.shape(self.c_left_pred), tf.constant(0.0 if dir_left_c is None else dir_left_c, tf.float32))
        self.c_right_true = tf.fill(tf.shape(self.c_right_pred), tf.constant(0.0 if dir_right_c is None else dir_right_c, tf.float32))
        self.J_left_true = tf.fill(tf.shape(self.J_left_pred), tf.constant(J_left_true_val, tf.float32))
        self.mass_balance_res = self.net_mass_balance_res(self.t_mass)
        self.mass_integral_res = self.net_mass_integral_res()

        psi_scale = tf.constant(max(float(SURF_INIT["SOPb0"] + SURF_INIT["Pp0"] + 1.0), 1.0), tf.float32)
        res_norm = tf.constant(float(PB_PAR.get("res_norm", PB_PAR.get("dae_res_norm", 50.0))), tf.float32)
        flux_norm = tf.constant(float(PB_PAR.get("flux_norm", PB_PAR.get("dae_flux_norm", 1.0))), tf.float32)
        constitutive_norm = tf.constant(float(PB_PAR.get("constitutive_norm", PB_PAR.get("flux_norm", 1.0))), tf.float32)
        Lx = max(float(CFG["DOMAIN"]["xmax"]) - float(CFG["DOMAIN"]["xmin"]), 1e-12)
        Tspan = max(float(CFG["DOMAIN"]["tmax"]) - float(CFG["DOMAIN"]["tmin"]), 1e-12)
        mass_norm = tf.constant(float(PB_PAR.get("mass_balance_norm", max((self.Psi_ic_store + 1.0) * Lx / Tspan, 1e-6))), tf.float32)
        mass_integral_norm = tf.constant(float(PB_PAR.get("mass_integral_norm", max(self.Psi_ic_store * Lx, 1e-6))), tf.float32)
        self.residual_res_nd = self.residual_res / res_norm
        self.constitutive_res_nd = self.constitutive_res / constitutive_norm
        w_res = float(PB_PAR.get("res_weight", PB_PAR.get("dae_res_weight", 1.0)))
        w_ic = float(PB_PAR.get("ic_weight", PB_PAR.get("dae_ic_weight", 20.0)))
        w_flux = float(PB_PAR.get("flux_weight", PB_PAR.get("dae_flux_weight", 100.0)))
        w_const = float(PB_PAR.get("constitutive_weight", 0.0))
        w_left_const = float(PB_PAR.get("left_constitutive_weight", 0.0))
        w_right_const = float(PB_PAR.get("right_constitutive_weight", 0.0))
        w_mass = float(PB_PAR.get("mass_balance_weight", 0.0))
        w_mass_int = float(PB_PAR.get("mass_integral_weight", 0.0))

        loss_terms = [
            w_res * tf.reduce_mean(tf.square(self.residual_res_nd)),
            w_ic  * tf.reduce_mean(tf.square((self.Psi_ic_pred - self.Psi_ic_true) / psi_scale)),
            w_const * tf.reduce_mean(tf.square(self.constitutive_res_nd)),
        ]

        if self.leftBC.lower() != "flux":
            raise ValueError("Pb left boundary must be flux with J=0.")
        if self.rightBC.lower() != "open_outflow":
            raise ValueError("Pb right boundary must be open_outflow.")
        loss_terms.append(w_flux * tf.reduce_mean(tf.square((self.J_left_pred - self.J_left_true) / flux_norm)))
        loss_terms.append(w_flux * tf.reduce_mean(tf.square(tf.nn.relu(-self.J_right_pred) / flux_norm)))
        loss_terms.append(w_left_const * tf.reduce_mean(tf.square(self.left_constitutive_res / constitutive_norm)))
        loss_terms.append(w_right_const * tf.reduce_mean(tf.square(self.right_constitutive_res / constitutive_norm)))
        mass_balance_loss = tf.constant(0.0, tf.float32)
        if w_mass > 0.0:
            mass_balance_loss = w_mass * tf.reduce_mean(tf.square(self.mass_balance_res / mass_norm))
        mass_integral_loss = tf.constant(0.0, tf.float32)
        if w_mass_int > 0.0:
            mass_integral_loss = w_mass_int * tf.reduce_mean(tf.square(self.mass_integral_res / mass_integral_norm))
        loss_terms.append(mass_balance_loss)
        loss_terms.append(mass_integral_loss)

        self.loss = tf.add_n(loss_terms)
        self.loss_parts = {"res": loss_terms[0], "ic": loss_terms[1], "constitutive": loss_terms[2],
                           "left_flux_zero": loss_terms[3], "right_no_inflow": loss_terms[4],
                           "left_constitutive": loss_terms[5], "right_constitutive": loss_terms[6],
                           "mass_balance": loss_terms[7], "mass_integral": loss_terms[8]}

        self.global_step = tf.Variable(0, trainable=False)
        lr = tf.compat.v1.train.exponential_decay(TRNC.get("pb_adam_lr", TRNC["adam_lr"]), self.global_step, 1000, 0.9)
        opt = tf.compat.v1.train.AdamOptimizer(lr)
        vars_all = [v for v in (self.mlp.weights + self.mlp.biases + self.mlp.A) if getattr(v, "trainable", True)]
        vars_ = [v for g, v in opt.compute_gradients(self.loss, var_list=vars_all) if g is not None]
        term_grads = [tf.gradients(term, vars_) for term in loss_terms]
        safe_grads, finite_counts, total_counts = [], [], []
        for i, v in enumerate(vars_):
            acc = tf.zeros_like(v)
            for grads in term_grads:
                g = grads[i]
                if g is None:
                    g = tf.zeros_like(v)
                g = tf.convert_to_tensor(g)
                finite_mask = tf.math.is_finite(g)
                finite_counts.append(tf.reduce_sum(tf.cast(finite_mask, tf.float32)))
                total_counts.append(tf.cast(tf.size(g), tf.float32))
                acc = acc + tf.where(finite_mask, g, tf.zeros_like(g))
            safe_grads.append(acc)
        self.grad_finite_frac = tf.add_n(finite_counts) / tf.maximum(tf.add_n(total_counts), 1.0)
        self.grad_norm = tf.linalg.global_norm(safe_grads)
        safe_grads, _ = tf.clip_by_global_norm(safe_grads, float(TRNC.get("pb_grad_clip", 0.5)))
        self.train_op = opt.apply_gradients(list(zip(safe_grads, vars_)), global_step=self.global_step)

        self.lbfgs = dde.optimizers.tensorflow_compat_v1.scipy_optimizer.ScipyOptimizerInterface(
            self.loss, method='L-BFGS-B', options=TRNC["lbfgs"])
        self.sess.run(tf.compat.v1.global_variables_initializer())
        self.loss_hist = []

    def _finite(self, z):
        return tf.where(tf.math.is_finite(z), z, tf.zeros_like(z))

    def net_raw(self, X):
        return self.mlp.forward(X)

    def net_Psi(self, X):
        raw_scale = tf.constant(float(PB_PAR.get("psi_raw_scale", 20.0)), tf.float32)
        raw = raw_scale * tf.tanh(self.net_raw(X)[:, 0:1] / tf.maximum(raw_scale, 1e-6))
        t = X[:, 0:1]
        x = X[:, 1:2]
        theta0 = self.chem.theta(t, x)
        Psi0 = theta0 * tf.fill(tf.shape(t), tf.constant(self.c_ic_water, tf.float32)) \
               + tf.fill(tf.shape(t), tf.constant(self.Psi_ic_store, tf.float32))
        Psi0 = tf.maximum(Psi0, 1e-6)
        eta0 = tf.math.log(tf.math.expm1(Psi0) + 1e-12)
        t0 = tf.constant(float(CFG["DOMAIN"]["tmin"]), tf.float32)
        tau = tf.constant(float(PB_PAR.get("psi_ic_gate_tau", 0.02)), tf.float32)
        gate = tf.where(t <= t0, tf.zeros_like(t), 1.0 - tf.exp(-(t - t0) / tf.maximum(tau, 1e-6)))
        Psi = tf.nn.softplus(eta0 + gate * raw)
        return tf.clip_by_value(self._finite(Psi), 0.0, self.chem.S_tot + 1.0e4)

    def net_J_aux(self, t, x):
        X = tf.concat([t, x], 1)
        raw_scale = tf.constant(float(PB_PAR.get("flux_raw_scale", 2.0)), tf.float32)
        raw = raw_scale * tf.tanh(self.net_raw(X)[:, 1:2] / tf.maximum(raw_scale, 1e-6))
        xmin = tf.constant(float(CFG["DOMAIN"]["xmin"]), tf.float32)
        xmax = tf.constant(float(CFG["DOMAIN"]["xmax"]), tf.float32)
        xrel = (x - xmin) / tf.maximum(xmax - xmin, 1e-12)
        return self._finite(xrel * raw)

    def _water_fields(self, t, x):
        psi = water_head_from_weights(self.water_w, self.water_b, self.water_a, t, x)
        theta = theta_function(psi)
        K = K_function(psi)
        psi_x = tf.gradients(psi, x)[0]
        q_hyd = -K * psi_x
        return theta, q_hyd

    def _phi_grad(self, t, x):
        H = nondim_tx(t, x)
        for l in range(len(self.elec_w)):
            H = tf.add(tf.matmul(H, self.elec_w[l]), self.elec_b[l])
            if l < len(self.elec_w) - 1:
                H = tf.tanh(NETC["act_scale"] * self.elec_a[l] * H)
        phi = H
        return phi, tf.gradients(phi, x)[0]

    def _precip_gate_override(self, t, x):
        if self.active_gate_weights is None:
            return None
        Psi_gate = pb_psi_from_weights(self.active_gate_weights, (self.water_w, self.water_b, self.water_a), t, x)
        *_, active_gate = self.chem.reconstruct_pb_equil(t, x, Psi_gate, return_gate=True)
        return active_gate > 0.5

    def species_from_Psi(self, t, x, Psi):
        cPb, SOPb, SOH, SOH2, SOm, Pp, cOH = self.chem.reconstruct_pb_equil(
            t, x, Psi, precip_gate_override=self._precip_gate_override(t, x),
            stop_acid=self.stop_acid_in_pb
        )
        si = tf.math.log((tf.maximum(cPb, 0.0) * tf.pow(tf.maximum(cOH, 1e-30), self.chem.m) + 1e-30) /
                         (self.chem.Ksp + 1e-30))
        return (self._finite(cPb), self._finite(SOPb), self._finite(SOH),
                self._finite(SOH2), self._finite(SOm), self._finite(Pp),
                self._finite(cOH), self._finite(si))

    def dissolved_c(self, t, x):
        X = tf.concat([t, x], 1)
        Psi = self.net_Psi(X)
        cPb, *_ = self.species_from_Psi(t, x, Psi)
        return cPb

    def dissolved_c_from_Psi(self, t, x, Psi):
        cPb, *_ = self.species_from_Psi(t, x, Psi)
        return cPb

    def net_J_components(self, t, x):
        X = tf.concat([t, x], 1)
        Psi = self.net_Psi(X)
        cPb, SOPb, SOH, SOH2, SOm, Pp, cOH_w, si = self.species_from_Psi(t, x, Psi)
        cx = self._finite(tf.gradients(cPb, x)[0])
        theta, q_hyd = self._water_fields(t, x)
        _, phi_x = self._phi_grad(t, x)
        q_adv = self._finite(q_hyd + (keo_of_theta(theta) * phi_x if PB_PAR.get("include_eo", True) else 0.0))
        Dstar = Dstar_of_theta(theta, Dw_Pb)
        Deff = self._finite(Dstar + DL_Pb * tf.abs(q_adv))
        z_transport = tf.constant(float(PB_PAR.get("transport_z", PB_PAR.get("z", 2.0))), tf.float32)
        u_star = self._finite(z_transport * tf.constant(float(CFG["GLOBAL"]["CONSTANTS"]["F"]) / (CFG["GLOBAL"]["CONSTANTS"]["R"] * CFG["GLOBAL"]["CONSTANTS"]["T"]), tf.float32) * Dstar)
        J_adv = self._finite(q_adv * cPb)
        J_diff = self._finite(-Deff * cx)
        J_em = self._finite(-u_star * cPb * phi_x)
        J_aux = self.net_J_aux(t, x)
        return J_aux, J_adv, J_diff, J_em, cPb, cx, q_adv, phi_x, Deff, u_star

    def net_J(self, t, x):
        return self.net_J_aux(t, x)

    def net_J_phys(self, t, x):
        _, J_adv, J_diff, J_em, *_ = self.net_J_components(t, x)
        return self._finite(J_adv + J_diff + J_em)

    def net_mass_balance_res(self, t):
        nx = max(int(PB_PAR.get("mass_balance_nx", 31)), 3)
        xmin = float(CFG["DOMAIN"]["xmin"]); xmax = float(CFG["DOMAIN"]["xmax"])
        x_line = tf.linspace(tf.constant(xmin, tf.float32), tf.constant(xmax, tf.float32), nx)[None, :]
        t_grid = tf.tile(t, [1, nx])
        x_grid = tf.tile(x_line, [tf.shape(t)[0], 1])
        tt = tf.reshape(t_grid, [-1, 1])
        xx = tf.reshape(x_grid, [-1, 1])
        Psi = self.net_Psi(tf.concat([tt, xx], 1))
        Psi_t = self._finite(tf.gradients(Psi, tt)[0])
        Psi_t_2d = tf.reshape(Psi_t, [-1, nx])
        dx = tf.constant((xmax - xmin) / float(nx - 1), tf.float32)
        weights = tf.concat([tf.ones([1], tf.float32) * 0.5, tf.ones([nx - 2], tf.float32), tf.ones([1], tf.float32) * 0.5], 0)[None, :]
        dMdt = dx * tf.reduce_sum(Psi_t_2d * weights, axis=1, keepdims=True)
        J_left = self.net_J_phys(t, tf.ones_like(t) * tf.constant(xmin, tf.float32))
        J_right = self.net_J_phys(t, tf.ones_like(t) * tf.constant(xmax, tf.float32))
        return self._finite(dMdt + J_right - J_left)

    def net_mass_integral_res(self):
        nx = max(int(PB_PAR.get("mass_integral_nx", 21)), 3)
        nt = max(int(PB_PAR.get("mass_integral_nt", 21)), 3)
        xmin = float(CFG["DOMAIN"]["xmin"]); xmax = float(CFG["DOMAIN"]["xmax"])
        tmin = float(CFG["DOMAIN"]["tmin"]); tmax = float(CFG["DOMAIN"]["tmax"])
        early = np.array([0.0, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.35], dtype=np.float32)
        uniform = np.linspace(tmin, tmax, nt, dtype=np.float32)
        t_vals = np.unique(np.clip(np.concatenate([uniform, early]), tmin, tmax)).astype(np.float32)
        t_vals.sort()
        nt_eff = int(t_vals.size)
        t_col = tf.constant(t_vals.reshape(-1, 1), tf.float32)
        x_row = tf.constant(np.linspace(xmin, xmax, nx, dtype=np.float32).reshape(1, -1), tf.float32)
        tt = tf.reshape(tf.tile(t_col, [1, nx]), [-1, 1])
        xx = tf.reshape(tf.tile(x_row, [nt_eff, 1]), [-1, 1])
        Psi = tf.reshape(self.net_Psi(tf.concat([tt, xx], 1)), [nt_eff, nx])
        dx = tf.constant((xmax - xmin) / float(nx - 1), tf.float32)
        xw = tf.concat([tf.ones([1], tf.float32) * 0.5, tf.ones([nx - 2], tf.float32), tf.ones([1], tf.float32) * 0.5], 0)[None, :]
        M = dx * tf.reduce_sum(Psi * xw, axis=1, keepdims=True)
        x_left = tf.ones_like(t_col) * tf.constant(xmin, tf.float32)
        x_right = tf.ones_like(t_col) * tf.constant(xmax, tf.float32)
        flux_out = self.net_J_phys(t_col, x_right) - self.net_J_phys(t_col, x_left)
        dt = t_col[1:] - t_col[:-1]
        step = 0.5 * (flux_out[1:] + flux_out[:-1]) * dt
        cum_flux = tf.concat([tf.zeros([1, 1], tf.float32), tf.cumsum(step, axis=0)], axis=0)
        return self._finite(M - M[0:1] + cum_flux)

    def net_res(self, t, x):
        X = tf.concat([t, x], 1)
        Psi = self.net_Psi(X)
        Psi_t = self._finite(tf.gradients(Psi, t)[0])
        J_aux, J_adv, J_diff, J_em, *_ = self.net_J_components(t, x)
        J_x = self._finite(tf.gradients(J_aux, x)[0])
        constitutive = self._finite(J_aux - (J_adv + J_diff + J_em))
        return self._finite(Psi), self._finite(Psi_t + J_x), constitutive

    def train(self, N_iter, batch=True, batch_size=512, face_feed=None):
        t_left, x_left = face_feed["left"]
        t_right, x_right = face_feed["right"]
        t_mass_all = face_feed.get("mass_t", t_right)
        mass_bs = min(int(PB_PAR.get("mass_balance_batch_size", 128)), t_mass_all.shape[0])
        names = sorted(self.loss_parts.keys())
        feed = None
        for it in range(N_iter):
            idx = np.random.choice(t_res.shape[0], batch_size, replace=False)
            tr, xr = t_res[idx, :], x_res[idx, :]
            idxm = np.random.choice(t_mass_all.shape[0], mass_bs, replace=False)
            tm = t_mass_all[idxm, :]
            feed = {self.t_res: tr, self.x_res: xr, self.t_ic: t_ic, self.x_ic: x_ic,
                    self.t_left: t_left, self.x_left: x_left, self.t_right: t_right, self.x_right: x_right,
                    self.t_mass: tm}
            self.sess.run(self.train_op, feed)
            if it % 100 == 0:
                vals = self.sess.run([self.loss, self.grad_norm, self.grad_finite_frac] + [self.loss_parts[n] for n in names], feed)
                print(f"[PbEq] It {it:5d} L={vals[0]:.3e} | grad={vals[1]:.2e} | finite_grad={vals[2]:.2f} | " + ", ".join([f"{n}={v:.2e}" for n, v in zip(names, vals[3:])]))
                self.loss_hist.append(vals[0])
        if bool(PB_PAR.get("use_lbfgs", PB_PAR.get("dae_lbfgs", False))) and feed is not None:
            self.lbfgs.minimize(self.sess, feed_dict=feed, fetches=[self.loss])
            print("[PbEq] LBFGS done.")
        else:
            print("[PbEq] LBFGS skipped.")

    def export_weights(self):
        return self.sess.run(self.mlp.weights), self.sess.run(self.mlp.biases), self.sess.run(self.mlp.A)


# ---------------------------
# 7) Sequential fixed-stress-style coupling helpers
# ---------------------------
def pb_psi_from_weights(weights, water_weights, t, x):
    """Evaluate the Pb invariant ansatz from frozen exported weights."""
    w_pb, b_pb, a_pb = weights
    raw_scale = tf.constant(float(PB_PAR.get("psi_raw_scale", 20.0)), tf.float32)
    raw = raw_scale * tf.tanh(
        dense_eval_from_weights(w_pb, b_pb, a_pb, t, x)[:, 0:1] / tf.maximum(raw_scale, 1e-6)
    )
    w_w, b_w, a_w = water_weights
    theta = theta_function(water_head_from_weights(w_w, b_w, a_w, t, x))
    psi0_store = tf.constant(float(SURF_INIT["SOPb0"] + SURF_INIT["Pp0"]), tf.float32)
    c_ic = tf.constant(float(PB_INIT["c_ic"]), tf.float32)
    psi0 = tf.maximum(theta * c_ic + psi0_store, 1e-6)
    eta0 = tf.math.log(tf.math.expm1(psi0) + 1e-12)
    t0 = tf.constant(float(DOMAIN["tmin"]), tf.float32)
    tau = tf.constant(float(PB_PAR.get("psi_ic_gate_tau", 0.02)), tf.float32)
    gate = tf.where(t <= t0, tf.zeros_like(t), 1.0 - tf.exp(-(t - t0) / tf.maximum(tau, 1e-6)))
    return tf.clip_by_value(tf.nn.softplus(eta0 + gate * raw), 0.0, 1.0e4 + float(SURF_INIT["SOH0"] + SURF_INIT["SOPb0"] + SURF_INIT["SOH2_0"] + SURF_INIT["SOm0"]))


def relative_parameter_change(previous, current):
    """Algorithm-1 parameter metric; warm starts make this comparison meaningful."""
    num = 0.0
    den = 0.0
    for old_group, new_group in zip(previous, current):
        for old, new in zip(old_group, new_group):
            old = np.asarray(old, dtype=np.float64)
            new = np.asarray(new, dtype=np.float64)
            num += float(np.sum(np.square(new - old)))
            den += float(np.sum(np.square(new)))
    return float(np.sqrt(num / max(den, 1e-30)))


def coupling_state(water_weights, ab_weights, pb_weights):
    """Physical field state on one fixed global grid for outer convergence checks."""
    nt = max(int(SEQC.get("diagnostic_nt", 81)), 3)
    nx = max(int(SEQC.get("diagnostic_nx", 121)), 3)
    t_line = np.linspace(DOMAIN["tmin"], DOMAIN["tmax"], nt, dtype=np.float32)
    x_line = np.linspace(DOMAIN["xmin"], DOMAIN["xmax"], nx, dtype=np.float32)
    t_grid, x_grid = np.meshgrid(t_line, x_line, indexing="ij")
    # Reuse the notebook's default graph: its material constants were created
    # there.  The following operators depend only on exported frozen weights.
    t = tf.constant(t_grid.reshape(-1, 1), tf.float32)
    x = tf.constant(x_grid.reshape(-1, 1), tf.float32)
    chem_eval = ChemEquilKernel(CFG["FIELDS"]["PB"]["CHEM"], water_weights, ab_weights, SURF_INIT)
    Psi = pb_psi_from_weights(pb_weights, water_weights, t, x)
    cPb, SOPb, _, _, _, Pp, _ = chem_eval.reconstruct_pb_equil(t, x, Psi)
    cH = chem_eval.cH_w(t, x)
    pH = -tf.math.log(tf.maximum(cH, 1e-30) / 1000.0) / tf.constant(np.log(10.0), tf.float32)
    theta = chem_eval.theta(t, x)
    mass_identity = Psi - (theta * cPb + SOPb + Pp)
    with tf.compat.v1.Session() as sess:
        pH_v, Psi_v, fixed_v, mass_v = sess.run([pH, Psi, SOPb + Pp, mass_identity])
    shape = (nt, nx)
    return {
        "pH": pH_v.reshape(shape),
        "Psi": Psi_v.reshape(shape),
        "fixed": fixed_v.reshape(shape),
        "mass_identity_max": float(np.max(np.abs(mass_v))),
    }


def coupling_field_metrics(previous, current):
    def relative_l2(new, old):
        return float(np.linalg.norm(new - old) / max(np.linalg.norm(new), 1e-30))
    dpH = current["pH"] - previous["pH"]
    return {
        "pH_rms": float(np.sqrt(np.mean(np.square(dpH)))),
        "pH_max": float(np.max(np.abs(dpH))),
        "Psi_rel": relative_l2(current["Psi"], previous["Psi"]),
        "fixed_rel": relative_l2(current["fixed"], previous["fixed"]),
        "mass_identity_max": float(current["mass_identity_max"]),
    }


def train_acidbase_sequential_block(model, n_iter, face_feed):
    """One global-domain acid/base block; no time-window relay or LBFGS restart."""
    keys = ("time_slab_training", "adaptive_residual_sampling", "training_schedule")
    saved = {key: H_PAR.get(key) for key in keys}
    saved_lbfgs = TRNC.get("hplus_use_lbfgs", True)
    H_PAR.update({
        "time_slab_training": False,
        "adaptive_residual_sampling": False,
        "training_schedule": None,
    })
    TRNC["hplus_use_lbfgs"] = False
    try:
        model.train(n_iter, batch=True,
                    batch_size=TRNC.get("hplus_batch_size", TRNC["batch_size"]),
                    face_feed=face_feed)
    finally:
        H_PAR.update(saved)
        TRNC["hplus_use_lbfgs"] = saved_lbfgs


def _coupled_acid_residual(h_model, pb_model, chem_live, t, x, active_gate_weights=None):
    a, cH, cOH, cHx, cOHx, theta, q_adv, phi_x = h_model._species_fields(t, x)
    Deff_H, u_H = diffusion_pieces(theta, q_adv, DL_H, Dw_H, z_H)
    Deff_OH, u_OH = diffusion_pieces(theta, q_adv, DL_OH, Dw_OH, z_OH)
    JH = q_adv * cH - Deff_H * cHx - u_H * cH * phi_x
    JOH = q_adv * cOH - Deff_OH * cOHx - u_OH * cOH * phi_x
    Ja = JH - JOH

    X = tf.concat([t, x], 1)
    Psi_live = pb_model.net_Psi(X)
    gate = None
    if active_gate_weights is not None:
        Psi_gate = pb_psi_from_weights(active_gate_weights, (wA, bA, aA), t, x)
        *_, active_gate = chem_live.reconstruct_pb_equil(t, x, Psi_gate, return_gate=True)
        gate = active_gate > 0.5

    _, _, _, _, _, Pp, _ = chem_live.reconstruct_pb_equil_from_fields(
        theta, cH, cOH, Psi_live, t,
        precip_gate_override=gate,
        stop_acid=False,
    )
    S_pb = 2.0 * grad0(Pp, t)
    RH = tf.constant(float(H_PAR.get("H_retardation", 1.0)), tf.float32)
    theta_t = grad0(theta, t)
    cH_t = grad0(cH, t)
    cOH_t = grad0(cOH, t)
    storage = RH * (theta * cH_t + theta_t * cH) - (theta * cOH_t + theta_t * cOH)
    res = storage + grad0(Ja, x) - S_pb
    return tf.where(tf.math.is_finite(res), res, tf.zeros_like(res))


def train_full_coupled_joint(ab_previous, pb_previous, n_iter):
    Hjoint = AcidBaseNet(layers_scalar, (wA, bA, aA), (w_phi_A, b_phi_A, a_phi_A),
                         init_from=ab_previous, freeze_k=0,
                         include_eo=H_PAR["include_eo"],
                         leftBC=CFG["FIELDS"]["HPLUS"]["BOUNDARY"]["left_BC"],
                         rightBC=CFG["FIELDS"]["HPLUS"]["BOUNDARY"]["right_BC"],
                         JH_left_true_val=JH_left, JOH_left_true_val=JOH_left,
                         JH_right_true_val=JH_right, JOH_right_true_val=JOH_right,
                         a_ic_val=A_ic,
                         chem_kernel=None, pb_weights=None,
                         include_pb_source=False)
    chem_live = ChemEquilKernel(
        PB_CHEM,
        (wA, bA, aA),
        (Hjoint.mlp.weights, Hjoint.mlp.biases, Hjoint.mlp.A),
        SURF_INIT,
    )
    Pjoint = PbInvNet(layers_scalar, (wA, bA, aA), (w_phi_A, b_phi_A, a_phi_A), chem_live,
                      leftBC=PB_leftBC, rightBC=PB_rightBC,
                      c_ic_water=float(PB_INIT["c_ic"]),
                      J_left_true_val=float(PB_BC.get("left", {}).get("J", 0.0)),
                      dir_left_c=_dir_c("left"), dir_right_c=_dir_c("right"),
                      init_from=pb_previous, freeze_k=0,
                      active_gate_weights=pb_previous,
                      stop_acid_in_pb=False)
    Hjoint.sess = Pjoint.sess

    finite = lambda z: tf.where(tf.math.is_finite(z), z, tf.zeros_like(z))
    joint_res = _coupled_acid_residual(Hjoint, Pjoint, chem_live, Hjoint.t_res, Hjoint.x_res,
                                       active_gate_weights=pb_previous)
    joint_res_nd = finite(joint_res / tf.maximum(Hjoint.res_a_ref, 1e-12))
    acid_parts = {k: v for k, v in Hjoint.loss_parts.items() if k != "res_a"}
    acid_parts["res_a_coupled"] = Hjoint.residual_weight_ph * tf.reduce_mean(tf.square(joint_res_nd))
    acid_loss = tf.add_n(list(acid_parts.values()))
    pb_loss = Pjoint.loss
    joint_loss = float(FULLC.get("acid_loss_weight", 1.0)) * acid_loss + float(FULLC.get("pb_loss_weight", 1.0)) * pb_loss

    joint_vars = [v for v in (Hjoint.mlp.weights + Hjoint.mlp.biases + Hjoint.mlp.A +
                              Pjoint.mlp.weights + Pjoint.mlp.biases + Pjoint.mlp.A)
                  if getattr(v, "trainable", True)]
    scope = "full_joint_adam_%d" % len(tf.compat.v1.global_variables())
    with tf.compat.v1.variable_scope(scope):
        opt = tf.compat.v1.train.AdamOptimizer(float(FULLC.get("adam_lr", 2.0e-4)))
        grads_vars = [(g, v) for g, v in opt.compute_gradients(joint_loss, var_list=joint_vars) if g is not None]
        grads, vars_ = zip(*grads_vars)
        safe_grads = [tf.where(tf.math.is_finite(g), g, tf.zeros_like(g)) for g in grads]
        grad_norm = tf.linalg.global_norm(safe_grads)
        safe_grads, _ = tf.clip_by_global_norm(safe_grads, float(FULLC.get("grad_clip", 1.0)))
        train_op = opt.apply_gradients(list(zip(safe_grads, vars_)))
    all_vars = tf.compat.v1.global_variables()
    init_flags = Pjoint.sess.run([tf.compat.v1.is_variable_initialized(v) for v in all_vars])
    uninit_vars = [v for v, ok in zip(all_vars, init_flags) if not ok]
    if uninit_vars:
        Pjoint.sess.run(tf.compat.v1.variables_initializer(uninit_vars))

    n_iter = int(n_iter)
    batch_size = int(FULLC.get("batch_size", TRNC.get("batch_size", 512)))
    print_every = max(int(FULLC.get("print_every", 100)), 1)
    mass_t_all = PB_t_right
    mass_bs = min(int(PB_PAR.get("mass_balance_batch_size", 128)), mass_t_all.shape[0])
    acid_names = sorted(acid_parts.keys())
    pb_names = sorted(Pjoint.loss_parts.keys())
    hist = []
    best_loss = np.inf
    best_it = -1
    best_ab_weights = None
    best_pb_weights = None

    def _ab_snapshot():
        return Pjoint.sess.run([Hjoint.mlp.weights, Hjoint.mlp.biases, Hjoint.mlp.A])

    def _restore_weights(ab_weights, pb_weights):
        assign_ops = []
        for vars_group, vals_group in zip((Hjoint.mlp.weights, Hjoint.mlp.biases, Hjoint.mlp.A), ab_weights):
            assign_ops.extend([tf.compat.v1.assign(v, val) for v, val in zip(vars_group, vals_group)])
        for vars_group, vals_group in zip((Pjoint.mlp.weights, Pjoint.mlp.biases, Pjoint.mlp.A), pb_weights):
            assign_ops.extend([tf.compat.v1.assign(v, val) for v, val in zip(vars_group, vals_group)])
        Pjoint.sess.run(assign_ops)

    for it in range(n_iter):
        bs = min(batch_size, t_res.shape[0])
        idx = np.random.choice(t_res.shape[0], bs, replace=(t_res.shape[0] < bs))
        tr, xr = t_res[idx, :], x_res[idx, :]
        idxm = np.random.choice(mass_t_all.shape[0], mass_bs, replace=(mass_t_all.shape[0] < mass_bs))
        feed = {
            Hjoint.t_res: tr, Hjoint.x_res: xr,
            Hjoint.t_ic: t_ic, Hjoint.x_ic: x_ic,
            Hjoint.t_left: H_t_left, Hjoint.x_left: H_x_left,
            Hjoint.t_right: H_t_right, Hjoint.x_right: H_x_right,
            Pjoint.t_res: tr, Pjoint.x_res: xr,
            Pjoint.t_ic: t_ic, Pjoint.x_ic: x_ic,
            Pjoint.t_left: PB_t_left, Pjoint.x_left: PB_x_left,
            Pjoint.t_right: PB_t_right, Pjoint.x_right: PB_x_right,
            Pjoint.t_mass: mass_t_all[idxm, :],
        }
        wfeed, _, _, _ = Hjoint._weight_feed(it, n_iter)
        feed.update(wfeed)
        if it == 0:
            initial_loss = float(Pjoint.sess.run(joint_loss, feed))
            if np.isfinite(initial_loss):
                best_loss = initial_loss
                best_it = -1
                best_ab_weights = _ab_snapshot()
                best_pb_weights = Pjoint.export_weights()
        Pjoint.sess.run(train_op, feed)
        if it % print_every == 0:
            vals = Pjoint.sess.run(
                [joint_loss, grad_norm, acid_loss, pb_loss] +
                [acid_parts[n] for n in acid_names] + [Pjoint.loss_parts[n] for n in pb_names],
                feed,
            )
            loss_now = float(vals[0])
            hist.append(loss_now)
            if np.isfinite(loss_now) and loss_now < best_loss:
                best_loss = loss_now
                best_it = it
                best_ab_weights = _ab_snapshot()
                best_pb_weights = Pjoint.export_weights()
            acid_vals = vals[4:4 + len(acid_names)]
            pb_vals = vals[4 + len(acid_names):]
            print(f"[FullJoint] It {it:5d} L={vals[0]:.3e} | grad={vals[1]:.2e} | acid={vals[2]:.3e} pb={vals[3]:.3e} | " +
                  ", ".join([f"A:{n}={v:.2e}" for n, v in zip(acid_names, acid_vals)]) + " | " +
                  ", ".join([f"Pb:{n}={v:.2e}" for n, v in zip(pb_names, pb_vals)]))

    if best_ab_weights is not None and best_pb_weights is not None:
        _restore_weights(best_ab_weights, best_pb_weights)
        print(f"[FullJoint] restored best checkpoint: It {best_it}, L={best_loss:.3e}")
    ab_weights = _ab_snapshot()
    pb_weights = Pjoint.export_weights()
    Hjoint.loss_hist = hist
    Hjoint.best_loss = best_loss
    Hjoint.best_iteration = best_it
    Pjoint.loss_hist = hist
    Pjoint.best_loss = best_loss
    Pjoint.best_iteration = best_it
    return Hjoint, Pjoint, ab_weights, pb_weights, hist


# ---------------------------
# 8) Build layers & BC samples
# ---------------------------
input_dim, output_dim = 2, 1
hidden, depth = NETC["width"], NETC["hidden_layers"]
layers_scalar = [input_dim] + [hidden]*depth + [output_dim]

# faces (water uses new config)
W_left_face  = CFG["FIELDS"]["WATER"]["BOUNDARY"]["left"].get("face", "LEFT")
W_right_face = CFG["FIELDS"]["WATER"]["BOUNDARY"]["right"].get("face", "RIGHT")
W_t_left,  W_x_left  = face_samples(W_left_face,  DATASET["n_left"])
W_t_right, W_x_right = face_samples(W_right_face, DATASET["n_right"])

# electric & ions faces (unchanged)
E_t_left,  E_x_left  = face_samples(ANODE_FACE,   DATASET["n_left"])
E_t_right, E_x_right = face_samples(CATHODE_FACE, DATASET["n_right"])
H_flux_tmin = float(DOMAIN["tmin"]) + float(H_PAR.get("bc_tmin_eps", 0.0))
H_t_left,  H_x_left  = face_samples(ANODE_FACE,   DATASET["n_left"], tmin_override=H_flux_tmin)
H_t_right, H_x_right = face_samples(CATHODE_FACE, DATASET["n_right"], tmin_override=H_flux_tmin)
PB_t_left, PB_x_left = face_samples("LEFT",  DATASET["n_left"])
PB_t_right,PB_x_right= face_samples("RIGHT", DATASET["n_right"])

# ---------------------------
# 8) 辅助：H+ 边界（注入为正、抽出为负），左右独立
# ---------------------------
def _theta_from_psi_num(psi, soil):
    h = float(psi)
    theta_s = float(soil["theta_s"])
    theta_r = float(soil["theta_r"])
    if h >= 0.0:
        return theta_s + float(soil.get("Ss_theta", 0.0)) * h
    n = float(soil["n"])
    m = 1.0 - 1.0 / n
    alpha = float(soil["alpha"])
    s = max(-alpha * h, 0.0)
    return theta_r + (theta_s - theta_r) * (1.0 + s**n) ** (-m)


def _initial_sigma_eff_num():
    ep = CFG["FIELDS"]["ELECTRIC"]["PARAM"]
    sigma_sat = float(ep.get("sigma_sat", ep.get("sigma_const", 7.425e-2)))
    model = str(ep.get("sigma_model", "constant")).lower()
    if not (model.startswith("archie") or model.startswith("rhoades")):
        return sigma_sat
    soil = CFG["FIELDS"]["WATER"]["PARAM"]["SOIL"]
    theta0 = CFG.get("SCENARIO", {}).get("theta0")
    if theta0 is None:
        theta0 = _theta_from_psi_num(CFG["FIELDS"]["WATER"]["INITIAL"].get("psi_ic", 0.0), soil)
    if model.startswith("rhoades") or model == "archie_rhoades":
        theta = max(float(theta0), 0.0)
        sigma = float(ep.get("sigma_s", 1.8e-2)) + float(ep.get("sigma_b", 2.5e-1)) * theta * (float(ep.get("rhoades_a", 1.3)) * theta + float(ep.get("rhoades_b", -0.2)))
        return max(sigma, 1e-12)
    theta_r = float(soil["theta_r"])
    theta_s = float(soil["theta_s"])
    Se = np.clip((float(theta0) - theta_r) / max(theta_s - theta_r, 1e-12), 0.0, 1.0)
    residual = float(ep.get("sigma_res_frac", 0.05))
    exponent = float(ep.get("archie_saturation_exp", 2.0))
    return sigma_sat * (residual + (1.0 - residual) * Se**exponent)


def _acidbase_current_density_from_cfg():
    Hbd = CFG["FIELDS"]["HPLUS"]["BOUNDARY"]
    if Hbd.get("I_app_Aperm2") is not None:
        return float(Hbd["I_app_Aperm2"])
    ep = CFG["FIELDS"]["ELECTRIC"]["PARAM"]
    el = CFG["FIELDS"]["ELECTRIC"]["BOUNDARY"]["electrodes"]
    L = max(float(CFG["DOMAIN"]["xmax"] - CFG["DOMAIN"]["xmin"]), 1e-12)
    dphi = abs(float(el["phi_anode"]) - float(el["phi_cathode"]))
    if str(Hbd.get("faraday_current_mode", "fixed_current")).lower() == "archie_voltage":
        sigma = _initial_sigma_eff_num()
    else:
        sigma = float(ep.get("sigma_sat", ep.get("sigma_const", 7.425e-2)))
    return sigma * dphi / L


def acidbase_flux_targets_from_cfg():
    Hbd = CFG["FIELDS"]["HPLUS"]["BOUNDARY"]
    Fnum = float(CFG["GLOBAL"]["CONSTANTS"]["F"])
    Jmag_day = float(Hbd.get("current_efficiency", 1.0)) * _acidbase_current_density_from_cfg() / Fnum * SEC_PER_DAY
    JH_left   = signed_flux_along_pos_x(Jmag_day, "LEFT",  "into_domain")
    JOH_left  = 0.0
    JH_right  = 0.0
    JOH_right = signed_flux_along_pos_x(Jmag_day, "RIGHT", "into_domain")
    return JH_left, JOH_left, JH_right, JOH_right


def acidbase_dirichlet_pH_targets_from_cfg(t_day):
    Hbd = CFG["FIELDS"]["HPLUS"]["BOUNDARY"]
    if str(Hbd.get("dirichlet_mode", "static")).lower() not in ("reservoir", "reservoir_ph"):
        return float(Hbd.get("pH_left", H_INIT["pH_ic"])), float(Hbd.get("pH_right", H_INIT["pH_ic"]))
    L = max(float(CFG["DOMAIN"]["xmax"] - CFG["DOMAIN"]["xmin"]), 1e-12)
    J_day = float(Hbd.get("current_efficiency", 1.0)) * _acidbase_current_density_from_cfg() / float(CFG["GLOBAL"]["CONSTANTS"]["F"]) * SEC_PER_DAY
    depth = max(float(Hbd.get("reservoir_depth_m", L)), 1e-12)
    cH0 = float(H_INIT["c_ic"])
    cOH0 = float(CFG["GLOBAL"]["CHEM"]["Kw"]) / max(cH0, 1e-30)
    cH = cH0 + (J_day / depth) * max(float(t_day) - float(CFG["DOMAIN"]["tmin"]), 0.0)
    cOH = cOH0 + (J_day / depth) * max(float(t_day) - float(CFG["DOMAIN"]["tmin"]), 0.0)
    pH_left = -np.log10(max(cH / 1000.0, 1e-14))
    pH_right = 14.0 + np.log10(max(cOH / 1000.0, 1e-14))
    if bool(Hbd.get("reservoir_clip_to_config", True)):
        pH_left = max(pH_left, float(Hbd.get("pH_left", pH_left)))
        pH_right = min(pH_right, float(Hbd.get("pH_right", pH_right)))
    return float(np.clip(pH_left, H_PAR.get("pH_min", 0.0), H_PAR.get("pH_max", 14.5))), float(np.clip(pH_right, H_PAR.get("pH_min", 0.0), H_PAR.get("pH_max", 14.5)))

# backward-compatible helper alias
hplus_flux_targets_from_cfg = acidbase_flux_targets_from_cfg

# ---------------------------
# 9) Pipeline  (Electric -> Water -> Acid/Base -> PbΨ)
# ---------------------------
print("== Stage A: Electric(phi) ==")
ElecA=ElectricNet(layers_scalar, init_from=None, freeze_k=0,
                  phi_left=phi_anode, phi_right=phi_cathode)
ElecA.train(TRNC["elec_A_iters"], batch=True, batch_size=TRNC["batch_size"],
            face_feed={"left":(E_t_left,E_x_left), "right":(E_t_right,E_x_right)})
if "record_loss_history" in globals(): record_loss_history("Stage A Electric", ElecA)
w_phi_A,b_phi_A,a_phi_A=ElecA.export_weights()

print("== Stage B: Water(phi from Stage A) ==")
if use_fixed_saturated_water():
    theta_s_val = float(CFG["FIELDS"]["WATER"]["PARAM"]["SOIL"].get("theta_s", 0.50))
    print(f"[Water] saturated theta case detected: skip WaterNet training and fix psi=0, theta={theta_s_val:.6g}.")
    WaterA = None
    wA, bA, aA = make_constant_water_weights(layers_scalar, psi_value=0.0)
else:
    WaterA=WaterNet(layers_scalar,
                    include_eo_in_water=CFG["FIELDS"]["WATER"]["PARAM"]["include_eo"],
                    elec_weights=(w_phi_A,b_phi_A,a_phi_A),
                    wbc_cfg=CFG["FIELDS"]["WATER"]["BOUNDARY"])
    WaterA.train(TRNC["water_iters"], batch=True, batch_size=TRNC["batch_size"],
                 face_feed={"left":(W_t_left,W_x_left), "right":(W_t_right,W_x_right)})
    if "record_loss_history" in globals(): record_loss_history("Stage B Water", WaterA)
    wA,bA,aA=WaterA.export_weights()

print("== Stage A2: Electric(Archie sigma_eff from water) ==")
ElecA=ElectricNet(layers_scalar, init_from=(w_phi_A,b_phi_A,a_phi_A), freeze_k=0,
                  phi_left=phi_anode, phi_right=phi_cathode,
                  water_weights=(wA,bA,aA))
ElecA.train(TRNC["elec_A_iters"], batch=True, batch_size=TRNC["batch_size"],
            face_feed={"left":(E_t_left,E_x_left), "right":(E_t_right,E_x_right)})
if "record_loss_history" in globals(): record_loss_history("Stage A2 Electric sigma(theta)", ElecA)
w_phi_A,b_phi_A,a_phi_A=ElecA.export_weights()

hydro_elec_outer = int(TRNC.get("hydro_elec_outer_iters", 1))
if (not use_fixed_saturated_water()) and hydro_elec_outer > 0:
    for he_iter in range(hydro_elec_outer):
        print(f"== Stage HE{he_iter+1}: Water/Electric sequential closure ==")
        WaterA=WaterNet(layers_scalar,
                        include_eo_in_water=CFG["FIELDS"]["WATER"]["PARAM"]["include_eo"],
                        elec_weights=(w_phi_A,b_phi_A,a_phi_A),
                        wbc_cfg=CFG["FIELDS"]["WATER"]["BOUNDARY"],
                        init_from=(wA,bA,aA))
        WaterA.train(int(TRNC.get("hydro_elec_water_iters", TRNC["water_iters"])),
                     batch=True, batch_size=TRNC["batch_size"],
                     face_feed={"left":(W_t_left,W_x_left), "right":(W_t_right,W_x_right)})
        if "record_loss_history" in globals(): record_loss_history(f"Stage HE{he_iter+1} Water", WaterA)
        wA,bA,aA=WaterA.export_weights()

        ElecA=ElectricNet(layers_scalar, init_from=(w_phi_A,b_phi_A,a_phi_A), freeze_k=0,
                          phi_left=phi_anode, phi_right=phi_cathode,
                          water_weights=(wA,bA,aA))
        ElecA.train(int(TRNC.get("hydro_elec_elec_iters", TRNC["elec_A_iters"])),
                    batch=True, batch_size=TRNC["batch_size"],
                    face_feed={"left":(E_t_left,E_x_left), "right":(E_t_right,E_x_right)})
        if "record_loss_history" in globals(): record_loss_history(f"Stage HE{he_iter+1} Electric", ElecA)
        w_phi_A,b_phi_A,a_phi_A=ElecA.export_weights()

# --- Acid/base initial state a = cH - cOH
H_c_ic = float(H_INIT["c_ic"])
OH_c_ic = float(CFG["GLOBAL"]["CHEM"]["Kw"]) / max(H_c_ic, 1e-30)
A_ic = H_c_ic - OH_c_ic
CFG["FIELDS"]["HPLUS"]["INITIAL"]["a_ic"] = A_ic

# --- Acid/base boundary targets
JH_left, JOH_left, JH_right, JOH_right = acidbase_flux_targets_from_cfg()
if CFG["FIELDS"]["HPLUS"]["BOUNDARY"]["left_BC"].lower() == "dirichlet" and CFG["FIELDS"]["HPLUS"]["BOUNDARY"]["right_BC"].lower() == "dirichlet":
    pHL_end, pHR_end = acidbase_dirichlet_pH_targets_from_cfg(DOMAIN["tmax"])
    print(f"[Acid/Base pH target] mode={H_BC.get('dirichlet_mode', 'static')} | final left={pHL_end:.3f}, right={pHR_end:.3f}")
else:
    print(f"[Acid/Base flux target] mode={H_BC.get('faraday_current_mode', 'fixed_current')} | nominal: H_left={JH_left:.6e}, OH_left={JOH_left:.6e}, H_right={JH_right:.6e}, OH_right={JOH_right:.6e}, Ja_left={JH_left-JOH_left:.6e}, Ja_right={JH_right-JOH_right:.6e} mol m^-2 day^-1")

print("== Stage C: Acid/base (a = c_H - c_OH with Kw closure) ==")
Hplus=AcidBaseNet(layers_scalar, (wA,bA,aA), (w_phi_A,b_phi_A,a_phi_A),
                  init_from=None, freeze_k=TRNC["freeze_k_c"],
                  include_eo=H_PAR["include_eo"],
                  leftBC=CFG["FIELDS"]["HPLUS"]["BOUNDARY"]["left_BC"],
                  rightBC=CFG["FIELDS"]["HPLUS"]["BOUNDARY"]["right_BC"],
                  JH_left_true_val=JH_left, JOH_left_true_val=JOH_left,
                  JH_right_true_val=JH_right, JOH_right_true_val=JOH_right,
                  a_ic_val=A_ic)
Hplus.train(TRNC["hplus_iters"], batch=True, batch_size=TRNC.get("hplus_batch_size", TRNC["batch_size"]),
            face_feed={"left":(H_t_left,H_x_left), "right":(H_t_right,H_x_right)})
record_loss_history("Stage C Acid/Base one-way", Hplus)
w_ab,b_ab,a_ab=Hplus.export_weights()
# backward-compatible aliases
wH,bH,aH = w_ab,b_ab,a_ab

# ---- Chemistry equilibrium kernel (uses Water + Acid/Base net)
PB_CHEM = CFG["FIELDS"]["PB"]["CHEM"]
PB_PAR["solver"] = "pinn_equilibrium"
chem = ChemEquilKernel(PB_CHEM, (wA,bA,aA), (w_ab,b_ab,a_ab), SURF_INIT)

# ---- Pb invariant transport (Ψ_Pb)
print("== Stage D: Pb component-PINN transport + supported local equilibrium ==")
PB_leftBC, PB_rightBC = PB_BC["left"]["type"].lower(), PB_BC["right"]["type"].lower()
def _dir_c(side): return PB_BC.get(side,{}).get("c")
PbPsi = PbInvNet(layers_scalar, (wA,bA,aA), (w_phi_A,b_phi_A,a_phi_A), chem,
                 leftBC=PB_leftBC, rightBC=PB_rightBC,
                 c_ic_water=float(PB_INIT["c_ic"]),
                 J_left_true_val=float(PB_BC.get("left", {}).get("J", 0.0)),
                 dir_left_c=_dir_c("left"), dir_right_c=_dir_c("right"))
PbPsi.train(TRNC["pb_inv_iters"], batch=True, batch_size=TRNC["batch_size"],
            face_feed={"left": (PB_t_left, PB_x_left), "right": (PB_t_right, PB_x_right)})
record_loss_history("Stage D Pb one-way", PbPsi)
w_pb, b_pb, a_pb = PbPsi.export_weights()

# ---- Algorithm 1 style sequential block Gauss-Seidel iteration.
# The base C/D solves above provide the compatible n=0 state.  Each subsequent
# acid/base block sees only the prior Pb/pH chemistry; the Pb block then sees
# the just-updated acid/base field.  There is no simultaneous loss.
previous_state = coupling_state((wA, bA, aA), (w_ab, b_ab, a_ab), (w_pb, b_pb, a_pb))
coupling_history = [{
    "outer_iteration": 0,
    "acid_parameter_rel": 0.0,
    "pb_parameter_rel": 0.0,
    "pH_rms": 0.0,
    "pH_max": 0.0,
    "Psi_rel": 0.0,
    "fixed_rel": 0.0,
    "mass_identity_max": previous_state["mass_identity_max"],
}]
print("[Sequential] n=0 mass-identity max = %.3e" % previous_state["mass_identity_max"])

for outer_iter in range(1, max(int(SEQC.get("outer_max_iters", 0)), 0) + 1):
    print("== Sequential iteration %d/%d: frozen old Pb -> acid/base -> Pb ==" %
          (outer_iter, int(SEQC.get("outer_max_iters", 0))))
    ab_previous = (w_ab, b_ab, a_ab)
    pb_previous = (w_pb, b_pb, a_pb)

    # This chemistry object carries the previous acid/base field, exactly as
    # the lagged field in the fixed-stress split of Algorithm 1.
    chem_previous = ChemEquilKernel(PB_CHEM, (wA, bA, aA), ab_previous, SURF_INIT)
    Hplus = AcidBaseNet(layers_scalar, (wA,bA,aA), (w_phi_A,b_phi_A,a_phi_A),
                        init_from=ab_previous, freeze_k=TRNC["freeze_k_c"],
                        include_eo=H_PAR["include_eo"],
                        leftBC=CFG["FIELDS"]["HPLUS"]["BOUNDARY"]["left_BC"],
                        rightBC=CFG["FIELDS"]["HPLUS"]["BOUNDARY"]["right_BC"],
                        JH_left_true_val=JH_left, JOH_left_true_val=JOH_left,
                        JH_right_true_val=JH_right, JOH_right_true_val=JOH_right,
                        a_ic_val=A_ic,
                        chem_kernel=chem_previous, pb_weights=pb_previous,
                        include_pb_source=True)
    train_acidbase_sequential_block(
        Hplus, int(SEQC.get("acid_iters", 1800)),
        face_feed={"left": (H_t_left, H_x_left), "right": (H_t_right, H_x_right)},
    )
    record_loss_history(f"Sequential {outer_iter} Acid/Base", Hplus)
    w_ab, b_ab, a_ab = Hplus.export_weights()
    wH, bH, aH = w_ab, b_ab, a_ab

    chem = ChemEquilKernel(PB_CHEM, (wA, bA, aA), (w_ab, b_ab, a_ab), SURF_INIT)
    PbPsi = PbInvNet(layers_scalar, (wA,bA,aA), (w_phi_A,b_phi_A,a_phi_A), chem,
                     leftBC=PB_leftBC, rightBC=PB_rightBC,
                     c_ic_water=float(PB_INIT["c_ic"]),
                     J_left_true_val=float(PB_BC.get("left", {}).get("J", 0.0)),
                     dir_left_c=_dir_c("left"), dir_right_c=_dir_c("right"),
                     init_from=pb_previous, freeze_k=0,
                     active_gate_weights=pb_previous)
    PbPsi.train(int(SEQC.get("pb_iters", 2600)), batch=True, batch_size=TRNC["batch_size"],
                face_feed={"left": (PB_t_left, PB_x_left), "right": (PB_t_right, PB_x_right)})
    record_loss_history(f"Sequential {outer_iter} Pb", PbPsi)
    w_pb, b_pb, a_pb = PbPsi.export_weights()

    current_state = coupling_state((wA, bA, aA), (w_ab, b_ab, a_ab), (w_pb, b_pb, a_pb))
    metrics = coupling_field_metrics(previous_state, current_state)
    metrics.update({
        "outer_iteration": outer_iter,
        "acid_parameter_rel": relative_parameter_change(ab_previous, (w_ab, b_ab, a_ab)),
        "pb_parameter_rel": relative_parameter_change(pb_previous, (w_pb, b_pb, a_pb)),
    })
    coupling_history.append(metrics)
    print("[Sequential] n=%d | dtheta_a=%.3e dtheta_pb=%.3e | pH_rms=%.3e pH_max=%.3e Psi_rel=%.3e fixed_rel=%.3e mass_id=%.3e" %
          (outer_iter, metrics["acid_parameter_rel"], metrics["pb_parameter_rel"],
           metrics["pH_rms"], metrics["pH_max"], metrics["Psi_rel"],
           metrics["fixed_rel"], metrics["mass_identity_max"]))
    parameter_ok = max(metrics["acid_parameter_rel"], metrics["pb_parameter_rel"]) <= float(SEQC.get("parameter_rel_tol", 5e-2))
    field_ok = (metrics["pH_rms"] <= float(SEQC.get("pH_rms_tol", 5e-2)) and
                metrics["pH_max"] <= float(SEQC.get("pH_max_tol", 2.5e-1)) and
                metrics["Psi_rel"] <= float(SEQC.get("psi_rel_tol", 1e-2)) and
                metrics["fixed_rel"] <= float(SEQC.get("fixed_rel_tol", 1e-2)))
    previous_state = current_state
    if outer_iter >= int(SEQC.get("outer_min_iters", 1)) and parameter_ok and field_ok:
        print("[Sequential] parameter and field convergence reached at iteration %d." % outer_iter)
        break
else:
    if int(SEQC.get("outer_max_iters", 0)) > 0:
        print("[Sequential] maximum outer iterations reached; inspect coupling_history before accepting the field.")

# ---- Final simultaneous full-coupled fine-tune.
# One optimizer updates Acid/Base and Pb together; water/electric stay frozen.
FULLC = CFG["NUMERICS"].get("FULL_COUPLED", {})
full_coupled_history = []
if bool(FULLC.get("enabled", False)):
    print("== Final simultaneous full-coupled fine-tune: Acid/Base + Pb ==")
    ab_previous = (w_ab, b_ab, a_ab)
    pb_previous = (w_pb, b_pb, a_pb)
    previous_state = coupling_state((wA, bA, aA), ab_previous, pb_previous)

    Hplus, PbPsi, (w_ab, b_ab, a_ab), (w_pb, b_pb, a_pb), joint_loss_hist = train_full_coupled_joint(
        ab_previous,
        pb_previous,
        int(FULLC.get("iters", 1600)),
    )
    wH, bH, aH = w_ab, b_ab, a_ab
    chem = ChemEquilKernel(PB_CHEM, (wA, bA, aA), (w_ab, b_ab, a_ab), SURF_INIT)
    record_loss_values("FullCoupled Joint Acid/Base+Pb", joint_loss_hist)

    current_state = coupling_state((wA, bA, aA), (w_ab, b_ab, a_ab), (w_pb, b_pb, a_pb))
    metrics = coupling_field_metrics(previous_state, current_state)
    metrics.update({
        "coupled_iteration": 1,
        "acid_parameter_rel": relative_parameter_change(ab_previous, (w_ab, b_ab, a_ab)),
        "pb_parameter_rel": relative_parameter_change(pb_previous, (w_pb, b_pb, a_pb)),
        "mode": "simultaneous",
    })
    full_coupled_history.append(metrics)
    print("[FullCoupled/Joint] dtheta_a=%.3e dtheta_pb=%.3e | pH_rms=%.3e pH_max=%.3e Psi_rel=%.3e fixed_rel=%.3e mass_id=%.3e" %
          (metrics["acid_parameter_rel"], metrics["pb_parameter_rel"],
           metrics["pH_rms"], metrics["pH_max"], metrics["Psi_rel"],
           metrics["fixed_rel"], metrics["mass_identity_max"]))

print("== All stages done ==")


# ---------------------------
# 10) sanity check (units)
# ---------------------------
with tf.compat.v1.Session() as _s:
    th_mid = _s.run(theta_function(tf.constant([[-1.0]], tf.float32)))
    keo_mid = _s.run(keo_of_theta(tf.constant(th_mid, tf.float32)))
print("[Units] theta(mid) =", float(np.asarray(th_mid).ravel()[0]), "  k_eo(mid) [m^2/(V?day)] =", float(np.asarray(keo_mid).ravel()[0]))
print("DONE.")


In [ ]:
# ===== Plotting cell 1: collect outputs from the CURRENT trained model =====
import os
import json
import math
import re
from pathlib import Path
from collections import OrderedDict

import numpy as np
import os
os.environ.setdefault("MPLBACKEND", "Agg")
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
try:
    from IPython.display import display
except Exception:
    def display(_obj):
        return None

PLOT_VERSION = "stress_split_sequential_v1"
def _resolve_oh_plot_root():
    env_root = os.environ.get("OH_PLOT_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    for var_name in ("__vsc_ipynb_file__", "__file__"):
        nb_file = globals().get(var_name)
        if nb_file:
            return Path(nb_file).expanduser().resolve().parent / "OH_model_outputs_stress_split_sequential"

    cwd = Path.cwd().resolve()
    for candidate in (
        cwd / "OH-DEA.ipynb",
        cwd / "OH.ipynb",
        cwd / "OH" / "OH-DEA.ipynb",
        cwd / "OH" / "OH.ipynb",
    ):
        if candidate.exists():
            return candidate.parent / "OH_model_outputs_stress_split_sequential"

    return cwd / "OH_model_outputs_stress_split_sequential"

PLOT_ROOT = _resolve_oh_plot_root()
try:
    os.chdir(PLOT_ROOT.parent)
except Exception:
    pass
print(f"[Plot setup] PLOT_ROOT={PLOT_ROOT}")
FIG_DIR = PLOT_ROOT / "figures"
EXPORT_DIR = PLOT_ROOT / "exports"
FIG_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

SHOW_INLINE = True
SAVE_FILES = True


def _need(name):
    if name not in globals():
        raise RuntimeError(f"[Plot] `{name}` not found. Run the training cell first.")


for _name in ["WaterA", "ElecA", "Hplus", "PbPsi", "chem", "CFG"]:
    _need(_name)
if WaterA is None:
    for _name in ["wA", "bA", "aA"]:
        _need(_name)


def safe_slug(text):
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9_.-]+", "_", text.strip())
    return text.strip("_") or "figure"


def show_and_save(fig, name, dpi=220):
    path = FIG_DIR / f"{safe_slug(name)}.png"
    if SAVE_FILES:
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
        print(f"[Plot] saved: {path}")
    if SHOW_INLINE:
        display(fig)
    plt.close(fig)
    return path


def finite_np(a, fill=0.0):
    a = np.asarray(a, dtype=float)
    finite = np.isfinite(a)
    if finite.any():
        hi = np.nanmax(a[finite])
        lo = np.nanmin(a[finite])
    else:
        hi = fill
        lo = fill
    return np.nan_to_num(a, nan=fill, posinf=hi, neginf=lo)


def pH_from_cH(cH_m3):
    return -np.log10(np.clip(cH_m3, 1e-30, None) / 1000.0)


def pOH_from_cOH(cOH_m3):
    return -np.log10(np.clip(cOH_m3, 1e-30, None) / 1000.0)


def paper_times():
    domain = CFG["DOMAIN"]
    tmin = float(domain["tmin"])
    tmax = float(domain["tmax"])
    step = 0.5
    wanted = list(np.arange(tmin, tmax + 0.5 * step, step, dtype=float))
    if not any(abs(tmax - tv) < 1e-9 for tv in wanted):
        wanted.append(tmax)
    out = []
    for tv in wanted:
        tv = min(max(float(tv), tmin), tmax)
        if not any(abs(tv - old) < 1e-9 for old in out):
            out.append(tv)
    return out


def sample_positions_cm():
    xmin = float(CFG["DOMAIN"]["xmin"])
    xmax = float(CFG["DOMAIN"]["xmax"])
    L_cm = (xmax - xmin) * 100.0
    # Old manuscript positions were 10/30/50/70/90 percent of the column.
    return np.array([0.10, 0.30, 0.50, 0.70, 0.90], dtype=float) * L_cm


def _reshape(arr, shape):
    return finite_np(np.asarray(arr).reshape(shape))


def _eval(sess, tensors, feed):
    return sess.run(tensors, feed)


def _feed_grid(t_grid, x_grid):
    T2, X2 = np.meshgrid(t_grid.astype(np.float32), x_grid.astype(np.float32), indexing="ij")
    TT = T2.reshape(-1, 1).astype(np.float32)
    XX = X2.reshape(-1, 1).astype(np.float32)
    T_ph = tf.compat.v1.placeholder(tf.float32, [None, 1])
    X_ph = tf.compat.v1.placeholder(tf.float32, [None, 1])
    X_cat = tf.concat([T_ph, X_ph], 1)
    feed = {T_ph: TT, X_ph: XX}
    return T2, X2, TT, XX, T_ph, X_ph, X_cat, feed


def collect_current_outputs(pred_n=None):
    domain = CFG["DOMAIN"]
    dataset = CFG["NUMERICS"]["DATASET"]
    nx = int(pred_n or max(int(dataset.get("pred_n", 121)), 201))
    nt = int(max(int(dataset.get("pred_n", 121)), 121))
    xmin, xmax = float(domain["xmin"]), float(domain["xmax"])
    tmin, tmax = float(domain["tmin"]), float(domain["tmax"])
    x_grid = np.linspace(xmin, xmax, nx, dtype=np.float32)
    t_grid = np.unique(np.r_[np.linspace(tmin, tmax, nt, dtype=np.float32), np.array(paper_times(), dtype=np.float32)])
    T2, X2, TT, XX, T_ph, X_ph, X_cat, feed = _feed_grid(t_grid, x_grid)
    shape = T2.shape

    # Water and electric fields.
    # Evaluate exported water weights directly so q_eo/q_total always use
    # the final current ElecA instead of any stale WaterNet graph constants.
    if all(name in globals() for name in ["wA", "bA", "aA"]):
        psi_tf = water_head_from_weights(wA, bA, aA, T_ph, X_ph)
        theta_tf = theta_function(psi_tf)
        _theta_s_tf, _theta_r_tf, Se_tf = sat_vars(theta_tf)
        K_tf = K_function(psi_tf)
        psi_x_tf = tf.gradients(psi_tf, X_ph)[0]
        if psi_x_tf is None:
            psi_x_tf = tf.zeros_like(psi_tf)
        q_hyd_tf = -K_tf * psi_x_tf
        q_eo_tf = tf.zeros_like(q_hyd_tf)
        # If an electric field has already been trained, include the same
        # electroosmotic contribution used by Acid/Base and Pb transport.
        if "ElecA" in globals() and ElecA is not None:
            phi_for_q = ElecA.net_phi(X_cat)
            phi_x_for_q = tf.gradients(phi_for_q, X_ph)[0]
            if phi_x_for_q is not None:
                q_eo_tf = keo_of_theta(theta_tf) * phi_x_for_q
        q_total_tf = q_hyd_tf + q_eo_tf
        water_tensors = [psi_tf, theta_tf, Se_tf, K_tf, psi_x_tf, q_hyd_tf, q_eo_tf, q_total_tf]
        water_names = ["psi", "theta", "Se", "K_hyd", "psi_x", "q_hyd", "q_eo", "q_total"]
        water_vals = _eval(ElecA.sess if ("ElecA" in globals() and ElecA is not None) else Hplus.sess, water_tensors, feed)
    else:
        psi_tf = WaterA.net_psi(X_cat)
        water_tensors = [psi_tf]
        water_names = ["psi"]
        if "theta_function" in globals():
            theta_tf = theta_function(psi_tf)
            _theta_s_tf, _theta_r_tf, Se_tf = sat_vars(theta_tf)
            psi_x_tf = WaterA.net_psix(T_ph, X_ph)
            if psi_x_tf is None:
                psi_x_tf = tf.zeros_like(psi_tf)
            water_tensors += [theta_tf, Se_tf, K_function(psi_tf), psi_x_tf]
            water_names += ["theta", "Se", "K_hyd", "psi_x"]
        if hasattr(WaterA, "net_q_parts"):
            q_hyd_tf, q_eo_tf, q_total_tf = WaterA.net_q_parts(T_ph, X_ph)
            water_tensors += [q_hyd_tf, q_eo_tf, q_total_tf]
            water_names += ["q_hyd", "q_eo", "q_total"]
        elif hasattr(WaterA, "net_q_total"):
            water_tensors.append(WaterA.net_q_total(T_ph, X_ph))
            water_names.append("q_total")
        water_vals = _eval(WaterA.sess, water_tensors, feed)
    water = {name: _reshape(val, shape) for name, val in zip(water_names, water_vals)}
    if "theta" not in water:
        soil = CFG["FIELDS"]["WATER"]["PARAM"]["SOIL"]
        n = float(soil["n"]); m = 1.0 - 1.0 / n
        alpha = float(soil["alpha"])
        theta_r = float(soil["theta_r"]); theta_s = float(soil["theta_s"])
        s = np.maximum(-alpha * water["psi"], 0.0)
        water["theta"] = theta_r + (theta_s - theta_r) * np.power(1.0 + np.power(s, n), -m)
    if "q_total" not in water:
        water["q_total"] = np.full(shape, np.nan)
    if "q_hyd" not in water:
        water["q_hyd"] = np.full(shape, np.nan)
    if "q_eo" not in water:
        water["q_eo"] = np.full(shape, np.nan)

    phi_tf = ElecA.net_phi(X_cat)
    phi_x_tf = tf.gradients(phi_tf, X_ph)[0]
    phi, phi_x = _eval(ElecA.sess, [phi_tf, phi_x_tf], feed)
    electric = {
        "phi": _reshape(phi, shape),
        "phi_x": _reshape(phi_x, shape),
    }

    # Acid/base fields from the current AcidBaseNet.
    h_tensors = [Hplus.net_c(X_cat), Hplus.net_cOH(X_cat)]
    h_names = ["cH", "cOH"]
    if hasattr(Hplus, "_species_fields"):
        try:
            a, cH2, cOH2, cHx, cOHx, theta_ab, q_ab, phi_x_ab = Hplus._species_fields(T_ph, X_ph)
            h_tensors += [a, cHx, cOHx, theta_ab, q_ab, phi_x_ab]
            h_names += ["a", "cH_x", "cOH_x", "theta_ab", "q_ab", "phi_x_ab"]
        except Exception:
            pass
    try:
        JH = Hplus.net_JH(T_ph, X_ph)
        JOH = Hplus.net_JOH(T_ph, X_ph)
        Ja = JH - JOH
        h_tensors += [JH, JOH, Ja]
        h_names += ["JH", "JOH", "Ja"]
        if hasattr(Hplus, "net_acid_equiv_nonadv"):
            h_tensors.append(Hplus.net_acid_equiv_nonadv(T_ph, X_ph))
            h_names.append("Ja_F")
    except Exception:
        pass
    h_vals = _eval(Hplus.sess, h_tensors, feed)
    acid = {name: _reshape(val, shape) for name, val in zip(h_names, h_vals)}
    acid["pH"] = pH_from_cH(acid["cH"])
    acid["pOH"] = pOH_from_cOH(acid["cOH"])

    # Pb fields from the current Pb model. Current supported path: Psi_Pb PINN + equilibrium reconstruction.
    pb_model_kind = "legacy_dae_pinn" if hasattr(PbPsi, "net_vars") and hasattr(PbPsi, "species_from_vars") else "component_equilibrium_pinn"
    if pb_model_kind == "legacy_dae_pinn":
        Psi_tf, cfree_tf = PbPsi.net_vars(X_cat)
        species = PbPsi.species_from_vars(T_ph, X_ph, Psi_tf, cfree_tf)
        cPb_tf, SOPb_tf, SOH_tf, SOH2_tf, SOm_tf, Pp_tf, cOH_pb_tf, Pp_raw_tf, si_tf, z_eff_tf = species
        pb_tensors = [Psi_tf, cfree_tf, cPb_tf, SOPb_tf, SOH_tf, SOH2_tf, SOm_tf, Pp_tf, cOH_pb_tf, Pp_raw_tf, si_tf, z_eff_tf]
        pb_names = ["Psi", "Pb_free", "Pb_aq", "SOPb", "SOH", "SOH2", "SOm", "Pp", "cOH_pb", "Pp_raw", "SI_ln", "z_eff"]
    else:
        Psi_tf = PbPsi.net_Psi(X_cat)
        cPb_tf, SOPb_tf, SOH_tf, SOH2_tf, SOm_tf, Pp_tf, cOH_pb_tf = chem.reconstruct_pb_equil(T_ph, X_ph, Psi_tf)
        si_tf = tf.math.log((tf.maximum(cPb_tf, 0.0) * tf.pow(tf.maximum(cOH_pb_tf, 1e-30), chem.m) + 1e-30) / (chem.Ksp + 1e-30))
        pb_tensors = [Psi_tf, cPb_tf, SOPb_tf, SOH_tf, SOH2_tf, SOm_tf, Pp_tf, cOH_pb_tf, si_tf]
        pb_names = ["Psi", "Pb_aq", "SOPb", "SOH", "SOH2", "SOm", "Pp", "cOH_pb", "SI_ln"]

    try:
        J_raw, J_adv, J_diff, J_em, c_flux, c_x, q_pb, phi_x_pb, Deff, u_star = PbPsi.net_J_components(T_ph, X_ph)
        J_phys = J_adv + J_diff + J_em
        pb_tensors += [J_raw, J_adv, J_diff, J_em, J_phys, J_raw - J_phys, c_flux, c_x, q_pb, phi_x_pb, Deff, u_star]
        pb_names += ["J_raw", "J_adv", "J_diff", "J_em", "J_phys", "J_constitutive_gap", "Pb_aq_flux_eval", "Pb_aq_x", "q_pb", "phi_x_pb", "Deff", "u_star"]
    except Exception:
        pass

    pb_vals = _eval(PbPsi.sess, pb_tensors, feed)
    pb = {name: _reshape(val, shape) for name, val in zip(pb_names, pb_vals)}
    if "SI_ln" in pb:
        pb["SI_log10"] = pb["SI_ln"] / np.log(10.0)
    else:
        pb["SI_log10"] = np.full(shape, np.nan)

    pb["aq_bulk"] = water["theta"] * pb["Pb_aq"]
    pb["ads_bulk"] = pb["SOPb"]
    pb["ppt_bulk"] = pb["Pp"]
    pb["total_reconstructed"] = pb["aq_bulk"] + pb["ads_bulk"] + pb["ppt_bulk"]
    pb["mass_gap"] = pb["Psi"] - pb["total_reconstructed"]

    out = {
        "version": PLOT_VERSION,
        "model_kind": pb_model_kind,
        "scenario": CFG.get("SCENARIO", {}).get("cathode_drainage", "unknown"),
        "x": x_grid,
        "x_cm": x_grid * 100.0,
        "t": t_grid,
        "T": T2,
        "X": X2,
        "paper_times": paper_times(),
        "sample_positions_cm": sample_positions_cm(),
        "water": water,
        "electric": electric,
        "acid": acid,
        "pb": pb,
    }
    return out


def profile_at(arr2d, t_value, data=None):
    data = data or PLOT_DATA
    k = int(np.argmin(np.abs(data["t"] - float(t_value))))
    return np.asarray(arr2d)[k, :], float(data["t"][k]), k


def nearest_x_index_cm(x_cm_value, data=None):
    data = data or PLOT_DATA
    return int(np.argmin(np.abs(data["x_cm"] - float(x_cm_value))))


def summarize_current_outputs(data=None):
    data = data or PLOT_DATA
    final_t = data["paper_times"][-1]
    out = OrderedDict()
    for key, label in [
        ("pH", "pH"),
        ("cOH", "cOH"),
    ]:
        y, tt, _ = profile_at(data["acid"][key], final_t, data)
        out[f"{label}_final_left"] = float(y[0])
        out[f"{label}_final_right"] = float(y[-1])
    for key in ["Psi", "Pb_aq", "aq_bulk", "SOPb", "Pp", "total_reconstructed", "J_raw", "J_phys"]:
        if key in data["pb"]:
            y, tt, _ = profile_at(data["pb"][key], final_t, data)
            out[f"{key}_final_left"] = float(y[0])
            out[f"{key}_final_right"] = float(y[-1])
            out[f"{key}_final_mean"] = float(np.nanmean(y))
    out["mass_gap_abs_max"] = float(np.nanmax(np.abs(data["pb"]["mass_gap"])))
    return out


PLOT_DATA = collect_current_outputs()
PLOT_SUMMARY = summarize_current_outputs(PLOT_DATA)

# Backward-compatible names used by older manuscript-style plotting snippets.
x = PLOT_DATA["x"]
t = PLOT_DATA["t"]
X = PLOT_DATA["X"]
T = PLOT_DATA["T"]
psi2D = PLOT_DATA["water"]["psi"]
theta2D = PLOT_DATA["water"]["theta"]
phi2D = PLOT_DATA["electric"]["phi"]
cH2D = PLOT_DATA["acid"]["cH"]
cOH2D = PLOT_DATA["acid"]["cOH"]
pH2D = PLOT_DATA["acid"]["pH"]
Psi2D = PLOT_DATA["pb"]["Psi"]
cPb2D = PLOT_DATA["pb"]["Pb_aq"]
SOPb2D = PLOT_DATA["pb"]["SOPb"]
Pp2D = PLOT_DATA["pb"]["Pp"]
aq_bulk2D = PLOT_DATA["pb"]["aq_bulk"]
ads_bulk2D = PLOT_DATA["pb"]["ads_bulk"]
ppt_bulk2D = PLOT_DATA["pb"]["ppt_bulk"]
tot_bulk2D = PLOT_DATA["pb"]["Psi"]

print(f"[Plot] collected {PLOT_VERSION} | scenario={PLOT_DATA['scenario']} | Pb model={PLOT_DATA['model_kind']}")
print("[Plot] final summary:")
for k, v in PLOT_SUMMARY.items():
    print(f"  {k}: {v:.6e}")


In [ ]:
# ===== Plotting cell 2: losses, residuals, and hydro-acid-base diagnostics =====
import numpy as np
import os
os.environ.setdefault("MPLBACKEND", "Agg")
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
import tensorflow as tf

# Always rebuild plot data from the current trained model.
# A version-only cache can leave figures out of sync with the latest PbPsi/Hplus objects.
PLOT_DATA = collect_current_outputs()
PLOT_SUMMARY = summarize_current_outputs(PLOT_DATA)
print(f"[Plot] refreshed {PLOT_VERSION} | scenario={PLOT_DATA['scenario']} | Pb model={PLOT_DATA['model_kind']}")
print(f"[Plot] sanity: final Psi mean={PLOT_SUMMARY.get('Psi_final_mean', float('nan')):.6e}, "
      f"final Pp mean={PLOT_SUMMARY.get('Pp_final_mean', float('nan')):.6e}, "
      f"mass_gap_abs_max={PLOT_SUMMARY.get('mass_gap_abs_max', float('nan')):.6e}")
print(f"[Plot] full-coupled history rows={len(globals().get('full_coupled_history', []))}")


def _loss_breakdown(model, feeds, label):
    names = sorted(model.loss_parts.keys())
    vals = model.sess.run([model.loss] + [model.loss_parts[n] for n in names], feeds)
    return label, float(vals[0]), OrderedDict((n, float(v)) for n, v in zip(names, vals[1:]))


def current_loss_breakdown():
    out = []
    if "ElecA" in globals() and ElecA is not None:
        feed_e = {
            ElecA.t_res: t_res[: min(2048, t_res.shape[0])],
            ElecA.x_res: x_res[: min(2048, x_res.shape[0])],
            ElecA.t_left: E_t_left,
            ElecA.x_left: E_x_left,
            ElecA.t_right: E_t_right,
            ElecA.x_right: E_x_right,
        }
        out.append(_loss_breakdown(ElecA, feed_e, "Electric"))
    if "WaterA" in globals() and WaterA is not None:
        feed_w = {
            WaterA.t_res: t_res[: min(2048, t_res.shape[0])],
            WaterA.x_res: x_res[: min(2048, x_res.shape[0])],
            WaterA.t_ic: t_ic,
            WaterA.x_ic: x_ic,
            WaterA.t_left: W_t_left,
            WaterA.x_left: W_x_left,
            WaterA.t_right: W_t_right,
            WaterA.x_right: W_x_right,
        }
        out.append(_loss_breakdown(WaterA, feed_w, "Water"))
    if "Hplus" in globals():
        feed_h = {
            Hplus.t_res: t_res,
            Hplus.x_res: x_res,
            Hplus.t_ic: t_ic,
            Hplus.x_ic: x_ic,
            Hplus.t_left: H_t_left,
            Hplus.x_left: H_x_left,
            Hplus.t_right: H_t_right,
            Hplus.x_right: H_x_right,
        }
        weights = getattr(Hplus, "_last_weight_values", None)
        if weights is not None:
            res_w, flux_w, coion_w = weights
            feed_h.update({
                Hplus.residual_weight_ph: res_w,
                Hplus.faraday_flux_weight_ph: flux_w,
                Hplus.coion_flux_weight_ph: coion_w,
            })
        out.append(_loss_breakdown(Hplus, feed_h, "Acid/Base"))
    if "PbPsi" in globals():
        feed_pb = {
            PbPsi.t_res: t_res[: min(2048, t_res.shape[0])],
            PbPsi.x_res: x_res[: min(2048, x_res.shape[0])],
            PbPsi.t_ic: t_ic,
            PbPsi.x_ic: x_ic,
            PbPsi.t_left: PB_t_left,
            PbPsi.x_left: PB_x_left,
            PbPsi.t_right: PB_t_right,
            PbPsi.x_right: PB_x_right,
            PbPsi.t_mass: PB_t_right[: min(256, PB_t_right.shape[0])],
        }
        out.append(_loss_breakdown(PbPsi, feed_pb, "Pb"))
    return out


LOSS_BREAKDOWN = current_loss_breakdown()
print("[Plot] current loss breakdown")
for label, total, parts in LOSS_BREAKDOWN:
    print(f"  {label}: total={total:.6e} | " + ", ".join(f"{k}={v:.3e}" for k, v in parts.items()))


def loss_history_rows():
    rows = []
    for item in globals().get("LOSS_HISTORY_BY_STAGE", []):
        stage = item.get("stage", "unknown")
        step_interval = int(item.get("step_interval", 100))
        for k, val in enumerate(item.get("loss", [])):
            rows.append({"stage": stage, "iteration": k * step_interval, "loss": float(val)})
    return rows


# A) Loss histories by real training stage, plus final current loss parts.
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
history_rows = loss_history_rows()
if history_rows:
    hist_df = pd.DataFrame(history_rows)
    for stage, df_stage in hist_df.groupby("stage", sort=False):
        axes[0].plot(df_stage["iteration"], df_stage["loss"], lw=1.8, marker="o", ms=3, label=stage)
else:
    fallback = [("Current Acid/Base", globals().get("Hplus")), ("Current Pb", globals().get("PbPsi"))]
    for stage, obj in fallback:
        if obj is None:
            continue
        hist = np.asarray(getattr(obj, "loss_hist", []), dtype=float)
        if hist.size:
            axes[0].plot(np.arange(hist.size) * 100, hist, lw=1.8, marker="o", ms=3, label=stage)
axes[0].set_yscale("log")
axes[0].set_xlabel("Iteration within that stage")
axes[0].set_ylabel("Total loss")
axes[0].set_title("Loss History By Stage")
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False, fontsize=8)

offset = 0
xticks = []
xticklabels = []
for label, total, parts in LOSS_BREAKDOWN:
    vals = list(parts.values())
    keys = list(parts.keys())
    xs = np.arange(len(vals)) + offset
    axes[1].bar(xs, vals, label=f"{label} total={total:.2e}")
    xticks.extend(xs)
    xticklabels.extend([f"{label}\n{k}" for k in keys])
    offset += len(vals) + 1
axes[1].set_yscale("log")
axes[1].set_ylabel("Loss component")
axes[1].set_title("Current loss components")
axes[1].set_xticks(xticks)
axes[1].set_xticklabels(xticklabels, rotation=45, ha="right", fontsize=8)
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout()
show_and_save(fig, f"loss_breakdown_{PLOT_DATA['scenario']}")


# B) Water/electric/acid-base profiles.
times = PLOT_DATA["paper_times"]
x_cm = PLOT_DATA["x_cm"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()
for tv in times:
    label = "Initial" if abs(tv - times[0]) < 1e-9 else f"{tv:.2f} d"
    axes[0].plot(x_cm, profile_at(PLOT_DATA["acid"]["pH"], tv)[0], lw=2, label=label)
    axes[1].plot(x_cm, profile_at(PLOT_DATA["water"]["psi"], tv)[0], lw=2, label=label)
    axes[2].plot(x_cm, profile_at(PLOT_DATA["water"]["theta"], tv)[0], lw=2, label=label)
    axes[3].plot(x_cm, profile_at(PLOT_DATA["water"]["q_total"], tv)[0], lw=2, label=label)
axes[0].set_title("pH profile")
axes[0].set_ylabel("pH")
axes[1].set_title("Pressure head")
axes[1].set_ylabel("psi (m)")
axes[2].set_title("Volumetric water content")
axes[2].set_ylabel("theta")
axes[3].set_title("Total water flux")
axes[3].set_ylabel("q (m/day)")
for ax in axes:
    ax.set_xlabel("Distance from anode (cm)")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, fontsize=8)
fig.suptitle(f"Hydro-acid-base profiles - {PLOT_DATA['scenario']}")
fig.tight_layout()
show_and_save(fig, f"hydro_acidbase_profiles_{PLOT_DATA['scenario']}")


# C) Space-time maps.
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
map_items = [
    (PLOT_DATA["acid"]["pH"], "pH", "pH"),
    (PLOT_DATA["acid"]["cOH"], "OH concentration", "OH (mol m$^{-3}$ water)"),
    (PLOT_DATA["pb"]["aq_bulk"], "Aqueous Pb, bulk basis", "Pb(aq) bulk"),
    (PLOT_DATA["pb"]["ppt_bulk"], "Precipitated Pb", "Pp"),
]
for ax, (arr, title, cbar_label) in zip(axes.ravel(), map_items):
    im = ax.contourf(PLOT_DATA["x_cm"], PLOT_DATA["t"], arr, levels=30, cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("Distance from anode (cm)")
    ax.set_ylabel("Time (day)")
    cb = fig.colorbar(im, ax=ax)
    cb.set_label(cbar_label)
fig.suptitle(f"Space-time maps - {PLOT_DATA['scenario']}")
fig.tight_layout()
show_and_save(fig, f"spacetime_maps_{PLOT_DATA['scenario']}")


# D) Water-process diagnostics.
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
water_map_items = [
    (PLOT_DATA["water"]["psi"], "Pressure head", "psi (m)"),
    (PLOT_DATA["water"]["theta"], "Water content", "theta"),
    (PLOT_DATA["water"].get("Se", np.full_like(PLOT_DATA["water"]["theta"], np.nan)), "Effective saturation", "Se"),
    (PLOT_DATA["water"].get("K_hyd", np.full_like(PLOT_DATA["water"]["theta"], np.nan)), "Hydraulic conductivity", "K (m/day)"),
    (PLOT_DATA["water"].get("psi_x", np.full_like(PLOT_DATA["water"]["theta"], np.nan)), "Head gradient", "dpsi/dx"),
    (PLOT_DATA["water"]["q_hyd"], "Pressure-driven flow", "q_hyd (m/day)"),
    (PLOT_DATA["water"]["q_eo"], "Electroosmotic flow", "q_eo (m/day)"),
    (PLOT_DATA["water"]["q_total"], "Total water flow", "q_total (m/day)"),
]
for ax, (arr, title, cbar_label) in zip(axes.ravel(), water_map_items):
    im = ax.contourf(PLOT_DATA["x_cm"], PLOT_DATA["t"], arr, levels=30, cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("Distance from anode (cm)")
    ax.set_ylabel("Time (day)")
    cb = fig.colorbar(im, ax=ax)
    cb.set_label(cbar_label)
fig.suptitle(f"Water diagnostics - {PLOT_DATA['scenario']}")
fig.tight_layout()
show_and_save(fig, f"water_diagnostics_maps_{PLOT_DATA['scenario']}")


In [ ]:
# ===== Plotting cell 3: manuscript-style figures matching current model outputs =====
import numpy as np
import os
os.environ.setdefault("MPLBACKEND", "Agg")
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt

# Always rebuild plot data from the current trained model.
# A version-only cache can leave figures out of sync with the latest PbPsi/Hplus objects.
PLOT_DATA = collect_current_outputs()
PLOT_SUMMARY = summarize_current_outputs(PLOT_DATA)
print(f"[Plot] refreshed {PLOT_VERSION} | scenario={PLOT_DATA['scenario']} | Pb model={PLOT_DATA['model_kind']}")
print(f"[Plot] sanity: final Psi mean={PLOT_SUMMARY.get('Psi_final_mean', float('nan')):.6e}, "
      f"final Pp mean={PLOT_SUMMARY.get('Pp_final_mean', float('nan')):.6e}, "
      f"mass_gap_abs_max={PLOT_SUMMARY.get('mass_gap_abs_max', float('nan')):.6e}")

times_days = PLOT_DATA["paper_times"]
x_cm = PLOT_DATA["x_cm"]


# Figure 1: paper-style Pb speciation profiles.
fig = plt.figure(figsize=(12.8, 9.4))
panels = [
    ("(a) Aqueous Pb concentration", PLOT_DATA["pb"]["aq_bulk"], r"Aqueous Pb (mol m$^{-3}$ bulk)"),
    ("(b) Adsorbed Pb concentration", PLOT_DATA["pb"]["ads_bulk"], r"Adsorbed Pb (mol m$^{-3}$ bulk)"),
    ("(c) Precipitated Pb concentration", PLOT_DATA["pb"]["ppt_bulk"], r"Precipitated Pb (mol m$^{-3}$ bulk)"),
    ("(d) Total Pb concentration distribution", PLOT_DATA["pb"]["Psi"], r"Total Pb, Psi$_{Pb}$ (mol m$^{-3}$ bulk)"),
]
for i, (title, arr, ylabel) in enumerate(panels, 1):
    ax = fig.add_subplot(2, 2, i)
    for tv in times_days:
        y, tt, _ = profile_at(arr, tv)
        label = "Initial" if abs(tt - times_days[0]) < 1e-9 else f"{tt:.2f} d"
        ax.plot(x_cm, y, lw=2.2, label=label)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Distance from anode (cm)")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.28)
    ax.legend(frameon=False)
fig.suptitle(f"Pb speciation profiles - {PLOT_DATA['scenario']}", fontsize=14, fontweight="bold")
fig.tight_layout()
show_and_save(fig, f"paper_like_pb_speciation_profiles_{PLOT_DATA['scenario']}")


# Figure 2: pH and OH profiles used to interpret Pb precipitation.
fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.6))
for tv in times_days:
    label = "Initial" if abs(tv - times_days[0]) < 1e-9 else f"{tv:.2f} d"
    axes[0].plot(x_cm, profile_at(PLOT_DATA["acid"]["pH"], tv)[0], lw=2.2, label=label)
    axes[1].plot(x_cm, profile_at(PLOT_DATA["acid"]["cOH"], tv)[0], lw=2.2, label=label)
axes[0].set_title("(a) pH distribution", fontweight="bold")
axes[0].set_ylabel("pH")
axes[1].set_title("(b) OH concentration", fontweight="bold")
axes[1].set_ylabel(r"OH$^-$ (mol m$^{-3}$ water)")
for ax in axes:
    ax.set_xlabel("Distance from anode (cm)")
    ax.grid(alpha=0.28)
    ax.legend(frameon=False)
fig.suptitle(f"Acid-base profiles - {PLOT_DATA['scenario']}", fontsize=14, fontweight="bold")
fig.tight_layout()
show_and_save(fig, f"paper_like_acidbase_profiles_{PLOT_DATA['scenario']}")


# Figure 3: stacked Pb forms at manuscript sampling positions.
positions_cm = PLOT_DATA["sample_positions_cm"]
idxs = [nearest_x_index_cm(v) for v in positions_cm]
labels = [f"{PLOT_DATA['x_cm'][i]:.1f}" for i in idxs]

def bars_at_time(t_value):
    aq, tt, k = profile_at(PLOT_DATA["pb"]["aq_bulk"], t_value)
    ads = PLOT_DATA["pb"]["ads_bulk"][k, :]
    ppt = PLOT_DATA["pb"]["ppt_bulk"][k, :]
    return tt, aq[idxs], ads[idxs], ppt[idxs]

fig, axes = plt.subplots(1, 2, figsize=(13.2, 4.8), sharey=True)
for ax, tv, title in [
    (axes[0], times_days[0], "Initial"),
    (axes[1], times_days[-1], f"{times_days[-1]:.2f} d"),
]:
    tt, aq, ads, ppt = bars_at_time(tv)
    xbar = np.arange(len(labels))
    ax.bar(xbar, aq, label="Aqueous Pb", color="#4C78A8")
    ax.bar(xbar, ads, bottom=aq, label="Adsorbed Pb", color="#F58518")
    ax.bar(xbar, ppt, bottom=aq + ads, label="Precipitated Pb", color="#54A24B")
    ax.set_title(title, fontweight="bold")
    ax.set_xticks(xbar)
    ax.set_xticklabels(labels)
    ax.set_xlabel("Distance from anode (cm)")
    ax.grid(axis="y", alpha=0.25)
axes[0].set_ylabel(r"Pb concentration (mol m$^{-3}$ bulk)")
axes[1].legend(frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
fig.suptitle(f"Forms of Pb at selected positions - {PLOT_DATA['scenario']}", fontsize=14, fontweight="bold")
fig.tight_layout()
show_and_save(fig, f"paper_like_pb_forms_stacked_bars_{PLOT_DATA['scenario']}")


# Figure 4: Pb flux and saturation diagnostics.
fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.6))
final_t = times_days[-1]
for key, label in [("J_phys", "Total Pb flux"), ("J_adv", "Advection"), ("J_diff", "Dispersion"), ("J_em", "Electromigration")]:
    if key in PLOT_DATA["pb"]:
        axes[0].plot(x_cm, profile_at(PLOT_DATA["pb"][key], final_t)[0], lw=2, label=label)
axes[0].axhline(0, color="k", lw=0.8)
axes[0].set_title("(a) Pb flux components at final time", fontweight="bold")
axes[0].set_xlabel("Distance from anode (cm)")
axes[0].set_ylabel(r"Flux (mol m$^{-2}$ day$^{-1}$)")
axes[0].grid(alpha=0.28)
axes[0].legend(frameon=False)

if np.isfinite(PLOT_DATA["pb"]["SI_log10"]).any():
    for tv in times_days:
        label = "Initial" if abs(tv - times_days[0]) < 1e-9 else f"{tv:.2f} d"
        axes[1].plot(x_cm, profile_at(PLOT_DATA["pb"]["SI_log10"], tv)[0], lw=2, label=label)
    axes[1].axhline(0, color="k", ls="--", lw=0.9)
    axes[1].set_ylabel("log10 saturation index")
else:
    axes[1].plot(x_cm, profile_at(PLOT_DATA["pb"]["mass_gap"], final_t)[0], lw=2)
    axes[1].axhline(0, color="k", ls="--", lw=0.9)
    axes[1].set_ylabel(r"Psi - reconstructed Pb (mol m$^{-3}$ bulk)")
axes[1].set_title("(b) Precipitation equilibrium check", fontweight="bold")
axes[1].set_xlabel("Distance from anode (cm)")
axes[1].grid(alpha=0.28)
axes[1].legend(frameon=False)
fig.suptitle(f"Pb transport and equilibrium diagnostics - {PLOT_DATA['scenario']}", fontsize=14, fontweight="bold")
fig.tight_layout()
show_and_save(fig, f"paper_like_pb_flux_equilibrium_{PLOT_DATA['scenario']}")


In [ ]:
# ===== Plotting cell 4: export exactly the data used by the current figures =====
import numpy as np
import pandas as pd
import json
from pathlib import Path

# Always rebuild plot data from the current trained model.
# A version-only cache can leave figures out of sync with the latest PbPsi/Hplus objects.
PLOT_DATA = collect_current_outputs()
PLOT_SUMMARY = summarize_current_outputs(PLOT_DATA)
print(f"[Plot] refreshed {PLOT_VERSION} | scenario={PLOT_DATA['scenario']} | Pb model={PLOT_DATA['model_kind']}")
print(f"[Plot] sanity: final Psi mean={PLOT_SUMMARY.get('Psi_final_mean', float('nan')):.6e}, "
      f"final Pp mean={PLOT_SUMMARY.get('Pp_final_mean', float('nan')):.6e}, "
      f"mass_gap_abs_max={PLOT_SUMMARY.get('mass_gap_abs_max', float('nan')):.6e}")


def df_profiles(arr2d, name):
    df = pd.DataFrame(arr2d, index=np.round(PLOT_DATA["t"], 6), columns=np.round(PLOT_DATA["x_cm"], 4))
    df.index.name = "time_day"
    df.columns.name = "distance_cm"
    return df


def df_selected_times(arr2d, name):
    rows = []
    for tv in PLOT_DATA["paper_times"]:
        y, tt, _ = profile_at(arr2d, tv)
        row = {"time_day": tt}
        for xval, val in zip(PLOT_DATA["x_cm"], y):
            row[f"{xval:.3f}_cm"] = float(val)
        rows.append(row)
    return pd.DataFrame(rows)


def make_loss_df():
    rows = []
    if "LOSS_BREAKDOWN" not in globals():
        return pd.DataFrame()
    for group, total, parts in LOSS_BREAKDOWN:
        rows.append({"group": group, "term": "total", "loss": total})
        for term, val in parts.items():
            rows.append({"group": group, "term": term, "loss": val})
    return pd.DataFrame(rows)


def make_loss_history_df():
    rows = []
    for item in globals().get("LOSS_HISTORY_BY_STAGE", []):
        stage = item.get("stage", "unknown")
        step_interval = int(item.get("step_interval", 100))
        for k, val in enumerate(item.get("loss", [])):
            rows.append({"stage": stage, "iteration": k * step_interval, "loss": float(val)})
    return pd.DataFrame(rows)


def make_position_summary():
    rows = []
    for tv in PLOT_DATA["paper_times"]:
        _, tt, k = profile_at(PLOT_DATA["pb"]["Psi"], tv)
        for pos in PLOT_DATA["sample_positions_cm"]:
            j = nearest_x_index_cm(pos)
            rows.append({
                "time_day": tt,
                "distance_cm": float(PLOT_DATA["x_cm"][j]),
                "pH": float(PLOT_DATA["acid"]["pH"][k, j]),
                "theta": float(PLOT_DATA["water"]["theta"][k, j]),
                "psi_m": float(PLOT_DATA["water"]["psi"][k, j]),
                "q_m_day": float(PLOT_DATA["water"]["q_total"][k, j]),
                "Pb_aq_water": float(PLOT_DATA["pb"]["Pb_aq"][k, j]),
                "Pb_aq_bulk": float(PLOT_DATA["pb"]["aq_bulk"][k, j]),
                "Pb_ads_bulk": float(PLOT_DATA["pb"]["ads_bulk"][k, j]),
                "Pb_ppt_bulk": float(PLOT_DATA["pb"]["ppt_bulk"][k, j]),
                "Psi_Pb_total": float(PLOT_DATA["pb"]["Psi"][k, j]),
                "Pb_species_sum": float(PLOT_DATA["pb"]["total_reconstructed"][k, j]),
                "mass_gap": float(PLOT_DATA["pb"]["mass_gap"][k, j]),
                "Pb_flux": float(PLOT_DATA["pb"].get("J_phys", np.full_like(PLOT_DATA["pb"]["Psi"], np.nan))[k, j]),
                "SI_log10": float(PLOT_DATA["pb"]["SI_log10"][k, j]),
            })
    return pd.DataFrame(rows)


def _cumtrapz_np(y, t):
    y = np.asarray(y, dtype=float)
    t = np.asarray(t, dtype=float)
    out = np.zeros_like(t, dtype=float)
    if len(t) > 1:
        out[1:] = np.cumsum(0.5 * (y[1:] + y[:-1]) * np.diff(t))
    return out


def make_boundary_flux_timeseries():
    tt = np.asarray(PLOT_DATA["t"], dtype=float)
    xx = np.asarray(PLOT_DATA["x"], dtype=float)
    water = PLOT_DATA["water"]
    pb = PLOT_DATA["pb"]
    nan = np.full_like(tt, np.nan, dtype=float)
    def edge(group, key, side):
        arr = group.get(key)
        if arr is None:
            return nan.copy()
        return np.asarray(arr[:, side], dtype=float)

    J_aux_right = edge(pb, "J_raw", -1)
    J_aux_left = edge(pb, "J_raw", 0)
    J_right = edge(pb, "J_phys", -1)
    J_left = edge(pb, "J_phys", 0)
    M_cath = _cumtrapz_np(J_right, tt)
    M_anode = _cumtrapz_np(-J_left, tt)
    remaining = np.trapezoid(np.asarray(pb["Psi"], dtype=float), xx, axis=1)
    initial = float(remaining[0]) if remaining.size else np.nan
    residual = initial - remaining - M_cath - M_anode
    return pd.DataFrame({
        "time_day": tt,
        "psi_left_m": edge(water, "psi", 0),
        "psi_right_m": edge(water, "psi", -1),
        "theta_left": edge(water, "theta", 0),
        "theta_right": edge(water, "theta", -1),
        "Se_left": edge(water, "Se", 0),
        "Se_right": edge(water, "Se", -1),
        "K_hyd_left_m_day": edge(water, "K_hyd", 0),
        "K_hyd_right_m_day": edge(water, "K_hyd", -1),
        "psi_x_left": edge(water, "psi_x", 0),
        "psi_x_right": edge(water, "psi_x", -1),
        "q_hyd_left_m_day": edge(water, "q_hyd", 0),
        "q_eo_left_m_day": edge(water, "q_eo", 0),
        "q_adv_left_m_day": edge(water, "q_total", 0),
        "q_hyd_right_m_day": edge(water, "q_hyd", -1),
        "q_eo_right_m_day": edge(water, "q_eo", -1),
        "q_adv_right_m_day": edge(water, "q_total", -1),
        "J_adv_Pb_right": edge(pb, "J_adv", -1),
        "J_diff_Pb_right": edge(pb, "J_diff", -1),
        "J_em_Pb_right": edge(pb, "J_em", -1),
        "J_phys_Pb_right": J_right,
        "J_phys_Pb_left": J_left,
        "J_aux_Pb_right": J_aux_right,
        "J_aux_Pb_left": J_aux_left,
        "J_constitutive_gap_right": edge(pb, "J_constitutive_gap", -1),
        "J_total_Pb_right": J_right,
        "J_total_Pb_left": J_left,
        "M_out_Pb_cathode": M_cath,
        "Pb_out_anode": M_anode,
        "Pb_initial": initial,
        "Pb_remaining_in_soil": remaining,
        "Pb_mass_balance_residual": residual,
    })


def make_pb_mass_balance():
    cols = ["time_day", "Pb_initial", "Pb_remaining_in_soil", "M_out_Pb_cathode", "Pb_out_anode", "Pb_mass_balance_residual"]
    return make_boundary_flux_timeseries()[cols]


full_coupled_history_export = globals().get("full_coupled_history", [])
meta = {
    "plot_version": PLOT_VERSION,
    "scenario": PLOT_DATA["scenario"],
    "pb_model_kind": PLOT_DATA["model_kind"],
    "domain": CFG["DOMAIN"],
    "soil": CFG["FIELDS"]["WATER"]["PARAM"]["SOIL"],
    "water_boundary": CFG["FIELDS"]["WATER"]["BOUNDARY"],
    "hplus_boundary": CFG["FIELDS"]["HPLUS"]["BOUNDARY"],
    "pb_boundary": CFG["FIELDS"]["PB"]["BOUNDARY"],
    "paper_times_day": PLOT_DATA["paper_times"],
    "sample_positions_cm": PLOT_DATA["sample_positions_cm"].tolist(),
    "summary": PLOT_SUMMARY,
    "sequential_coupling": CFG["NUMERICS"].get("SEQUENTIAL", {}),
    "coupling_history": globals().get("coupling_history", []),
    "full_coupled": CFG["NUMERICS"].get("FULL_COUPLED", {}),
    "full_coupled_history": full_coupled_history_export,
    "loss_history_stages": [item.get("stage", "unknown") for item in globals().get("LOSS_HISTORY_BY_STAGE", [])],
}

out_xlsx = EXPORT_DIR / f"current_model_plot_data_{PLOT_DATA['scenario']}.xlsx"
try:
    with pd.ExcelWriter(out_xlsx) as writer:
        pd.DataFrame([meta]).to_excel(writer, sheet_name="metadata", index=False)
        make_loss_df().to_excel(writer, sheet_name="loss_breakdown", index=False)
        make_loss_history_df().to_excel(writer, sheet_name="loss_history_by_stage", index=False)
        make_position_summary().to_excel(writer, sheet_name="selected_positions", index=False)
        make_boundary_flux_timeseries().to_excel(writer, sheet_name="boundary_fluxes", index=False)
        make_pb_mass_balance().to_excel(writer, sheet_name="pb_mass_balance", index=False)
        pd.DataFrame(globals().get("coupling_history", [])).to_excel(writer, sheet_name="coupling_history", index=False)
        pd.DataFrame(full_coupled_history_export).to_excel(writer, sheet_name="full_coupled_history", index=False)
        for sheet, arr in [
            ("pH", PLOT_DATA["acid"]["pH"]),
            ("cH", PLOT_DATA["acid"]["cH"]),
            ("cOH", PLOT_DATA["acid"]["cOH"]),
            ("psi", PLOT_DATA["water"]["psi"]),
            ("theta", PLOT_DATA["water"]["theta"]),
            ("Se", PLOT_DATA["water"].get("Se", np.full_like(PLOT_DATA["water"]["theta"], np.nan))),
            ("K_hyd", PLOT_DATA["water"].get("K_hyd", np.full_like(PLOT_DATA["water"]["theta"], np.nan))),
            ("psi_x", PLOT_DATA["water"].get("psi_x", np.full_like(PLOT_DATA["water"]["theta"], np.nan))),
            ("q_hyd", PLOT_DATA["water"]["q_hyd"]),
            ("q_eo", PLOT_DATA["water"]["q_eo"]),
            ("q_total", PLOT_DATA["water"]["q_total"]),
            ("phi", PLOT_DATA["electric"]["phi"]),
            ("Psi_Pb", PLOT_DATA["pb"]["Psi"]),
            ("Pb_aq_water", PLOT_DATA["pb"]["Pb_aq"]),
            ("Pb_aq_bulk", PLOT_DATA["pb"]["aq_bulk"]),
            ("Pb_ads_bulk", PLOT_DATA["pb"]["ads_bulk"]),
            ("Pb_ppt_bulk", PLOT_DATA["pb"]["ppt_bulk"]),
            ("Psi_Pb_total", PLOT_DATA["pb"]["Psi"]),
            ("Pb_species_sum", PLOT_DATA["pb"]["total_reconstructed"]),
            ("Pb_mass_gap", PLOT_DATA["pb"]["mass_gap"]),
            ("Pb_flux_total", PLOT_DATA["pb"].get("J_phys", np.full_like(PLOT_DATA["pb"]["Psi"], np.nan))),
            ("Pb_flux_aux", PLOT_DATA["pb"].get("J_raw", np.full_like(PLOT_DATA["pb"]["Psi"], np.nan))),
            ("Pb_flux_adv", PLOT_DATA["pb"].get("J_adv", np.full_like(PLOT_DATA["pb"]["Psi"], np.nan))),
            ("Pb_flux_diff", PLOT_DATA["pb"].get("J_diff", np.full_like(PLOT_DATA["pb"]["Psi"], np.nan))),
            ("Pb_flux_em", PLOT_DATA["pb"].get("J_em", np.full_like(PLOT_DATA["pb"]["Psi"], np.nan))),
            ("Pb_flux_phys", PLOT_DATA["pb"].get("J_phys", np.full_like(PLOT_DATA["pb"]["Psi"], np.nan))),
            ("Pb_flux_constitutive_gap", PLOT_DATA["pb"].get("J_constitutive_gap", np.full_like(PLOT_DATA["pb"]["Psi"], np.nan))),
            ("SI_log10", PLOT_DATA["pb"]["SI_log10"]),
        ]:
            df_profiles(arr, sheet).to_excel(writer, sheet_name=sheet[:31])
        for sheet, arr in [
            ("pH_lines", PLOT_DATA["acid"]["pH"]),
            ("Pb_aq_bulk_lines", PLOT_DATA["pb"]["aq_bulk"]),
            ("Pb_ads_lines", PLOT_DATA["pb"]["ads_bulk"]),
            ("Pb_ppt_lines", PLOT_DATA["pb"]["ppt_bulk"]),
            ("Psi_Pb_total_lines", PLOT_DATA["pb"]["Psi"]),
            ("Pb_species_sum_lines", PLOT_DATA["pb"]["total_reconstructed"]),
        ]:
            df_selected_times(arr, sheet).to_excel(writer, sheet_name=sheet[:31], index=False)
    print(f"[Export] current plotting data saved to: {out_xlsx}")
except Exception as exc:
    csv_dir = EXPORT_DIR / f"csv_{PLOT_DATA['scenario']}"
    csv_dir.mkdir(parents=True, exist_ok=True)
    print(f"[Export] Excel export failed ({exc}); writing CSV files to {csv_dir}")
    (csv_dir / "metadata.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    make_loss_df().to_csv(csv_dir / "loss_breakdown.csv", index=False)
    make_loss_history_df().to_csv(csv_dir / "loss_history_by_stage.csv", index=False)
    make_position_summary().to_csv(csv_dir / "selected_positions.csv", index=False)
    make_boundary_flux_timeseries().to_csv(csv_dir / "boundary_flux_timeseries.csv", index=False)
    make_pb_mass_balance().to_csv(csv_dir / "pb_mass_balance.csv", index=False)
    pd.DataFrame(globals().get("coupling_history", [])).to_csv(csv_dir / "coupling_history.csv", index=False)
    pd.DataFrame(full_coupled_history_export).to_csv(csv_dir / "full_coupled_history.csv", index=False)
    for sheet, arr in [
        ("pH", PLOT_DATA["acid"]["pH"]),
        ("psi", PLOT_DATA["water"]["psi"]),
        ("theta", PLOT_DATA["water"]["theta"]),
        ("Se", PLOT_DATA["water"].get("Se", np.full_like(PLOT_DATA["water"]["theta"], np.nan))),
        ("K_hyd", PLOT_DATA["water"].get("K_hyd", np.full_like(PLOT_DATA["water"]["theta"], np.nan))),
        ("psi_x", PLOT_DATA["water"].get("psi_x", np.full_like(PLOT_DATA["water"]["theta"], np.nan))),
        ("q_hyd", PLOT_DATA["water"]["q_hyd"]),
        ("q_eo", PLOT_DATA["water"]["q_eo"]),
        ("q_total", PLOT_DATA["water"]["q_total"]),
        ("Psi_Pb", PLOT_DATA["pb"]["Psi"]),
        ("Pb_aq_bulk", PLOT_DATA["pb"]["aq_bulk"]),
        ("Pb_ads_bulk", PLOT_DATA["pb"]["ads_bulk"]),
        ("Pb_ppt_bulk", PLOT_DATA["pb"]["ppt_bulk"]),
        ("Psi_Pb_total", PLOT_DATA["pb"]["Psi"]),
        ("Pb_species_sum", PLOT_DATA["pb"]["total_reconstructed"]),
        ("Pb_flux_total", PLOT_DATA["pb"].get("J_phys", np.full_like(PLOT_DATA["pb"]["Psi"], np.nan))),
        ("Pb_flux_aux", PLOT_DATA["pb"].get("J_raw", np.full_like(PLOT_DATA["pb"]["Psi"], np.nan))),
        ("Pb_flux_adv", PLOT_DATA["pb"].get("J_adv", np.full_like(PLOT_DATA["pb"]["Psi"], np.nan))),
        ("Pb_flux_diff", PLOT_DATA["pb"].get("J_diff", np.full_like(PLOT_DATA["pb"]["Psi"], np.nan))),
        ("Pb_flux_em", PLOT_DATA["pb"].get("J_em", np.full_like(PLOT_DATA["pb"]["Psi"], np.nan))),
        ("Pb_flux_phys", PLOT_DATA["pb"].get("J_phys", np.full_like(PLOT_DATA["pb"]["Psi"], np.nan))),
        ("Pb_flux_constitutive_gap", PLOT_DATA["pb"].get("J_constitutive_gap", np.full_like(PLOT_DATA["pb"]["Psi"], np.nan))),
    ]:
        df_profiles(arr, sheet).to_csv(csv_dir / f"{sheet}.csv")
